# Arabic Sentiment Analysis

A reproducible study of Arabic sentiment classification on the ArSarcasm-v2 dataset.

## Notebook roadmap

1. Setup and configuration
2. Dataset loading and validation
3. Dataset cleaning and conflict resolution
4. Arabic text preprocessing
5. Leakage-safe grouped splitting
6. Training-only exploratory analysis
7. Feature representations
8. Model development and tuning
9. Preprocessing ablation
10. Data augmentation
11. Class-imbalance handling
12. Comparative evaluation and error analysis

## 1. Setup and configuration

This section loads the shared experiment configuration, fixes sources of randomness, and selects the available compute device.

In [1]:
from __future__ import annotations

import hashlib
import html
import json
import platform
import random
import re
import shutil
import unicodedata
from functools import lru_cache
from pathlib import Path
from urllib.request import urlopen

import emoji
import joblib
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import qalsadi.lemmatizer
import seaborn as sns
import torch
import yaml
from gensim.models import FastText, Word2Vec
from IPython.display import display
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer
from scipy import sparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import ParameterGrid, StratifiedGroupKFold
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [2]:
def find_project_root(start: Path) -> Path:
    """Find the repository root from a notebook or project-root kernel."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repository root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
CONFIG_PATH = PROJECT_ROOT / "configs" / "default.yaml"

with CONFIG_PATH.open(encoding="utf-8") as config_file:
    CONFIG = yaml.safe_load(config_file)

PATHS = {
    name: PROJECT_ROOT / relative_path
    for name, relative_path in CONFIG["paths"].items()
}

CONFIG_PATH.relative_to(PROJECT_ROOT)

PosixPath('configs/default.yaml')

In [3]:
SEED = int(CONFIG["project"]["seed"])
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if CONFIG["runtime"]["deterministic"]:
    torch.use_deterministic_algorithms(True, warn_only=True)

device_preference = CONFIG["runtime"]["device"]
if device_preference == "auto":
    device_name = "mps" if torch.backends.mps.is_available() else "cpu"
else:
    device_name = device_preference

DEVICE = torch.device(device_name)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(context="notebook", style="whitegrid")

In [4]:
pd.DataFrame(
    {
        "value": [
            CONFIG["project"]["name"],
            platform.python_version(),
            torch.__version__,
            str(DEVICE),
            SEED,
        ]
    },
    index=["project", "python", "pytorch", "device", "seed"],
)

,value
project,arabic-sentiment-analysis
python,3.12.4
pytorch,2.14.0
device,mps
seed,42


## 2. Dataset loading and validation

ArSarcasm-v2 contains Arabic tweets annotated for sentiment, sarcasm, and dialect. The dataset authors provide 12,548 training records and 3,000 testing records. Both files are fully labeled.

Sources:

- [Official ArSarcasm-v2 repository](https://github.com/iabufarha/ArSarcasm-v2)
- [WANLP 2021 task overview](https://aclanthology.org/2021.wanlp-1.36/)
- [Dataset license](https://github.com/iabufarha/ArSarcasm-v2/blob/main/LICENSE)

The source revision and SHA-256 digests are pinned below. Raw CSV files are downloaded into the Git-ignored `data/raw/` directory and are never modified in place.

In [5]:
DATASET_REVISION = "eb85513e133cfc67e30aca4968fadbe2594fcb42"
OFFICIAL_RAW_BASE = (
    "https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/"
    f"{DATASET_REVISION}/ArSarcasm-v2"
)

DATASET_MANIFEST = {
    "training": {
        "filename": "training_data.csv",
        "url": f"{OFFICIAL_RAW_BASE}/training_data.csv",
        "sha256": ("1da727eb763459f436c4e61f52c6a0bc63d5300af8fcc99b5675ae4616dae04a"),
        "rows": 12_548,
    },
    "testing": {
        "filename": "testing_data.csv",
        "url": f"{OFFICIAL_RAW_BASE}/testing_data.csv",
        "sha256": ("0f0a77ba8a4a0a9370846a42a93a63244092f1fb30a909595b70e17e299bcf6c"),
        "rows": 3_000,
    },
}

DATASET_MANIFEST

{'training': {'filename': 'training_data.csv',
  'url': 'https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/eb85513e133cfc67e30aca4968fadbe2594fcb42/ArSarcasm-v2/training_data.csv',
  'sha256': '1da727eb763459f436c4e61f52c6a0bc63d5300af8fcc99b5675ae4616dae04a',
  'rows': 12548},
 'testing': {'filename': 'testing_data.csv',
  'url': 'https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/eb85513e133cfc67e30aca4968fadbe2594fcb42/ArSarcasm-v2/testing_data.csv',
  'sha256': '0f0a77ba8a4a0a9370846a42a93a63244092f1fb30a909595b70e17e299bcf6c',
  'rows': 3000}}

In [6]:
def file_sha256(path: Path) -> str:
    """Calculate a file's SHA-256 digest without loading it into memory."""
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_verified_file(specification: dict[str, object]) -> Path:
    """Download one immutable dataset file and verify its checksum."""
    raw_directory = PATHS["raw_data"]
    raw_directory.mkdir(parents=True, exist_ok=True)

    destination = raw_directory / str(specification["filename"])
    expected_digest = str(specification["sha256"])

    if destination.exists():
        observed_digest = file_sha256(destination)
        if observed_digest != expected_digest:
            raise ValueError(f"Checksum mismatch for existing file: {destination.name}")
        return destination

    temporary_path = destination.with_suffix(".download")
    temporary_path.unlink(missing_ok=True)

    try:
        with (
            urlopen(str(specification["url"]), timeout=60) as response,
            temporary_path.open("wb") as target,
        ):
            shutil.copyfileobj(response, target)

        observed_digest = file_sha256(temporary_path)
        if observed_digest != expected_digest:
            raise ValueError(f"Checksum mismatch after downloading {destination.name}")
        temporary_path.replace(destination)
    finally:
        temporary_path.unlink(missing_ok=True)

    return destination


DATASET_PATHS = {
    split_name: download_verified_file(specification)
    for split_name, specification in DATASET_MANIFEST.items()
}

{name: path.relative_to(PROJECT_ROOT) for name, path in DATASET_PATHS.items()}

{'training': PosixPath('data/raw/training_data.csv'),
 'testing': PosixPath('data/raw/testing_data.csv')}

In [7]:
EXPECTED_DIALECTS = {"egypt", "gulf", "levant", "magreb", "msa"}


def validate_dataset(
    frame: pd.DataFrame,
    split_name: str,
    expected_rows: int,
) -> None:
    """Reject incomplete or structurally unexpected source data."""
    expected_columns = list(CONFIG["data"]["expected_columns"])
    expected_sentiments = set(CONFIG["data"]["classes"])

    problems = []
    if len(frame) != expected_rows:
        problems.append(f"expected {expected_rows} rows, found {len(frame)}")
    if list(frame.columns) != expected_columns:
        problems.append("unexpected columns or column order")
    if frame.isna().any().any():
        problems.append("missing values found")
    if set(frame["sentiment"].unique()) != expected_sentiments:
        problems.append("unexpected sentiment labels")
    if set(frame["dialect"].unique()) != EXPECTED_DIALECTS:
        problems.append("unexpected dialect labels")
    if not pd.api.types.is_bool_dtype(frame["sarcasm"]):
        problems.append("sarcasm must be boolean")

    normalized_text = (
        frame["tweet"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
    )
    if normalized_text.eq("").any():
        problems.append("blank tweet text found")

    if problems:
        raise ValueError(f"{split_name}: {'; '.join(problems)}")


DATASET_FRAMES = {}
for split_name, specification in DATASET_MANIFEST.items():
    frame = pd.read_csv(DATASET_PATHS[split_name])
    validate_dataset(frame, split_name, int(specification["rows"]))
    DATASET_FRAMES[split_name] = frame

raw_data = pd.concat(
    [
        frame.assign(source_split=split_name, source_row=np.arange(len(frame)))
        for split_name, frame in DATASET_FRAMES.items()
    ],
    ignore_index=True,
)

raw_data.shape

(15548, 6)

In [8]:
source_summary = pd.DataFrame(
    [
        {
            "source_split": split_name,
            "rows": len(DATASET_FRAMES[split_name]),
            "columns": len(DATASET_FRAMES[split_name].columns),
            "missing_values": int(DATASET_FRAMES[split_name].isna().sum().sum()),
            "sha256": file_sha256(DATASET_PATHS[split_name]),
        }
        for split_name in DATASET_MANIFEST
    ]
)

sentiment_distribution = pd.crosstab(
    raw_data["sentiment"],
    raw_data["source_split"],
    margins=True,
).rename_axis(index="sentiment", columns="source")

display(source_summary)
display(sentiment_distribution)

,source_split,rows,columns,missing_values,sha256
0,training,12548,4,0,1da727eb763459f436c4e61f52c6a0bc63d5300af8fcc9...
1,testing,3000,4,0,0f0a77ba8a4a0a9370846a42a93a63244092f1fb30a909...


source,testing,training,All
sentiment,,,
NEG,1677,4621,6298
NEU,748,5747,6495
POS,575,2180,2755
All,3000,12548,15548


In [9]:
audit_text = (
    raw_data["tweet"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
)
duplicate_mask = audit_text.duplicated(keep=False)

duplicate_groups = (
    raw_data.assign(_audit_text=audit_text)
    .groupby("_audit_text", sort=False)
    .agg(
        rows=("tweet", "size"),
        source_splits=("source_split", "nunique"),
        sentiment_labels=("sentiment", "nunique"),
        sarcasm_labels=("sarcasm", "nunique"),
        dialect_labels=("dialect", "nunique"),
    )
)
duplicate_groups = duplicate_groups[duplicate_groups["rows"] > 1]

duplicate_audit = pd.Series(
    {
        "combined_rows": len(raw_data),
        "exact_duplicate_rows": int(
            raw_data.duplicated(
                subset=["tweet", "sarcasm", "sentiment", "dialect"]
            ).sum()
        ),
        "rows_in_duplicate_text_groups": int(duplicate_mask.sum()),
        "duplicate_text_groups": len(duplicate_groups),
        "groups_crossing_source_splits": int(
            duplicate_groups["source_splits"].gt(1).sum()
        ),
        "groups_with_sentiment_conflicts": int(
            duplicate_groups["sentiment_labels"].gt(1).sum()
        ),
        "groups_with_sarcasm_conflicts": int(
            duplicate_groups["sarcasm_labels"].gt(1).sum()
        ),
        "groups_with_dialect_conflicts": int(
            duplicate_groups["dialect_labels"].gt(1).sum()
        ),
    },
    name="count",
).to_frame()

duplicate_audit

,count
combined_rows,15548
exact_duplicate_rows,18
rows_in_duplicate_text_groups,132
duplicate_text_groups,65
groups_crossing_source_splits,2
groups_with_sentiment_conflicts,26
groups_with_sarcasm_conflicts,20
groups_with_dialect_conflicts,21


### Validation notes

The notebook deliberately reports duplicate counts without displaying tweet contents. Duplicate normalized texts will be treated as groups during the new 60/20/20 split so equivalent text cannot leak across partitions. Conflicting duplicate annotations will be resolved by an explicit rule in the splitting section rather than silently overwritten here.

## 3. Dataset cleaning and conflict resolution

The assignment requires a new 60/20/20 split. Before splitting, whitespace-equivalent tweet text is grouped to prevent identical content from leaking across partitions. Groups containing contradictory sentiment targets are removed in full. Remaining duplicate groups are collapsed to one representative; deterministic modes resolve auxiliary sarcasm and dialect disagreements.

In [10]:
def deterministic_mode(values: pd.Series) -> object:
    """Return a stable mode, breaking count ties lexicographically."""
    counts = values.value_counts(dropna=False)
    winners = counts[counts.eq(counts.max())].index.tolist()
    return sorted(winners, key=str)[0]


data_with_split_key = raw_data.assign(_split_text=audit_text)
target_label_counts = data_with_split_key.groupby("_split_text")[
    CONFIG["data"]["target"]
].nunique()
conflicting_target_keys = set(target_label_counts[target_label_counts.gt(1)].index)
target_conflict_mask = data_with_split_key["_split_text"].isin(conflicting_target_keys)
eligible_data = data_with_split_key.loc[~target_conflict_mask].copy()

clean_data = (
    eligible_data.groupby("_split_text", sort=False, as_index=False)
    .agg(
        tweet=("tweet", "first"),
        sentiment=("sentiment", "first"),
        sarcasm=("sarcasm", deterministic_mode),
        dialect=("dialect", deterministic_mode),
        source_split=("source_split", "first"),
        source_row=("source_row", "first"),
        duplicate_count=("tweet", "size"),
    )
    .rename(columns={"_split_text": "split_text_key"})
)
clean_data["record_id"] = clean_data["split_text_key"].map(
    lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]
)
clean_data = clean_data.sort_values("record_id").reset_index(drop=True)

if clean_data["record_id"].duplicated().any():
    raise ValueError("Record ID collision detected.")

cleaning_summary = pd.Series(
    {
        "original_rows": len(raw_data),
        "target_conflict_groups_dropped": len(conflicting_target_keys),
        "rows_in_target_conflict_groups": int(target_conflict_mask.sum()),
        "redundant_duplicate_rows_collapsed": len(eligible_data) - len(clean_data),
        "clean_rows_retained": len(clean_data),
        "total_rows_removed": len(raw_data) - len(clean_data),
    },
    name="count",
).to_frame()

cleaning_summary

,count
original_rows,15548
target_conflict_groups_dropped,26
rows_in_target_conflict_groups,52
redundant_duplicate_rows_collapsed,41
clean_rows_retained,15455
total_rows_removed,93


## 4. Arabic text preprocessing

Arabic social-media text contains orthographic variation, elongation, diacritics, URLs, mentions, hashtags, and emojis. The pipeline below exposes each operation as a configuration switch so the later ablation study can remove one component at a time. Negation terms are explicitly protected because removing or morphologically reducing them can invert sentiment.

In [11]:
NLTK_DATA_DIRECTORY = PATHS["artifacts"] / "nltk_data"
NLTK_DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
nltk_data_path = str(NLTK_DATA_DIRECTORY)
if nltk_data_path not in nltk.data.path:
    nltk.data.path.insert(0, nltk_data_path)

try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    downloaded = nltk.download(
        "stopwords",
        download_dir=NLTK_DATA_DIRECTORY,
        quiet=True,
        raise_on_error=True,
    )
    if not downloaded:
        raise RuntimeError("Could not download the NLTK stop-word corpus.") from None

NEGATION_TERMS = {
    "لا",
    "ليس",
    "ليست",
    "لست",
    "لن",
    "لم",
    "ما",
    "مش",
    "مو",
    "مفيش",
    "بدون",
    "غير",
}
ALL_ARABIC_STOPWORDS = set(stopwords.words("arabic"))
ARABIC_STOPWORDS = ALL_ARABIC_STOPWORDS - NEGATION_TERMS
stopword_digest = hashlib.sha256(
    "\n".join(sorted(ARABIC_STOPWORDS)).encode("utf-8")
).hexdigest()

pd.Series(
    {
        "stopword_count": len(ARABIC_STOPWORDS),
        "protected_negations": len(NEGATION_TERMS),
        "stopword_sha256": stopword_digest,
    },
    name="value",
).to_frame()

,value
stopword_count,693
protected_negations,12
stopword_sha256,c9b74cded5222de75ca75f704c2ecffe7827246d139931...


In [12]:
URL_PATTERN = re.compile(r"(?:https?://|www\.)\S+", flags=re.IGNORECASE)
HTML_TAG_PATTERN = re.compile(r"<[^>]+>")
MENTION_PATTERN = re.compile(r"(?<!\w)@[\w_]+")
HASHTAG_PATTERN = re.compile(r"#([\w_]+)")
EMOJI_ALIAS_PATTERN = re.compile(r":([a-zA-Z0-9_+&-]+):")
ARABIC_DIACRITICS_PATTERN = re.compile(
    r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]"
)
TATWEEL_PATTERN = re.compile(r"\u0640")
DIGIT_PATTERN = re.compile(r"\d+")
REPEATED_CHARACTER_PATTERN = re.compile(r"(\S)\1{2,}")
WHITESPACE_PATTERN = re.compile(r"\s+")
ARABIC_TOKEN_PATTERN = re.compile(r"[\u0600-\u06FF]+")

ARABIC_CHARACTER_TRANSLATION = str.maketrans(
    {
        "أ": "ا",
        "إ": "ا",
        "آ": "ا",
        "ٱ": "ا",
        "ى": "ي",
    }
)
NORMALIZED_ARABIC_STOPWORDS = {
    word.translate(ARABIC_CHARACTER_TRANSLATION) for word in ARABIC_STOPWORDS
}
NORMALIZED_ALL_ARABIC_STOPWORDS = {
    word.translate(ARABIC_CHARACTER_TRANSLATION) for word in ALL_ARABIC_STOPWORDS
}
ALWAYS_PROTECTED_TOKENS = {"مستخدم", "رقم"}
QALSADI_LEMMATIZER = qalsadi.lemmatizer.Lemmatizer()
ISRI_STEMMER = ISRIStemmer()

In [13]:
def replace_emoji_alias(match: re.Match[str]) -> str:
    """Turn a demojized alias into one stable token."""
    alias = match.group(1).replace("-", "_")
    return f" emoji_{alias} "


def remove_unicode_punctuation_and_symbols(text: str) -> str:
    """Replace Unicode punctuation and symbols with spaces."""
    return "".join(
        character
        if character == "_" or unicodedata.category(character)[0] not in {"P", "S"}
        else " "
        for character in text
    )


@lru_cache(maxsize=50_000)
def lemmatize_arabic_token(token: str) -> str:
    """Return a cached Qalsadi lemma, falling back to the input token."""
    lemma = QALSADI_LEMMATIZER.lemmatize(token)
    return lemma if isinstance(lemma, str) and lemma else token


def preprocess_arabic(
    text: str,
    overrides: dict[str, bool] | None = None,
) -> str:
    """Apply the configurable Arabic social-media preprocessing pipeline."""
    if not isinstance(text, str):
        raise TypeError("preprocess_arabic expects a string.")

    options = dict(CONFIG["preprocessing"])
    if overrides:
        unknown_options = set(overrides) - set(options)
        if unknown_options:
            raise KeyError(f"Unknown preprocessing options: {unknown_options}")
        options.update(overrides)
    if options["lemmatize_arabic_tokens"] and options["stem_arabic_tokens"]:
        raise ValueError("Lemmatization and stemming cannot both be enabled.")

    processed = text
    if options["decode_html_entities"]:
        processed = html.unescape(processed)
    if options["remove_html_tags"]:
        processed = HTML_TAG_PATTERN.sub(" ", processed)
    if options["remove_urls"]:
        processed = URL_PATTERN.sub(" ", processed)
    if options["replace_mentions"]:
        processed = MENTION_PATTERN.sub(" مستخدم ", processed)
    if options["demojize_emojis"]:
        processed = emoji.demojize(processed)
        processed = EMOJI_ALIAS_PATTERN.sub(replace_emoji_alias, processed)
    if options["unpack_hashtags"]:
        processed = HASHTAG_PATTERN.sub(
            lambda match: f" {match.group(1).replace('_', ' ')} ",
            processed,
        )
    if options["remove_diacritics"]:
        processed = ARABIC_DIACRITICS_PATTERN.sub("", processed)
    if options["remove_tatweel"]:
        processed = TATWEEL_PATTERN.sub("", processed)
    if options["normalize_digits"]:
        processed = DIGIT_PATTERN.sub(" رقم ", processed)
    if options["remove_punctuation_symbols"]:
        processed = remove_unicode_punctuation_and_symbols(processed)
    if options["collapse_repeated_characters"]:
        processed = REPEATED_CHARACTER_PATTERN.sub(r"\1\1", processed)

    tokens = WHITESPACE_PATTERN.sub(" ", processed).strip().split()
    if options["preserve_negation"]:
        raw_stopwords = ARABIC_STOPWORDS
        normalized_stopwords = NORMALIZED_ARABIC_STOPWORDS
    else:
        raw_stopwords = ALL_ARABIC_STOPWORDS
        normalized_stopwords = NORMALIZED_ALL_ARABIC_STOPWORDS
    stopword_vocabulary = (
        normalized_stopwords
        if options["normalize_arabic_characters"]
        else raw_stopwords
    )
    transformed_tokens = []
    for token in tokens:
        normalized_token = token.translate(ARABIC_CHARACTER_TRANSLATION)
        stopword_form = (
            normalized_token if options["normalize_arabic_characters"] else token
        )
        if options["remove_stopwords"] and stopword_form in stopword_vocabulary:
            continue

        is_negation = token in NEGATION_TERMS or normalized_token in NEGATION_TERMS
        protected = (
            token in ALWAYS_PROTECTED_TOKENS
            or token.startswith("emoji_")
            or (options["preserve_negation"] and is_negation)
        )
        if (
            options["lemmatize_arabic_tokens"]
            and not protected
            and ARABIC_TOKEN_PATTERN.fullmatch(token)
        ):
            token = lemmatize_arabic_token(token)
        if options["normalize_arabic_characters"]:
            token = token.translate(ARABIC_CHARACTER_TRANSLATION)
        if (
            options["stem_arabic_tokens"]
            and not protected
            and ARABIC_TOKEN_PATTERN.fullmatch(token)
        ):
            token = ISRI_STEMMER.stem(token)
        transformed_tokens.append(token)

    return " ".join(transformed_tokens)


def prepare_model_text(text: str) -> str:
    """Preprocess text and retain minimal cleaned content if it becomes blank."""
    processed = preprocess_arabic(text)
    if processed:
        return processed

    fallback = preprocess_arabic(
        text,
        {
            "remove_stopwords": False,
            "lemmatize_arabic_tokens": False,
            "stem_arabic_tokens": False,
        },
    )
    return fallback or "نص_فارغ"


MODEL_TEXT_COLUMN = "processed_tweet"

In [14]:
synthetic_examples = pd.DataFrame(
    {
        "original": [
            "لستُ سعيداً بهذاااا!!! 😡 #خدمة_سيئة https://example.com",
            "<b>مش</b> معقول... @customer المنتج ممتاز 😍",
            "ما أحببت الخدمة، ولكن السعر ١٢٣ شيكل",
        ]
    }
)
synthetic_examples["processed"] = synthetic_examples["original"].map(preprocess_arabic)

sanity_checks = pd.Series(
    {
        "URLs removed": synthetic_examples["processed"]
        .str.contains(r"https?://|www\.", regex=True)
        .eq(False)
        .all(),
        "HTML removed": synthetic_examples["processed"]
        .str.contains(r"<[^>]+>", regex=True)
        .eq(False)
        .all(),
        "mentions replaced": "مستخدم" in synthetic_examples.loc[1, "processed"],
        "hashtags unpacked": "#" not in synthetic_examples.loc[0, "processed"],
        "emojis tokenized": synthetic_examples["processed"]
        .str.contains("emoji_", regex=False)
        .any(),
        "negation preserved": all(
            negation in synthetic_examples.loc[index, "processed"].split()
            for index, negation in ((0, "لست"), (1, "مش"), (2, "ما"))
        ),
        "digits normalized": "رقم" in synthetic_examples.loc[2, "processed"],
        "lemmatization active": lemmatize_arabic_token("يحتاج") == "احتاج",
        "lemma meaning retained": "سيئ"
        in synthetic_examples.loc[0, "processed"].split(),
        "diacritics removed": not synthetic_examples["processed"]
        .str.contains(ARABIC_DIACRITICS_PATTERN)
        .any(),
    },
    name="passed",
)

if not sanity_checks.all():
    failed_checks = sanity_checks.index[~sanity_checks].tolist()
    raise AssertionError(f"Preprocessing sanity checks failed: {failed_checks}")

display(synthetic_examples)
display(sanity_checks.to_frame())

,original,processed
0,لستُ سعيداً بهذاااا!!! 😡 #خدمة_سيئة https://ex...,لست سعيد بهذاا emoji_enraged_face خدمة سيئ
1,<b>مش</b> معقول... @customer المنتج ممتاز 😍,مش معقول مستخدم منتج ممتاز emoji_smiling_face_...
2,ما أحببت الخدمة، ولكن السعر ١٢٣ شيكل,ما حبب خدمة سعر رقم


,passed
URLs removed,True
HTML removed,True
mentions replaced,True
hashtags unpacked,True
emojis tokenized,True
negation preserved,True
digits normalized,True
lemmatization active,True
lemma meaning retained,True
diacritics removed,True


### Preprocessing rationale

The pipeline removes transport and markup noise while retaining sentiment-bearing information. Emoji are converted to stable semantic tokens instead of deleted, hashtag content is retained without the hash marker, and negations are excluded from stop-word removal and morphological reduction. Arabic spelling variants are normalized conservatively; taa marbuta is intentionally preserved because mapping it to haa can conflate distinct words. Qalsadi lemmatization is the default because it generally preserves lexical meaning better than aggressive stemming. ISRI stemming remains available, but disabled, as an explicit ablation alternative. The later validation experiments will determine whether lemmatization actually improves sentiment classification on ArSarcasm-v2.

## 5. Leakage-safe grouped splitting

The final 60/20/20 split groups records by their complete default processed representation. A deterministic five-fold stratified group assignment keeps equivalent model inputs together while maintaining similar sentiment proportions across partitions.

In [15]:
configured_proportions = np.array(
    [
        CONFIG["split"]["train_size"],
        CONFIG["split"]["validation_size"],
        CONFIG["split"]["test_size"],
    ],
    dtype=float,
)
n_group_folds = int(CONFIG["split"]["n_group_folds"])
partition_folds = {
    name: set(folds) for name, folds in CONFIG["split"]["partition_folds"].items()
}
expected_proportions = np.array(
    [
        len(partition_folds[name]) / n_group_folds
        for name in ("train", "validation", "test")
    ]
)
if not np.allclose(configured_proportions, expected_proportions):
    raise ValueError("Configured proportions do not match grouped folds.")

split_ready_data = clean_data.copy()
split_ready_data[MODEL_TEXT_COLUMN] = split_ready_data["tweet"].map(prepare_model_text)
stratification_target = CONFIG["split"]["stratify_by"]
grouped_splitter = StratifiedGroupKFold(
    n_splits=n_group_folds,
    shuffle=True,
    random_state=SEED,
)
fold_assignments = np.full(len(split_ready_data), -1, dtype=np.int8)
for fold_index, (_, fold_rows) in enumerate(
    grouped_splitter.split(
        split_ready_data,
        y=split_ready_data[stratification_target],
        groups=split_ready_data[MODEL_TEXT_COLUMN],
    )
):
    fold_assignments[fold_rows] = fold_index

if (fold_assignments < 0).any():
    raise ValueError("One or more records were not assigned to a fold.")

split_ready_data["partition_fold"] = fold_assignments
PARTITIONS = {
    partition_name: split_ready_data.loc[split_ready_data["partition_fold"].isin(folds)]
    .sort_values("record_id")
    .reset_index(drop=True)
    for partition_name, folds in partition_folds.items()
}
split_data = pd.concat(
    [frame.assign(partition=name) for name, frame in PARTITIONS.items()],
    ignore_index=True,
)

In [16]:
partition_ids = {name: set(frame["record_id"]) for name, frame in PARTITIONS.items()}
partition_texts = {
    name: set(frame[MODEL_TEXT_COLUMN]) for name, frame in PARTITIONS.items()
}
partition_names = list(partition_ids)
for left_index, left_name in enumerate(partition_names):
    for right_name in partition_names[left_index + 1 :]:
        overlap = partition_ids[left_name] & partition_ids[right_name]
        if overlap:
            raise ValueError(f"Record leakage between {left_name} and {right_name}.")
        text_overlap = partition_texts[left_name] & partition_texts[right_name]
        if text_overlap:
            raise ValueError(
                f"Processed-text leakage between {left_name} and {right_name}."
            )

if len(split_data) != len(clean_data):
    raise ValueError("Split row counts do not match the clean dataset.")
if split_data["record_id"].duplicated().any():
    raise ValueError("Duplicate record IDs remain after splitting.")

split_counts = split_data["partition"].value_counts().reindex(PARTITIONS)
split_proportions = (split_counts / len(split_data)).rename("proportion")
split_summary = pd.concat([split_counts.rename("rows"), split_proportions], axis=1)

sentiment_counts = pd.crosstab(
    split_data["partition"],
    split_data["sentiment"],
).reindex(index=PARTITIONS, columns=CONFIG["data"]["classes"])
sentiment_proportions = sentiment_counts.div(sentiment_counts.sum(axis=1), axis=0)

processed_directory = PATHS["processed_data"]
processed_directory.mkdir(parents=True, exist_ok=True)
split_manifest_path = processed_directory / "split_manifest.csv"
split_data[
    [
        "record_id",
        "partition",
        "sentiment",
        "sarcasm",
        "dialect",
        "source_split",
        "source_row",
        "duplicate_count",
        "partition_fold",
    ]
].to_csv(split_manifest_path, index=False)

split_summary_display = split_summary.copy()
split_summary_display["proportion"] = split_summary_display["proportion"].map(
    "{:.2%}".format
)
sentiment_proportions_display = sentiment_proportions.map("{:.2%}".format)

display(split_summary_display)
display(sentiment_counts)
display(sentiment_proportions_display)
print(f"Saved local manifest: {split_manifest_path.relative_to(PROJECT_ROOT)}")

,rows,proportion
partition,,
train,9273,60.00%
validation,3091,20.00%
test,3091,20.00%


sentiment,NEG,NEU,POS
partition,,,
train,3750,3876,1647
validation,1249,1292,550
test,1249,1293,549


sentiment,NEG,NEU,POS
partition,,,
train,40.44%,41.80%,17.76%
validation,40.41%,41.80%,17.79%
test,40.41%,41.83%,17.76%


Saved local manifest: data/processed/split_manifest.csv


### Split policy

All future fitting operations use `PARTITIONS["train"]`. Validation data is reserved for model and hyperparameter selection. The test partition remains untouched until configurations are frozen. Grouping uses the complete default processed text, so no representation that is identical after cleaning and lemmatization can cross partition boundaries. Exploratory analysis in the next section uses training data, while cross-partition summaries are limited to integrity and label-balance checks. The generated split manifest is local and reproducible; it is excluded from Git with the processed data.

## 6. Training-only exploratory analysis

The following analysis uses only the training partition. It characterizes the target imbalance, auxiliary labels, and surface-level text lengths without inspecting validation or test text.

In [17]:
def distribution_table(values: pd.Series, order: list[object]) -> pd.DataFrame:
    """Return ordered counts and proportions for a categorical series."""
    counts = values.value_counts().reindex(order, fill_value=0)
    return pd.DataFrame(
        {
            "count": counts,
            "proportion": counts / counts.sum(),
        }
    )


training_eda = PARTITIONS["train"].copy()
training_eda["character_count"] = training_eda["tweet"].str.len()
training_eda["token_count"] = training_eda["tweet"].str.split().str.len()
training_eda["sarcasm_label"] = training_eda["sarcasm"].map(
    {False: "Non-sarcastic", True: "Sarcastic"}
)

sentiment_order = list(CONFIG["data"]["classes"])
sarcasm_order = ["Non-sarcastic", "Sarcastic"]
dialect_order = (
    training_eda["dialect"].value_counts().sort_values(ascending=False).index.tolist()
)

sentiment_table = distribution_table(training_eda["sentiment"], sentiment_order)
sarcasm_table = distribution_table(training_eda["sarcasm_label"], sarcasm_order)
dialect_table = distribution_table(training_eda["dialect"], dialect_order)

for title, table in (
    ("Sentiment distribution", sentiment_table),
    ("Sarcasm distribution", sarcasm_table),
    ("Dialect distribution", dialect_table),
):
    print(title)
    display(table.assign(proportion=table["proportion"].map("{:.2%}".format)))

Sentiment distribution


,count,proportion
sentiment,,
NEG,3750,40.44%
NEU,3876,41.80%
POS,1647,17.76%


Sarcasm distribution


,count,proportion
sarcasm_label,,
Non-sarcastic,7499,80.87%
Sarcastic,1774,19.13%


Dialect distribution


,count,proportion
dialect,,
msa,6471,69.78%
egypt,1763,19.01%
gulf,598,6.45%
levant,411,4.43%
magreb,30,0.32%


In [18]:
figure, axes = plt.subplots(1, 3, figsize=(16, 4.5))

distribution_specs = [
    (
        sentiment_table,
        ["Negative", "Neutral", "Positive"],
        ["#c44e52", "#8c8c8c", "#55a868"],
        "Sentiment",
    ),
    (
        sarcasm_table,
        sarcasm_order,
        ["#4c72b0", "#dd8452"],
        "Sarcasm",
    ),
    (
        dialect_table,
        [label.upper() for label in dialect_order],
        sns.color_palette("crest", len(dialect_order)),
        "Dialect",
    ),
]

for axis, (table, labels, colors, title) in zip(axes, distribution_specs, strict=True):
    bars = axis.bar(labels, table["count"], color=colors)
    axis.bar_label(bars, fmt="{:,.0f}", padding=3, fontsize=9)
    axis.set_title(f"Training {title.lower()} distribution")
    axis.set_xlabel(title)
    axis.set_ylabel("Tweets")
    axis.tick_params(axis="x", rotation=25)
    axis.margins(y=0.15)

figure.suptitle("ArSarcasm-v2 training-label distributions", fontsize=14)
figure.tight_layout()
plt.show()

/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/3241024268.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
length_summary = (
    training_eda.groupby("sentiment", observed=True)[["character_count", "token_count"]]
    .agg(["mean", "median", "max"])
    .reindex(sentiment_order)
    .round(2)
)
display(length_summary)

figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sentiment_colors = {
    "NEG": "#c44e52",
    "NEU": "#8c8c8c",
    "POS": "#55a868",
}

sns.boxplot(
    data=training_eda,
    x="sentiment",
    y="character_count",
    order=sentiment_order,
    hue="sentiment",
    palette=sentiment_colors,
    legend=False,
    showfliers=False,
    ax=axes[0],
)
sns.boxplot(
    data=training_eda,
    x="sentiment",
    y="token_count",
    order=sentiment_order,
    hue="sentiment",
    palette=sentiment_colors,
    legend=False,
    showfliers=False,
    ax=axes[1],
)
axes[0].set(title="Character counts", xlabel="Sentiment", ylabel="Characters")
axes[1].set(title="Whitespace-token counts", xlabel="Sentiment", ylabel="Tokens")
figure.suptitle("Training-text length by sentiment", fontsize=14)
figure.tight_layout()
plt.show()

character_count             token_count           
                     mean median  max        mean median max
sentiment                                                   
NEG                109.25  111.0  295       17.82   18.0  56
NEU                101.92  106.0  303       13.86   13.0  54
POS                 87.58   85.0  302       13.50   12.0  53

/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/436921732.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
sentiment_by_sarcasm = (
    pd.crosstab(
        training_eda["sentiment"],
        training_eda["sarcasm_label"],
        normalize="columns",
    )
    .reindex(index=sentiment_order, columns=sarcasm_order)
    .mul(100)
)
sentiment_by_dialect = (
    pd.crosstab(
        training_eda["sentiment"],
        training_eda["dialect"],
        normalize="columns",
    )
    .reindex(index=sentiment_order, columns=dialect_order)
    .mul(100)
)

figure, axes = plt.subplots(1, 2, figsize=(14, 4.8))
sns.heatmap(
    sentiment_by_sarcasm,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    cbar_kws={"label": "Column percentage"},
    ax=axes[0],
)
sns.heatmap(
    sentiment_by_dialect,
    annot=True,
    fmt=".1f",
    cmap="YlOrBr",
    cbar_kws={"label": "Column percentage"},
    ax=axes[1],
)
axes[0].set(title="Sentiment within sarcasm groups", xlabel="Sarcasm")
axes[1].set(title="Sentiment within dialect groups", xlabel="Dialect")
for axis in axes:
    axis.set_ylabel("Sentiment")
figure.tight_layout()
plt.show()

/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/4241998124.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Exploratory findings

- Positive sentiment is the minority target class at 17.76% of training data, compared with 40.44% negative and 41.80% neutral. This supports macro F1 as the primary selection metric and motivates the later augmentation and class-weighting experiments.
- Sarcastic tweets form 19.13% of training data. Approximately 88% of sarcastic training tweets are negative, confirming a strong relationship between sarcasm and negative sentiment.
- MSA accounts for 69.78% of training data, while Maghrebi has only 30 examples. Dialect-specific results must therefore be interpreted cautiously.
- The length summaries show whether any sentiment class is associated with systematically longer text; length itself will not be used as a target-derived preprocessing rule.

## 7. Feature representations

### 7.1 Verify modeling text

Apply the same deterministic pipeline to every fixed partition. The preprocessing resources are external, fixed lexicons rather than statistics learned from this dataset. Record identities and labels must remain unchanged, and the processed representation is audited again for empty text and cross-partition collisions before any feature extractor is fitted.

In [21]:
PROCESSED_PARTITIONS = {}
preprocessing_rows = []

for partition_name, partition_frame in PARTITIONS.items():
    processed_frame = partition_frame.copy()
    primary_text = processed_frame["tweet"].map(preprocess_arabic)
    fallback_mask = primary_text.str.strip().eq("")
    processed_text = processed_frame["tweet"].map(prepare_model_text)
    placeholder_mask = processed_text.eq("نص_فارغ")
    processed_frame[MODEL_TEXT_COLUMN] = processed_text

    for protected_column in ("record_id", "sentiment", "sarcasm", "dialect"):
        if not processed_frame[protected_column].equals(
            partition_frame[protected_column]
        ):
            raise ValueError(
                f"Preprocessing changed {protected_column} in {partition_name}."
            )

    blank_mask = processed_frame[MODEL_TEXT_COLUMN].str.strip().eq("")
    preprocessing_rows.append(
        {
            "partition": partition_name,
            "rows": len(processed_frame),
            "changed_texts": int(
                processed_frame[MODEL_TEXT_COLUMN].ne(processed_frame["tweet"]).sum()
            ),
            "blank_texts": int(blank_mask.sum()),
            "fallback_texts": int(fallback_mask.sum()),
            "placeholder_texts": int(placeholder_mask.sum()),
            "mean_tokens_before": processed_frame["tweet"].str.split().str.len().mean(),
            "mean_tokens_after": processed_frame[MODEL_TEXT_COLUMN]
            .str.split()
            .str.len()
            .mean(),
        }
    )
    PROCESSED_PARTITIONS[partition_name] = processed_frame

preprocessing_summary = pd.DataFrame(preprocessing_rows).set_index("partition").round(2)
modeling_data = pd.concat(
    [
        frame.assign(partition=partition_name)
        for partition_name, frame in PROCESSED_PARTITIONS.items()
    ],
    ignore_index=True,
)

if len(modeling_data) != len(split_data):
    raise ValueError("Preprocessing changed the total number of records.")
if modeling_data["record_id"].duplicated().any():
    raise ValueError("Duplicate record IDs appeared during preprocessing.")
if preprocessing_summary["blank_texts"].sum() > 0:
    raise ValueError("Preprocessing produced one or more blank texts.")

display(preprocessing_summary)

,rows,changed_texts,blank_texts,fallback_texts,placeholder_texts,mean_tokens_before,mean_tokens_after
partition,,,,,,,
train,9273,9257,0,2,0,15.40,13.00
validation,3091,3085,0,1,0,15.42,13.18
test,3091,3085,0,1,0,15.23,12.78


In [22]:
processed_text_groups = modeling_data.groupby(MODEL_TEXT_COLUMN, sort=False).agg(
    records=("record_id", "size"),
    partitions=("partition", "nunique"),
    sentiments=("sentiment", "nunique"),
)
processed_duplicate_groups = processed_text_groups[
    processed_text_groups["records"].gt(1)
]
cross_partition_groups = processed_duplicate_groups[
    processed_duplicate_groups["partitions"].gt(1)
]

processed_text_audit = pd.Series(
    {
        "unique_processed_texts": modeling_data[MODEL_TEXT_COLUMN].nunique(),
        "duplicate_processed_text_groups": len(processed_duplicate_groups),
        "cross_partition_processed_groups": len(cross_partition_groups),
        "conflicting_sentiment_groups": int(
            processed_duplicate_groups["sentiments"].gt(1).sum()
        ),
    },
    name="count",
).to_frame()

if not cross_partition_groups.empty:
    raise ValueError(
        f"{len(cross_partition_groups)} equivalent processed-text groups cross "
        "partition boundaries."
    )

modeling_data_path = PATHS["processed_data"] / "modeling_data.csv"
modeling_data.to_csv(modeling_data_path, index=False)

display(processed_text_audit)
print(f"Saved local modeling data: {modeling_data_path.relative_to(PROJECT_ROOT)}")

,count
unique_processed_texts,15036
duplicate_processed_text_groups,407
cross_partition_processed_groups,0
conflicting_sentiment_groups,108


Saved local modeling data: data/processed/modeling_data.csv


### Modeling-text audit notes

Four records require the conservative non-empty fallback, and none require the final placeholder. Preprocessing reduces the corpus to 15,036 unique model inputs. The 407 duplicate processed-text groups—including 108 with conflicting sentiment annotations—are kept intact within single partitions rather than discarded. Their ambiguity is a property of the available representation and will be revisited during error analysis. No processed-text group crosses a partition boundary.

### 7.2 TF-IDF representation

Fit a word-level TF-IDF vectorizer on training text only, then reuse the frozen vocabulary and inverse-document-frequency weights for validation and test data. The initial representation includes unigrams and bigrams, preserves already-normalized casing, and filters terms that appear in only one training record. These are transparent baseline settings rather than final tuned choices; later validation experiments will tune representation and classifier hyperparameters without consulting the test set.

In [23]:
tfidf_config = CONFIG["features"]["tfidf"]
tfidf_dtype = np.dtype(tfidf_config["dtype"]).type

TFIDF_VECTORIZER = TfidfVectorizer(
    analyzer=tfidf_config["analyzer"],
    token_pattern=tfidf_config["token_pattern"],
    lowercase=bool(tfidf_config["lowercase"]),
    ngram_range=tuple(tfidf_config["ngram_range"]),
    min_df=tfidf_config["min_df"],
    max_df=tfidf_config["max_df"],
    max_features=tfidf_config["max_features"],
    sublinear_tf=bool(tfidf_config["sublinear_tf"]),
    norm=tfidf_config["norm"],
    dtype=tfidf_dtype,
)

training_text = PROCESSED_PARTITIONS["train"][MODEL_TEXT_COLUMN]
TFIDF_MATRICES = {
    "train": TFIDF_VECTORIZER.fit_transform(training_text),
}
fitted_vocabulary = dict(TFIDF_VECTORIZER.vocabulary_)
for partition_name in ("validation", "test"):
    TFIDF_MATRICES[partition_name] = TFIDF_VECTORIZER.transform(
        PROCESSED_PARTITIONS[partition_name][MODEL_TEXT_COLUMN]
    )

if TFIDF_VECTORIZER.vocabulary_ != fitted_vocabulary:
    raise ValueError("The TF-IDF vocabulary changed while transforming held-out data.")

TFIDF_TARGETS = {
    partition_name: PROCESSED_PARTITIONS[partition_name][
        CONFIG["data"]["target"]
    ].to_numpy(copy=True)
    for partition_name in PARTITIONS
}

In [24]:
tfidf_audit_rows = []
feature_count = len(TFIDF_VECTORIZER.vocabulary_)
for partition_name, matrix in TFIDF_MATRICES.items():
    expected_rows = len(PROCESSED_PARTITIONS[partition_name])
    if matrix.shape != (expected_rows, feature_count):
        raise ValueError(f"Unexpected TF-IDF shape for {partition_name}.")
    if matrix.dtype != tfidf_dtype:
        raise TypeError(f"Unexpected TF-IDF dtype for {partition_name}.")
    if not np.isfinite(matrix.data).all():
        raise ValueError(f"Non-finite TF-IDF values found in {partition_name}.")

    matrix_bytes = matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes
    tfidf_audit_rows.append(
        {
            "partition": partition_name,
            "rows": matrix.shape[0],
            "features": matrix.shape[1],
            "nonzero_values": matrix.nnz,
            "density_percent": 100 * matrix.nnz / np.prod(matrix.shape),
            "zero_vector_rows": int((np.diff(matrix.indptr) == 0).sum()),
            "memory_mib": matrix_bytes / (1024**2),
        }
    )

tfidf_audit = pd.DataFrame(tfidf_audit_rows).set_index("partition")
feature_names = TFIDF_VECTORIZER.get_feature_names_out()
ngram_orders = np.char.count(feature_names.astype(str), " ") + 1
training_document_frequency = np.asarray(
    (TFIDF_MATRICES["train"] > 0).sum(axis=0)
).ravel()
most_common_indices = np.argsort(-training_document_frequency)[:15]
most_common_training_features = pd.DataFrame(
    {
        "feature": feature_names[most_common_indices],
        "training_document_frequency": training_document_frequency[most_common_indices],
        "idf": TFIDF_VECTORIZER.idf_[most_common_indices],
    }
)
tfidf_feature_summary = pd.Series(
    {
        "vocabulary_size": feature_count,
        "unigram_features": int((ngram_orders == 1).sum()),
        "bigram_features": int((ngram_orders == 2).sum()),
        "minimum_idf": float(TFIDF_VECTORIZER.idf_.min()),
        "maximum_idf": float(TFIDF_VECTORIZER.idf_.max()),
    },
    name="value",
).to_frame()

display(tfidf_feature_summary.round(4))
display(tfidf_audit.round({"density_percent": 4, "memory_mib": 2}))
display(most_common_training_features.round({"idf": 4}))

,value
vocabulary_size,16496.0000
unigram_features,8017.0000
bigram_features,8479.0000
minimum_idf,2.7096
maximum_idf,9.0364


,rows,features,nonzero_values,density_percent,zero_vector_rows,memory_mib
partition,,,,,,
train,9273,16496,126453,0.0827,34,1.00
validation,3091,16496,36721,0.0720,16,0.29
test,3091,16496,35874,0.0704,9,0.29


,feature,training_document_frequency,idf
0,مستخدم,1677,2.7096
1,رقم,1285,2.9757
2,RT,758,3.5030
3,RT مستخدم,757,3.5043
4,الله,673,3.6217
5,ما,643,3.6673
6,لا,616,3.7101
7,مصر,542,3.8379
8,يوم,497,3.9244
9,لي,336,4.3149


In [25]:
TFIDF_ARTIFACT_DIRECTORY = PATHS["artifacts"] / "features" / "tfidf"
TFIDF_ARTIFACT_DIRECTORY.mkdir(parents=True, exist_ok=True)

vectorizer_path = TFIDF_ARTIFACT_DIRECTORY / "vectorizer.joblib"
joblib.dump(TFIDF_VECTORIZER, vectorizer_path)
matrix_paths = {}
for partition_name, matrix in TFIDF_MATRICES.items():
    matrix_path = TFIDF_ARTIFACT_DIRECTORY / f"{partition_name}.npz"
    sparse.save_npz(matrix_path, matrix, compressed=True)
    matrix_paths[partition_name] = matrix_path

tfidf_row_index = pd.concat(
    [
        frame[["record_id", CONFIG["data"]["target"]]].assign(
            partition=partition_name,
            matrix_row=np.arange(len(frame)),
        )
        for partition_name, frame in PROCESSED_PARTITIONS.items()
    ],
    ignore_index=True,
)
row_index_path = TFIDF_ARTIFACT_DIRECTORY / "row_index.csv"
tfidf_row_index.to_csv(row_index_path, index=False)

saved_tfidf_artifacts = [vectorizer_path, *matrix_paths.values(), row_index_path]
pd.DataFrame(
    {
        "artifact": [
            str(path.relative_to(PROJECT_ROOT)) for path in saved_tfidf_artifacts
        ],
        "size_mib": [path.stat().st_size / (1024**2) for path in saved_tfidf_artifacts],
    }
).round({"size_mib": 3})

,artifact,size_mib
0,artifacts/features/tfidf/vectorizer.joblib,0.618
1,artifacts/features/tfidf/train.npz,0.636
2,artifacts/features/tfidf/validation.npz,0.189
3,artifacts/features/tfidf/test.npz,0.185
4,artifacts/features/tfidf/row_index.csv,0.480


### TF-IDF notes

The representation is deliberately sparse and compact. Validation and test matrices use the frozen training vocabulary and IDF weights, preventing held-out text from influencing feature construction. Rows with no retained training-vocabulary term are valid all-zero vectors and are reported explicitly rather than silently removed. TF-IDF will be paired with Multinomial Naive Bayes and Random Forest; it is not used with CNN, RNN, or LSTM because sparse vocabulary dimensions do not form an ordered linguistic sequence.

### 7.3 Word2Vec representation

Train skip-gram Word2Vec embeddings exclusively on tokenized training text. A stable SHA-256-based token hash and one training worker make initialization and updates reproducible without relying on the interpreter's randomized hash seed. Validation and test tokens are only projected through the frozen training vocabulary. Mean document embeddings support Gaussian Naive Bayes and Random Forest, while the learned token vectors will initialize the later CNN, RNN, and LSTM models.

In [26]:
def stable_text_hash(value: str) -> int:
    """Return a process-independent integer hash for Gensim initialization."""
    digest = hashlib.sha256(value.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], byteorder="little", signed=False)


word2vec_config = CONFIG["features"]["word2vec"]
WORD2VEC_TOKENIZED_PARTITIONS = {
    partition_name: frame[MODEL_TEXT_COLUMN].str.split().tolist()
    for partition_name, frame in PROCESSED_PARTITIONS.items()
}
WORD2VEC_MODEL = Word2Vec(
    sentences=WORD2VEC_TOKENIZED_PARTITIONS["train"],
    vector_size=int(word2vec_config["vector_size"]),
    window=int(word2vec_config["window"]),
    min_count=int(word2vec_config["min_count"]),
    sg=int(word2vec_config["sg"]),
    negative=int(word2vec_config["negative"]),
    sample=float(word2vec_config["sample"]),
    workers=int(word2vec_config["workers"]),
    epochs=int(word2vec_config["epochs"]),
    seed=SEED,
    hashfxn=stable_text_hash,
)

training_token_vocabulary = {
    token for document in WORD2VEC_TOKENIZED_PARTITIONS["train"] for token in document
}
if not set(WORD2VEC_MODEL.wv.index_to_key).issubset(training_token_vocabulary):
    raise ValueError("Word2Vec learned a token absent from the training partition.")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [27]:
def mean_word2vec_embedding(tokens: list[str]) -> np.ndarray:
    """Average in-vocabulary token vectors, or return a stable zero vector."""
    retained_vectors = [
        WORD2VEC_MODEL.wv[token] for token in tokens if token in WORD2VEC_MODEL.wv
    ]
    if not retained_vectors:
        return np.zeros(WORD2VEC_MODEL.vector_size, dtype=np.float32)
    return np.asarray(retained_vectors, dtype=np.float32).mean(axis=0, dtype=np.float32)


WORD2VEC_DOCUMENT_MATRICES = {
    partition_name: np.vstack(
        [mean_word2vec_embedding(tokens) for tokens in tokenized_documents]
    ).astype(np.float32, copy=False)
    for partition_name, tokenized_documents in WORD2VEC_TOKENIZED_PARTITIONS.items()
}

In [28]:
word2vec_audit_rows = []
for partition_name, tokenized_documents in WORD2VEC_TOKENIZED_PARTITIONS.items():
    matrix = WORD2VEC_DOCUMENT_MATRICES[partition_name]
    expected_shape = (
        len(PROCESSED_PARTITIONS[partition_name]),
        WORD2VEC_MODEL.vector_size,
    )
    if matrix.shape != expected_shape:
        raise ValueError(f"Unexpected Word2Vec shape for {partition_name}.")
    if matrix.dtype != np.float32 or not np.isfinite(matrix).all():
        raise ValueError(f"Invalid Word2Vec values for {partition_name}.")

    total_tokens = sum(len(tokens) for tokens in tokenized_documents)
    in_vocabulary_tokens = sum(
        token in WORD2VEC_MODEL.wv for tokens in tokenized_documents for token in tokens
    )
    zero_vector_rows = int(np.all(matrix == 0, axis=1).sum())
    word2vec_audit_rows.append(
        {
            "partition": partition_name,
            "rows": matrix.shape[0],
            "dimensions": matrix.shape[1],
            "tokens": total_tokens,
            "in_vocabulary_tokens": in_vocabulary_tokens,
            "coverage_percent": 100 * in_vocabulary_tokens / total_tokens,
            "zero_vector_rows": zero_vector_rows,
            "memory_mib": matrix.nbytes / (1024**2),
        }
    )

word2vec_audit = pd.DataFrame(word2vec_audit_rows).set_index("partition")
word2vec_summary = pd.Series(
    {
        "vocabulary_size": len(WORD2VEC_MODEL.wv),
        "vector_dimensions": WORD2VEC_MODEL.vector_size,
        "training_documents": WORD2VEC_MODEL.corpus_count,
        "training_tokens": WORD2VEC_MODEL.corpus_total_words,
        "epochs": WORD2VEC_MODEL.epochs,
    },
    name="value",
).to_frame()
most_frequent_training_tokens = pd.DataFrame(
    {
        "token": WORD2VEC_MODEL.wv.index_to_key[:15],
        "training_count": [
            WORD2VEC_MODEL.wv.get_vecattr(token, "count")
            for token in WORD2VEC_MODEL.wv.index_to_key[:15]
        ],
    }
)

display(word2vec_summary)
display(word2vec_audit.round({"coverage_percent": 2, "memory_mib": 2}))
display(most_frequent_training_tokens)

,value
vocabulary_size,8255
vector_dimensions,200
training_documents,9273
training_tokens,120526
epochs,20


,rows,dimensions,tokens,in_vocabulary_tokens,coverage_percent,zero_vector_rows,memory_mib
partition,,,,,,,
train,9273,200,120526,108790,90.26,33,7.07
validation,3091,200,40725,34781,85.40,16,2.36
test,3091,200,39518,33735,85.37,9,2.36


,token,training_count
0,مستخدم,2124
1,رقم,2060
2,الله,774
3,RT,758
4,ما,720
5,لا,701
6,مصر,620
7,يوم,545
8,emoji_face_with_tears_of_joy,408
9,حلبة,406


In [29]:
WORD2VEC_ARTIFACT_DIRECTORY = PATHS["artifacts"] / "features" / "word2vec"
WORD2VEC_ARTIFACT_DIRECTORY.mkdir(parents=True, exist_ok=True)

word2vec_vectors_path = WORD2VEC_ARTIFACT_DIRECTORY / "word_vectors.kv"
WORD2VEC_MODEL.wv.save(str(word2vec_vectors_path), separately=[])
word2vec_matrix_paths = {}
for partition_name, matrix in WORD2VEC_DOCUMENT_MATRICES.items():
    matrix_path = WORD2VEC_ARTIFACT_DIRECTORY / f"{partition_name}.npy"
    np.save(matrix_path, matrix, allow_pickle=False)
    word2vec_matrix_paths[partition_name] = matrix_path

word2vec_row_index = pd.concat(
    [
        frame[["record_id", CONFIG["data"]["target"]]].assign(
            partition=partition_name,
            matrix_row=np.arange(len(frame)),
        )
        for partition_name, frame in PROCESSED_PARTITIONS.items()
    ],
    ignore_index=True,
)
word2vec_row_index_path = WORD2VEC_ARTIFACT_DIRECTORY / "row_index.csv"
word2vec_row_index.to_csv(word2vec_row_index_path, index=False)

saved_word2vec_artifacts = [
    word2vec_vectors_path,
    *word2vec_matrix_paths.values(),
    word2vec_row_index_path,
]
pd.DataFrame(
    {
        "artifact": [
            str(path.relative_to(PROJECT_ROOT)) for path in saved_word2vec_artifacts
        ],
        "size_mib": [
            path.stat().st_size / (1024**2) for path in saved_word2vec_artifacts
        ],
    }
).round({"size_mib": 3})

,artifact,size_mib
0,artifacts/features/word2vec/word_vectors.kv,6.552
1,artifacts/features/word2vec/train.npy,7.075
2,artifacts/features/word2vec/validation.npy,2.358
3,artifacts/features/word2vec/test.npy,2.358
4,artifacts/features/word2vec/row_index.csv,0.480


### Word2Vec notes

The embedding model and all vocabulary decisions use training text only. Mean pooling produces fixed-width dense vectors for Gaussian Naive Bayes and Random Forest, while the token-level embedding table remains available for neural sequence models. Out-of-vocabulary tokens and documents with no retained token are reported explicitly rather than removed.

### 7.4 FastText representation

Train skip-gram FastText embeddings exclusively on tokenized training text. The word-level settings match Word2Vec, while character n-grams of length 3–6 allow training-derived subword vectors to represent unseen validation and test word forms. A bounded hashing bucket controls memory use. Mean document embeddings support Gaussian Naive Bayes and Random Forest, while token vectors will initialize the later FastText CNN, RNN, and LSTM experiments.

In [30]:
fasttext_config = CONFIG["features"]["fasttext"]
FASTTEXT_TOKENIZED_PARTITIONS = WORD2VEC_TOKENIZED_PARTITIONS
FASTTEXT_MODEL = FastText(
    sentences=FASTTEXT_TOKENIZED_PARTITIONS["train"],
    vector_size=int(fasttext_config["vector_size"]),
    window=int(fasttext_config["window"]),
    min_count=int(fasttext_config["min_count"]),
    sg=int(fasttext_config["sg"]),
    negative=int(fasttext_config["negative"]),
    sample=float(fasttext_config["sample"]),
    workers=int(fasttext_config["workers"]),
    epochs=int(fasttext_config["epochs"]),
    min_n=int(fasttext_config["min_n"]),
    max_n=int(fasttext_config["max_n"]),
    bucket=int(fasttext_config["bucket"]),
    seed=SEED,
    hashfxn=stable_text_hash,
)

if not set(FASTTEXT_MODEL.wv.index_to_key).issubset(training_token_vocabulary):
    raise ValueError("FastText learned a token absent from the training partition.")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [31]:
def mean_fasttext_embedding(tokens: list[str]) -> np.ndarray:
    """Average exact or inferred FastText vectors, with a zero fallback."""
    if not tokens:
        return np.zeros(FASTTEXT_MODEL.vector_size, dtype=np.float32)
    vectors = np.asarray(
        [FASTTEXT_MODEL.wv[token] for token in tokens], dtype=np.float32
    )
    return vectors.mean(axis=0, dtype=np.float32)


FASTTEXT_DOCUMENT_MATRICES = {
    partition_name: np.vstack(
        [mean_fasttext_embedding(tokens) for tokens in tokenized_documents]
    ).astype(np.float32, copy=False)
    for partition_name, tokenized_documents in FASTTEXT_TOKENIZED_PARTITIONS.items()
}

In [32]:
fasttext_audit_rows = []
fasttext_exact_vocabulary = set(FASTTEXT_MODEL.wv.key_to_index)
for partition_name, tokenized_documents in FASTTEXT_TOKENIZED_PARTITIONS.items():
    matrix = FASTTEXT_DOCUMENT_MATRICES[partition_name]
    expected_shape = (
        len(PROCESSED_PARTITIONS[partition_name]),
        FASTTEXT_MODEL.vector_size,
    )
    if matrix.shape != expected_shape:
        raise ValueError(f"Unexpected FastText shape for {partition_name}.")
    if matrix.dtype != np.float32 or not np.isfinite(matrix).all():
        raise ValueError(f"Invalid FastText values for {partition_name}.")

    total_tokens = sum(len(tokens) for tokens in tokenized_documents)
    exact_tokens = sum(
        token in fasttext_exact_vocabulary
        for tokens in tokenized_documents
        for token in tokens
    )
    inferred_tokens = total_tokens - exact_tokens
    zero_vector_rows = int(np.all(matrix == 0, axis=1).sum())
    fasttext_audit_rows.append(
        {
            "partition": partition_name,
            "rows": matrix.shape[0],
            "dimensions": matrix.shape[1],
            "tokens": total_tokens,
            "exact_vocabulary_tokens": exact_tokens,
            "subword_inferred_tokens": inferred_tokens,
            "exact_coverage_percent": 100 * exact_tokens / total_tokens,
            "zero_vector_rows": zero_vector_rows,
            "memory_mib": matrix.nbytes / (1024**2),
        }
    )

fasttext_audit = pd.DataFrame(fasttext_audit_rows).set_index("partition")
fasttext_summary = pd.Series(
    {
        "vocabulary_size": len(FASTTEXT_MODEL.wv),
        "vector_dimensions": FASTTEXT_MODEL.vector_size,
        "training_documents": FASTTEXT_MODEL.corpus_count,
        "training_tokens": FASTTEXT_MODEL.corpus_total_words,
        "character_ngram_range": (
            int(fasttext_config["min_n"]),
            int(fasttext_config["max_n"]),
        ),
        "subword_buckets": int(fasttext_config["bucket"]),
        "epochs": FASTTEXT_MODEL.epochs,
    },
    name="value",
).to_frame()

display(fasttext_summary)
display(
    fasttext_audit.round(
        {"exact_coverage_percent": 2, "memory_mib": 2}
    )
)

,value
vocabulary_size,8255
vector_dimensions,200
training_documents,9273
training_tokens,120526
character_ngram_range,"(3, 6)"
subword_buckets,200000
epochs,20


,rows,dimensions,tokens,exact_vocabulary_tokens,subword_inferred_tokens,exact_coverage_percent,zero_vector_rows,memory_mib
partition,,,,,,,,
train,9273,200,120526,108790,11736,90.26,0,7.07
validation,3091,200,40725,34781,5944,85.40,0,2.36
test,3091,200,39518,33735,5783,85.37,0,2.36


In [33]:
FASTTEXT_ARTIFACT_DIRECTORY = PATHS["artifacts"] / "features" / "fasttext"
FASTTEXT_ARTIFACT_DIRECTORY.mkdir(parents=True, exist_ok=True)

fasttext_vectors_path = FASTTEXT_ARTIFACT_DIRECTORY / "word_vectors.kv"
FASTTEXT_MODEL.wv.save(str(fasttext_vectors_path), separately=[])
fasttext_matrix_paths = {}
for partition_name, matrix in FASTTEXT_DOCUMENT_MATRICES.items():
    matrix_path = FASTTEXT_ARTIFACT_DIRECTORY / f"{partition_name}.npy"
    np.save(matrix_path, matrix, allow_pickle=False)
    fasttext_matrix_paths[partition_name] = matrix_path

fasttext_row_index_path = FASTTEXT_ARTIFACT_DIRECTORY / "row_index.csv"
word2vec_row_index.to_csv(fasttext_row_index_path, index=False)
saved_fasttext_artifacts = [
    fasttext_vectors_path,
    *fasttext_matrix_paths.values(),
    fasttext_row_index_path,
]
pd.DataFrame(
    {
        "artifact": [
            str(path.relative_to(PROJECT_ROOT)) for path in saved_fasttext_artifacts
        ],
        "size_mib": [
            path.stat().st_size / (1024**2) for path in saved_fasttext_artifacts
        ],
    }
).round({"size_mib": 3})

,artifact,size_mib
0,artifacts/features/fasttext/word_vectors.kv,159.140
1,artifacts/features/fasttext/train.npy,7.075
2,artifacts/features/fasttext/validation.npy,2.358
3,artifacts/features/fasttext/test.npy,2.358
4,artifacts/features/fasttext/row_index.csv,0.480


### FastText notes

The FastText vocabulary, word vectors, and subword buckets are learned from training text only. Validation and test word forms absent from the exact training vocabulary are represented through frozen training-derived character n-grams rather than used to update the embedding model. Exact-vocabulary coverage and subword-inferred token counts are reported separately.

## 8. Model development and tuning

### 8.1 TF-IDF with Multinomial Naive Bayes

Multinomial Naive Bayes is a natural sparse baseline for non-negative TF-IDF features. The search below varies additive smoothing and whether empirical class priors are used. Every candidate is fitted on the training partition and ranked on validation macro F1; weighted F1, accuracy, smaller alpha, and empirical priors provide deterministic tie-breaks. The test partition is not scored during tuning.

In [34]:
SENTIMENT_LABELS = list(CONFIG["data"]["classes"])


def aggregate_classification_metrics(
    y_true: np.ndarray,
    y_predicted: np.ndarray,
) -> dict[str, float]:
    """Calculate deterministic aggregate metrics in the configured class order."""
    metrics = {"accuracy": accuracy_score(y_true, y_predicted)}
    for average in ("macro", "weighted"):
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_predicted,
            labels=SENTIMENT_LABELS,
            average=average,
            zero_division=0,
        )
        metrics.update(
            {
                f"precision_{average}": precision,
                f"recall_{average}": recall,
                f"f1_{average}": f1,
            }
        )
    return {name: float(value) for name, value in metrics.items()}


def per_class_metrics(
    y_true: np.ndarray,
    y_predicted: np.ndarray,
) -> pd.DataFrame:
    """Calculate precision, recall, F1, and support for every sentiment."""
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_predicted,
        labels=SENTIMENT_LABELS,
        average=None,
        zero_division=0,
    )
    return pd.DataFrame(
        {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support.astype(int),
        },
        index=pd.Index(SENTIMENT_LABELS, name="sentiment"),
    )

In [35]:
mnb_search_space = CONFIG["models"]["multinomial_nb"]["search_space"]
mnb_parameter_candidates = list(ParameterGrid(mnb_search_space))
if not mnb_parameter_candidates:
    raise ValueError("The Multinomial Naive Bayes search space is empty.")

mnb_tuning_rows = []
for trial_number, parameters in enumerate(mnb_parameter_candidates, start=1):
    candidate_model = MultinomialNB(**parameters)
    candidate_model.fit(TFIDF_MATRICES["train"], TFIDF_TARGETS["train"])
    validation_predictions = candidate_model.predict(TFIDF_MATRICES["validation"])
    mnb_tuning_rows.append(
        {
            "trial": trial_number,
            **parameters,
            **aggregate_classification_metrics(
                TFIDF_TARGETS["validation"], validation_predictions
            ),
        }
    )

mnb_tuning_results = (
    pd.DataFrame(mnb_tuning_rows)
    .sort_values(
        ["f1_macro", "f1_weighted", "accuracy", "alpha", "fit_prior"],
        ascending=[False, False, False, True, False],
        kind="mergesort",
    )
    .reset_index(drop=True)
)
best_mnb_row = mnb_tuning_results.iloc[0]
BEST_TFIDF_MNB_PARAMETERS = {
    "alpha": float(best_mnb_row["alpha"]),
    "fit_prior": bool(best_mnb_row["fit_prior"]),
}
BEST_TFIDF_MNB = MultinomialNB(**BEST_TFIDF_MNB_PARAMETERS).fit(
    TFIDF_MATRICES["train"], TFIDF_TARGETS["train"]
)
TFIDF_MNB_VALIDATION_PREDICTIONS = BEST_TFIDF_MNB.predict(TFIDF_MATRICES["validation"])

display(mnb_tuning_results.round(4))
print(f"Selected parameters: {BEST_TFIDF_MNB_PARAMETERS}")

,trial,alpha,fit_prior,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,6,0.50,False,0.6635,0.6410,0.6317,0.6335,0.6653,0.6635,0.6611
1,8,1.00,False,0.6687,0.6552,0.6235,0.6317,0.6690,0.6687,0.6628
2,4,0.10,False,0.6483,0.6251,0.6368,0.6270,0.6594,0.6483,0.6502
3,3,0.10,True,0.6629,0.6485,0.6137,0.6229,0.6602,0.6629,0.6561
4,10,2.00,False,0.6658,0.6631,0.6071,0.6173,0.6682,0.6658,0.6551
5,1,0.01,True,0.6500,0.6263,0.6022,0.6089,0.6454,0.6500,0.6439
6,2,0.01,False,0.6351,0.6053,0.6098,0.6054,0.6419,0.6351,0.6361
7,5,0.50,True,0.6613,0.6758,0.5866,0.5945,0.6688,0.6613,0.6435
8,7,1.00,True,0.6506,0.6887,0.5593,0.5550,0.6698,0.6506,0.6209
9,9,2.00,True,0.6409,0.7045,0.5377,0.5175,0.6741,0.6409,0.5992


Selected parameters: {'alpha': 0.5, 'fit_prior': False}


In [36]:
tfidf_mnb_validation_metrics = aggregate_classification_metrics(
    TFIDF_TARGETS["validation"], TFIDF_MNB_VALIDATION_PREDICTIONS
)
tfidf_mnb_validation_per_class = per_class_metrics(
    TFIDF_TARGETS["validation"], TFIDF_MNB_VALIDATION_PREDICTIONS
)
tfidf_mnb_validation_confusion = pd.DataFrame(
    confusion_matrix(
        TFIDF_TARGETS["validation"],
        TFIDF_MNB_VALIDATION_PREDICTIONS,
        labels=SENTIMENT_LABELS,
    ),
    index=pd.Index(SENTIMENT_LABELS, name="actual"),
    columns=pd.Index(SENTIMENT_LABELS, name="predicted"),
)

display(pd.Series(tfidf_mnb_validation_metrics, name="value").to_frame().round(4))
display(tfidf_mnb_validation_per_class.round(4))

figure, axis = plt.subplots(figsize=(6, 5))
sns.heatmap(
    tfidf_mnb_validation_confusion,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    ax=axis,
)
axis.set_title("TF-IDF + Multinomial NB validation confusion matrix")
figure.tight_layout()
plt.show()

,value
accuracy,0.6635
precision_macro,0.6410
recall_macro,0.6317
f1_macro,0.6335
precision_weighted,0.6653
recall_weighted,0.6635
f1_weighted,0.6611


,precision,recall,f1,support
sentiment,,,,
NEG,0.6580,0.7702,0.7097,1249
NEU,0.7261,0.6339,0.6769,1292
POS,0.5389,0.4909,0.5138,550


/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/326966219.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [37]:
tuning_results_directory = PATHS["results"] / "tuning"
validation_results_directory = PATHS["results"] / "validation"
tuning_results_directory.mkdir(parents=True, exist_ok=True)
validation_results_directory.mkdir(parents=True, exist_ok=True)

mnb_tuning_path = tuning_results_directory / "tfidf_multinomial_nb.csv"
mnb_tuning_results.to_csv(mnb_tuning_path, index=False)
mnb_validation_path = validation_results_directory / "tfidf_multinomial_nb.json"
mnb_validation_result = {
    "experiment_id": "tfidf_multinomial_nb_validation",
    "representation": "tfidf",
    "classifier": "multinomial_naive_bayes",
    "seed": SEED,
    "selection_metric": CONFIG["evaluation"]["primary_metric"],
    "selected_parameters": BEST_TFIDF_MNB_PARAMETERS,
    "training_rows": len(PROCESSED_PARTITIONS["train"]),
    "validation_rows": len(PROCESSED_PARTITIONS["validation"]),
    "feature_count": TFIDF_MATRICES["train"].shape[1],
    "aggregate_metrics": tfidf_mnb_validation_metrics,
    "per_class_metrics": tfidf_mnb_validation_per_class.to_dict(orient="index"),
    "confusion_matrix": tfidf_mnb_validation_confusion.to_dict(orient="index"),
    "test_evaluated": False,
}
with mnb_validation_path.open("w", encoding="utf-8") as result_file:
    json.dump(
        mnb_validation_result,
        result_file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    result_file.write("\n")

pd.DataFrame(
    {
        "result": [
            str(mnb_tuning_path.relative_to(PROJECT_ROOT)),
            str(mnb_validation_path.relative_to(PROJECT_ROOT)),
        ]
    }
)

,result
0,results/tuning/tfidf_multinomial_nb.csv
1,results/validation/tfidf_multinomial_nb.json


### Multinomial Naive Bayes validation notes

The selected configuration is frozen from validation performance only. The per-class table makes minority positive-sentiment behavior visible alongside aggregate metrics, while the confusion matrix shows which sentiment pairs drive errors. Test evaluation remains deferred until the required model configurations and experimental decisions are frozen.

### 8.2 TF-IDF with Random Forest

Random Forest provides a nonlinear traditional baseline for the sparse TF-IDF representation. The compact grid varies ensemble size, maximum depth, and minimum leaf size while holding `max_features="sqrt"` to control the cost of high-dimensional sparse splits. Class weighting remains disabled here so its effect can be isolated in the later imbalance experiment. Candidates are ranked by validation macro F1 with weighted F1, accuracy, and configured trial order as deterministic tie-breaks; the test partition remains unscored.

In [38]:
rf_config = CONFIG["models"]["random_forest"]
rf_fixed_parameters = {
    **rf_config["fixed_parameters"],
    "random_state": SEED,
}
rf_parameter_candidates = list(ParameterGrid(rf_config["search_space"]))
if not rf_parameter_candidates:
    raise ValueError("The Random Forest search space is empty.")

rf_tuning_rows = []
for trial_number, parameters in enumerate(rf_parameter_candidates, start=1):
    candidate_model = RandomForestClassifier(
        **rf_fixed_parameters,
        **parameters,
    )
    candidate_model.fit(TFIDF_MATRICES["train"], TFIDF_TARGETS["train"])
    validation_predictions = candidate_model.predict(TFIDF_MATRICES["validation"])
    rf_tuning_rows.append(
        {
            "trial": trial_number,
            **parameters,
            **aggregate_classification_metrics(
                TFIDF_TARGETS["validation"], validation_predictions
            ),
        }
    )

rf_tuning_results = (
    pd.DataFrame(rf_tuning_rows)
    .sort_values(
        ["f1_macro", "f1_weighted", "accuracy", "trial"],
        ascending=[False, False, False, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)
best_rf_trial = int(rf_tuning_results.iloc[0]["trial"])
BEST_TFIDF_RF_PARAMETERS = dict(rf_parameter_candidates[best_rf_trial - 1])
BEST_TFIDF_RF = RandomForestClassifier(
    **rf_fixed_parameters,
    **BEST_TFIDF_RF_PARAMETERS,
).fit(TFIDF_MATRICES["train"], TFIDF_TARGETS["train"])
TFIDF_RF_VALIDATION_PREDICTIONS = BEST_TFIDF_RF.predict(TFIDF_MATRICES["validation"])

display(rf_tuning_results.round(4))
print(f"Selected parameters: {BEST_TFIDF_RF_PARAMETERS}")

,trial,max_depth,min_samples_leaf,n_estimators,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,1,NaN,1,200,0.6370,0.6368,0.5772,0.5886,0.6369,0.6370,0.6263
1,4,NaN,2,400,0.6445,0.6521,0.5775,0.5881,0.6475,0.6445,0.6306
2,2,NaN,1,400,0.6351,0.6357,0.5729,0.5837,0.6352,0.6351,0.6233
3,3,NaN,2,200,0.6370,0.6392,0.5721,0.5825,0.6379,0.6370,0.6241
4,5,40.0,1,200,0.6043,0.6568,0.5131,0.5038,0.6309,0.6043,0.5712
5,6,40.0,1,400,0.6069,0.6667,0.5139,0.5028,0.6372,0.6069,0.5723
6,8,40.0,2,400,0.6085,0.6852,0.5113,0.4952,0.6480,0.6085,0.5699
7,7,40.0,2,200,0.6047,0.6654,0.5092,0.4946,0.6358,0.6047,0.5674


Selected parameters: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 200}


In [39]:
tfidf_rf_validation_metrics = aggregate_classification_metrics(
    TFIDF_TARGETS["validation"], TFIDF_RF_VALIDATION_PREDICTIONS
)
tfidf_rf_validation_per_class = per_class_metrics(
    TFIDF_TARGETS["validation"], TFIDF_RF_VALIDATION_PREDICTIONS
)
tfidf_rf_validation_confusion = pd.DataFrame(
    confusion_matrix(
        TFIDF_TARGETS["validation"],
        TFIDF_RF_VALIDATION_PREDICTIONS,
        labels=SENTIMENT_LABELS,
    ),
    index=pd.Index(SENTIMENT_LABELS, name="actual"),
    columns=pd.Index(SENTIMENT_LABELS, name="predicted"),
)

display(pd.Series(tfidf_rf_validation_metrics, name="value").to_frame().round(4))
display(tfidf_rf_validation_per_class.round(4))

figure, axis = plt.subplots(figsize=(6, 5))
sns.heatmap(
    tfidf_rf_validation_confusion,
    annot=True,
    fmt="d",
    cmap="Greens",
    cbar=False,
    ax=axis,
)
axis.set_title("TF-IDF + Random Forest validation confusion matrix")
figure.tight_layout()
plt.show()

,value
accuracy,0.6370
precision_macro,0.6368
recall_macro,0.5772
f1_macro,0.5886
precision_weighted,0.6369
recall_weighted,0.6370
f1_weighted,0.6263


,precision,recall,f1,support
sentiment,,,,
NEG,0.6305,0.6845,0.6564,1249
NEU,0.6431,0.7252,0.6817,1292
POS,0.6367,0.3218,0.4275,550


/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/2185769855.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [40]:
rf_tuning_path = tuning_results_directory / "tfidf_random_forest.csv"
rf_tuning_results.to_csv(rf_tuning_path, index=False)
rf_validation_path = validation_results_directory / "tfidf_random_forest.json"
rf_validation_result = {
    "experiment_id": "tfidf_random_forest_validation",
    "representation": "tfidf",
    "classifier": "random_forest",
    "seed": SEED,
    "selection_metric": CONFIG["evaluation"]["primary_metric"],
    "fixed_parameters": rf_fixed_parameters,
    "selected_parameters": BEST_TFIDF_RF_PARAMETERS,
    "training_rows": len(PROCESSED_PARTITIONS["train"]),
    "validation_rows": len(PROCESSED_PARTITIONS["validation"]),
    "feature_count": TFIDF_MATRICES["train"].shape[1],
    "aggregate_metrics": tfidf_rf_validation_metrics,
    "per_class_metrics": tfidf_rf_validation_per_class.to_dict(orient="index"),
    "confusion_matrix": tfidf_rf_validation_confusion.to_dict(orient="index"),
    "test_evaluated": False,
}
with rf_validation_path.open("w", encoding="utf-8") as result_file:
    json.dump(
        rf_validation_result,
        result_file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    result_file.write("\n")

pd.DataFrame(
    {
        "result": [
            str(rf_tuning_path.relative_to(PROJECT_ROOT)),
            str(rf_validation_path.relative_to(PROJECT_ROOT)),
        ]
    }
)

,result
0,results/tuning/tfidf_random_forest.csv
1,results/validation/tfidf_random_forest.json


### Random Forest validation notes

This unweighted baseline is selected using validation data only. Its aggregate and per-class results can be compared directly with the Multinomial Naive Bayes baseline because both use the same fixed TF-IDF matrices and validation partition. Test evaluation and class weighting remain deferred.

### 8.3 Word2Vec with Gaussian Naive Bayes

Gaussian Naive Bayes is used for the dense mean Word2Vec document vectors because Multinomial Naive Bayes requires non-negative count-like inputs. The logarithmic search varies variance smoothing while retaining empirical class priors. Candidates are ranked by validation macro F1 with weighted F1, accuracy, and configured trial order as deterministic tie-breaks. The test partition remains unscored.

In [41]:
gnb_search_space = CONFIG["models"]["gaussian_nb"]["search_space"]
gnb_parameter_candidates = list(ParameterGrid(gnb_search_space))
if not gnb_parameter_candidates:
    raise ValueError("The Gaussian Naive Bayes search space is empty.")

gnb_tuning_rows = []
for trial_number, parameters in enumerate(gnb_parameter_candidates, start=1):
    candidate_model = GaussianNB(**parameters)
    candidate_model.fit(WORD2VEC_DOCUMENT_MATRICES["train"], TFIDF_TARGETS["train"])
    validation_predictions = candidate_model.predict(
        WORD2VEC_DOCUMENT_MATRICES["validation"]
    )
    gnb_tuning_rows.append(
        {
            "trial": trial_number,
            **parameters,
            **aggregate_classification_metrics(
                TFIDF_TARGETS["validation"], validation_predictions
            ),
        }
    )

gnb_tuning_results = (
    pd.DataFrame(gnb_tuning_rows)
    .sort_values(
        ["f1_macro", "f1_weighted", "accuracy", "trial"],
        ascending=[False, False, False, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)
best_gnb_trial = int(gnb_tuning_results.iloc[0]["trial"])
BEST_WORD2VEC_GNB_PARAMETERS = dict(gnb_parameter_candidates[best_gnb_trial - 1])
BEST_WORD2VEC_GNB = GaussianNB(**BEST_WORD2VEC_GNB_PARAMETERS).fit(
    WORD2VEC_DOCUMENT_MATRICES["train"], TFIDF_TARGETS["train"]
)
WORD2VEC_GNB_VALIDATION_PREDICTIONS = BEST_WORD2VEC_GNB.predict(
    WORD2VEC_DOCUMENT_MATRICES["validation"]
)

display(gnb_tuning_results.round(6))
print(f"Selected parameters: {BEST_WORD2VEC_GNB_PARAMETERS}")

,trial,var_smoothing,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,4,0.001000,0.569719,0.574302,0.535853,0.527922,0.606000,0.569719,0.554865
1,1,0.000001,0.568748,0.573059,0.535044,0.527335,0.604454,0.568748,0.554152
2,2,0.000010,0.568748,0.573059,0.535044,0.527335,0.604454,0.568748,0.554152
3,3,0.000100,0.568748,0.573059,0.535044,0.527335,0.604454,0.568748,0.554152
4,5,0.010000,0.568101,0.574751,0.533564,0.525675,0.605142,0.568101,0.552227
5,6,0.100000,0.548690,0.575142,0.509791,0.497293,0.601321,0.548690,0.522065
6,7,1.000000,0.463604,0.593151,0.413498,0.347368,0.617646,0.463604,0.365543


Selected parameters: {'var_smoothing': 0.001}


In [42]:
word2vec_gnb_validation_metrics = aggregate_classification_metrics(
    TFIDF_TARGETS["validation"], WORD2VEC_GNB_VALIDATION_PREDICTIONS
)
word2vec_gnb_validation_per_class = per_class_metrics(
    TFIDF_TARGETS["validation"], WORD2VEC_GNB_VALIDATION_PREDICTIONS
)
word2vec_gnb_validation_confusion = pd.DataFrame(
    confusion_matrix(
        TFIDF_TARGETS["validation"],
        WORD2VEC_GNB_VALIDATION_PREDICTIONS,
        labels=SENTIMENT_LABELS,
    ),
    index=pd.Index(SENTIMENT_LABELS, name="actual"),
    columns=pd.Index(SENTIMENT_LABELS, name="predicted"),
)

display(pd.Series(word2vec_gnb_validation_metrics, name="value").to_frame().round(4))
display(word2vec_gnb_validation_per_class.round(4))

figure, axis = plt.subplots(figsize=(6, 5))
sns.heatmap(
    word2vec_gnb_validation_confusion,
    annot=True,
    fmt="d",
    cmap="Purples",
    cbar=False,
    ax=axis,
)
axis.set_title("Word2Vec + Gaussian NB validation confusion matrix")
figure.tight_layout()
plt.show()

,value
accuracy,0.5697
precision_macro,0.5743
recall_macro,0.5359
f1_macro,0.5279
precision_weighted,0.6060
recall_weighted,0.5697
f1_weighted,0.5549


,precision,recall,f1,support
sentiment,,,,
NEG,0.5366,0.8223,0.6494,1249
NEU,0.7419,0.4071,0.5257,1292
POS,0.4444,0.3782,0.4086,550


/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/25856159.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [43]:
gnb_tuning_path = tuning_results_directory / "word2vec_gaussian_nb.csv"
gnb_tuning_results.to_csv(gnb_tuning_path, index=False)
gnb_validation_path = validation_results_directory / "word2vec_gaussian_nb.json"
gnb_validation_result = {
    "experiment_id": "word2vec_gaussian_nb_validation",
    "representation": "word2vec_mean",
    "classifier": "gaussian_naive_bayes",
    "seed": SEED,
    "selection_metric": CONFIG["evaluation"]["primary_metric"],
    "selected_parameters": BEST_WORD2VEC_GNB_PARAMETERS,
    "training_rows": len(PROCESSED_PARTITIONS["train"]),
    "validation_rows": len(PROCESSED_PARTITIONS["validation"]),
    "feature_count": WORD2VEC_DOCUMENT_MATRICES["train"].shape[1],
    "aggregate_metrics": word2vec_gnb_validation_metrics,
    "per_class_metrics": word2vec_gnb_validation_per_class.to_dict(orient="index"),
    "confusion_matrix": word2vec_gnb_validation_confusion.to_dict(orient="index"),
    "test_evaluated": False,
}
with gnb_validation_path.open("w", encoding="utf-8") as result_file:
    json.dump(
        gnb_validation_result,
        result_file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    result_file.write("\n")

pd.DataFrame(
    {
        "result": [
            str(gnb_tuning_path.relative_to(PROJECT_ROOT)),
            str(gnb_validation_path.relative_to(PROJECT_ROOT)),
        ]
    }
)

,result
0,results/tuning/word2vec_gaussian_nb.csv
1,results/validation/word2vec_gaussian_nb.json


### Word2Vec Gaussian Naive Bayes validation notes

This baseline evaluates whether class-conditional Gaussian distributions can separate the dense mean embeddings. Selection and reporting use validation data only, with minority positive-sentiment performance shown explicitly. Test evaluation remains deferred.

### 8.4 Word2Vec with Random Forest

Apply the same unweighted Random Forest search used for TF-IDF to the dense mean Word2Vec vectors. Holding the candidate grid constant makes performance differences more directly attributable to the representation. Candidates are fitted on training data and ranked on validation macro F1, followed by weighted F1, accuracy, and configured trial order. The test partition remains unscored.

In [44]:
word2vec_rf_config = CONFIG["models"]["random_forest"]
word2vec_rf_fixed_parameters = {
    **word2vec_rf_config["fixed_parameters"],
    "random_state": SEED,
}
word2vec_rf_parameter_candidates = list(
    ParameterGrid(word2vec_rf_config["search_space"])
)
if not word2vec_rf_parameter_candidates:
    raise ValueError("The Word2Vec Random Forest search space is empty.")

word2vec_rf_tuning_rows = []
for trial_number, parameters in enumerate(word2vec_rf_parameter_candidates, start=1):
    candidate_model = RandomForestClassifier(
        **word2vec_rf_fixed_parameters,
        **parameters,
    )
    candidate_model.fit(WORD2VEC_DOCUMENT_MATRICES["train"], TFIDF_TARGETS["train"])
    validation_predictions = candidate_model.predict(
        WORD2VEC_DOCUMENT_MATRICES["validation"]
    )
    word2vec_rf_tuning_rows.append(
        {
            "trial": trial_number,
            **parameters,
            **aggregate_classification_metrics(
                TFIDF_TARGETS["validation"], validation_predictions
            ),
        }
    )

word2vec_rf_tuning_results = (
    pd.DataFrame(word2vec_rf_tuning_rows)
    .sort_values(
        ["f1_macro", "f1_weighted", "accuracy", "trial"],
        ascending=[False, False, False, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)
best_word2vec_rf_trial = int(word2vec_rf_tuning_results.iloc[0]["trial"])
BEST_WORD2VEC_RF_PARAMETERS = dict(
    word2vec_rf_parameter_candidates[best_word2vec_rf_trial - 1]
)
BEST_WORD2VEC_RF = RandomForestClassifier(
    **word2vec_rf_fixed_parameters,
    **BEST_WORD2VEC_RF_PARAMETERS,
).fit(WORD2VEC_DOCUMENT_MATRICES["train"], TFIDF_TARGETS["train"])
WORD2VEC_RF_VALIDATION_PREDICTIONS = BEST_WORD2VEC_RF.predict(
    WORD2VEC_DOCUMENT_MATRICES["validation"]
)

display(word2vec_rf_tuning_results.round(4))
print(f"Selected parameters: {BEST_WORD2VEC_RF_PARAMETERS}")

,trial,max_depth,min_samples_leaf,n_estimators,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,5,40.0,1,200,0.6470,0.6562,0.5803,0.5900,0.6530,0.6470,0.6328
1,2,NaN,1,400,0.6487,0.6552,0.5806,0.5896,0.6533,0.6487,0.6339
2,7,40.0,2,200,0.6496,0.6588,0.5803,0.5893,0.6554,0.6496,0.6343
3,3,NaN,2,200,0.6490,0.6592,0.5798,0.5889,0.6552,0.6490,0.6337
4,1,NaN,1,200,0.6461,0.6510,0.5791,0.5884,0.6500,0.6461,0.6319
5,6,40.0,1,400,0.6474,0.6554,0.5792,0.5882,0.6525,0.6474,0.6324
6,4,NaN,2,400,0.6457,0.6587,0.5764,0.5857,0.6530,0.6457,0.6303
7,8,40.0,2,400,0.6457,0.6591,0.5761,0.5852,0.6532,0.6457,0.6301


Selected parameters: {'max_depth': 40, 'min_samples_leaf': 1, 'n_estimators': 200}


In [45]:
word2vec_rf_validation_metrics = aggregate_classification_metrics(
    TFIDF_TARGETS["validation"], WORD2VEC_RF_VALIDATION_PREDICTIONS
)
word2vec_rf_validation_per_class = per_class_metrics(
    TFIDF_TARGETS["validation"], WORD2VEC_RF_VALIDATION_PREDICTIONS
)
word2vec_rf_validation_confusion = pd.DataFrame(
    confusion_matrix(
        TFIDF_TARGETS["validation"],
        WORD2VEC_RF_VALIDATION_PREDICTIONS,
        labels=SENTIMENT_LABELS,
    ),
    index=pd.Index(SENTIMENT_LABELS, name="actual"),
    columns=pd.Index(SENTIMENT_LABELS, name="predicted"),
)

display(pd.Series(word2vec_rf_validation_metrics, name="value").to_frame().round(4))
display(word2vec_rf_validation_per_class.round(4))

figure, axis = plt.subplots(figsize=(6, 5))
sns.heatmap(
    word2vec_rf_validation_confusion,
    annot=True,
    fmt="d",
    cmap="Oranges",
    cbar=False,
    ax=axis,
)
axis.set_title("Word2Vec + Random Forest validation confusion matrix")
figure.tight_layout()
plt.show()

,value
accuracy,0.6470
precision_macro,0.6562
recall_macro,0.5803
f1_macro,0.5900
precision_weighted,0.6530
recall_weighted,0.6470
f1_weighted,0.6328


,precision,recall,f1,support
sentiment,,,,
NEG,0.6165,0.7774,0.6877,1249
NEU,0.6800,0.6726,0.6763,1292
POS,0.6723,0.2909,0.4061,550


/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/180946517.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [46]:
word2vec_rf_tuning_path = tuning_results_directory / "word2vec_random_forest.csv"
word2vec_rf_tuning_results.to_csv(word2vec_rf_tuning_path, index=False)
word2vec_rf_validation_path = (
    validation_results_directory / "word2vec_random_forest.json"
)
word2vec_rf_validation_result = {
    "experiment_id": "word2vec_random_forest_validation",
    "representation": "word2vec_mean",
    "classifier": "random_forest",
    "seed": SEED,
    "selection_metric": CONFIG["evaluation"]["primary_metric"],
    "fixed_parameters": word2vec_rf_fixed_parameters,
    "selected_parameters": BEST_WORD2VEC_RF_PARAMETERS,
    "training_rows": len(PROCESSED_PARTITIONS["train"]),
    "validation_rows": len(PROCESSED_PARTITIONS["validation"]),
    "feature_count": WORD2VEC_DOCUMENT_MATRICES["train"].shape[1],
    "aggregate_metrics": word2vec_rf_validation_metrics,
    "per_class_metrics": word2vec_rf_validation_per_class.to_dict(orient="index"),
    "confusion_matrix": word2vec_rf_validation_confusion.to_dict(orient="index"),
    "test_evaluated": False,
}
with word2vec_rf_validation_path.open("w", encoding="utf-8") as result_file:
    json.dump(
        word2vec_rf_validation_result,
        result_file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    result_file.write("\n")

pd.DataFrame(
    {
        "result": [
            str(word2vec_rf_tuning_path.relative_to(PROJECT_ROOT)),
            str(word2vec_rf_validation_path.relative_to(PROJECT_ROOT)),
        ]
    }
)

,result
0,results/tuning/word2vec_random_forest.csv
1,results/validation/word2vec_random_forest.json


### Word2Vec Random Forest validation notes

This unweighted baseline uses the same model search and validation partition as the TF-IDF Random Forest experiment. The comparison therefore isolates the change from sparse lexical TF-IDF features to dense mean Word2Vec features. Test evaluation and class weighting remain deferred.

### 8.5 Word2Vec with CNN

Encode each processed tweet as a fixed-length sequence of training-vocabulary Word2Vec indices. Index 0 is padding and index 1 represents out-of-vocabulary tokens using the mean training embedding. The CNN applies parallel one-dimensional convolutions over token embeddings, global max pooling, dropout, and a three-class output layer. The compact search varies filter count and dropout, uses validation macro F1 for early stopping and model selection, and leaves embeddings frozen so representation learning remains separate from classifier learning. The loss is intentionally unweighted; the test partition is not scored.

In [47]:
cnn_config = CONFIG["models"]["cnn"]
CNN_MAX_SEQUENCE_LENGTH = int(cnn_config["max_sequence_length"])
CNN_PADDING_INDEX = 0
CNN_UNKNOWN_INDEX = 1
CNN_TOKEN_TO_INDEX = {
    token: index + 2 for index, token in enumerate(WORD2VEC_MODEL.wv.index_to_key)
}
CNN_EMBEDDING_WEIGHTS = np.zeros(
    (len(CNN_TOKEN_TO_INDEX) + 2, WORD2VEC_MODEL.vector_size),
    dtype=np.float32,
)
CNN_EMBEDDING_WEIGHTS[CNN_UNKNOWN_INDEX] = WORD2VEC_MODEL.wv.vectors.mean(
    axis=0, dtype=np.float32
)
CNN_EMBEDDING_WEIGHTS[2:] = WORD2VEC_MODEL.wv.vectors


def encode_word2vec_sequence(tokens: list[str]) -> np.ndarray:
    """Map tokens to a padded, truncated sequence of training-vocabulary IDs."""
    token_ids = [CNN_TOKEN_TO_INDEX.get(token, CNN_UNKNOWN_INDEX) for token in tokens]
    encoded = np.full(CNN_MAX_SEQUENCE_LENGTH, CNN_PADDING_INDEX, dtype=np.int64)
    retained_length = min(len(token_ids), CNN_MAX_SEQUENCE_LENGTH)
    encoded[:retained_length] = token_ids[:retained_length]
    return encoded


CNN_SEQUENCE_MATRICES = {
    partition_name: np.vstack(
        [encode_word2vec_sequence(tokens) for tokens in tokenized_documents]
    )
    for partition_name, tokenized_documents in WORD2VEC_TOKENIZED_PARTITIONS.items()
}
CNN_LABEL_TO_INDEX = {label: index for index, label in enumerate(SENTIMENT_LABELS)}
CNN_LABEL_ARRAYS = {
    partition_name: np.asarray(
        [CNN_LABEL_TO_INDEX[label] for label in TFIDF_TARGETS[partition_name]],
        dtype=np.int64,
    )
    for partition_name in PARTITIONS
}

In [48]:
cnn_sequence_audit_rows = []
for partition_name, tokenized_documents in WORD2VEC_TOKENIZED_PARTITIONS.items():
    sequences = CNN_SEQUENCE_MATRICES[partition_name]
    expected_shape = (len(tokenized_documents), CNN_MAX_SEQUENCE_LENGTH)
    if sequences.shape != expected_shape or sequences.dtype != np.int64:
        raise ValueError(f"Invalid CNN sequences for {partition_name}.")
    if sequences.min() < 0 or sequences.max() >= len(CNN_EMBEDDING_WEIGHTS):
        raise ValueError(f"Out-of-range CNN token index in {partition_name}.")

    total_tokens = sum(len(tokens) for tokens in tokenized_documents)
    retained_tokens = sum(
        min(len(tokens), CNN_MAX_SEQUENCE_LENGTH) for tokens in tokenized_documents
    )
    unknown_tokens = int((sequences == CNN_UNKNOWN_INDEX).sum())
    cnn_sequence_audit_rows.append(
        {
            "partition": partition_name,
            "rows": len(sequences),
            "sequence_length": CNN_MAX_SEQUENCE_LENGTH,
            "truncated_rows": sum(
                len(tokens) > CNN_MAX_SEQUENCE_LENGTH for tokens in tokenized_documents
            ),
            "retained_token_percent": 100 * retained_tokens / total_tokens,
            "unknown_token_percent": 100 * unknown_tokens / retained_tokens,
        }
    )

cnn_sequence_audit = pd.DataFrame(cnn_sequence_audit_rows).set_index("partition")
cnn_embedding_summary = pd.Series(
    {
        "embedding_rows": len(CNN_EMBEDDING_WEIGHTS),
        "embedding_dimensions": CNN_EMBEDDING_WEIGHTS.shape[1],
        "padding_index": CNN_PADDING_INDEX,
        "unknown_index": CNN_UNKNOWN_INDEX,
    },
    name="value",
).to_frame()

display(cnn_embedding_summary)
display(
    cnn_sequence_audit.round({"retained_token_percent": 2, "unknown_token_percent": 2})
)

,value
embedding_rows,8257
embedding_dimensions,200
padding_index,0
unknown_index,1


,rows,sequence_length,truncated_rows,retained_token_percent,unknown_token_percent
partition,,,,,
train,9273,64,3,100.00,9.74
validation,3091,64,1,99.88,14.61
test,3091,64,0,100.00,14.63


In [49]:
class Word2VecTextCNN(nn.Module):
    """Parallel temporal convolutions over pretrained Word2Vec embeddings."""

    def __init__(
        self,
        embedding_weights: np.ndarray,
        num_filters: int,
        kernel_sizes: list[int],
        dropout: float,
        freeze_embeddings: bool,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.from_numpy(embedding_weights.copy()),
            freeze=freeze_embeddings,
            padding_idx=CNN_PADDING_INDEX,
        )
        embedding_dimensions = embedding_weights.shape[1]
        self.convolutions = nn.ModuleList(
            [
                nn.Conv1d(
                    embedding_dimensions,
                    num_filters,
                    kernel_size,
                    bias=False,
                )
                for kernel_size in kernel_sizes
            ]
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(
            num_filters * len(kernel_sizes), len(SENTIMENT_LABELS)
        )

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(token_ids).transpose(1, 2)
        pooled = [
            torch.relu(convolution(embedded)).amax(dim=2)
            for convolution in self.convolutions
        ]
        return self.classifier(self.dropout(torch.cat(pooled, dim=1)))

In [50]:
def reset_neural_random_state(seed: int) -> None:
    """Reset all random generators used by the neural training loop."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_cnn_loader(
    partition_name: str,
    batch_size: int,
    shuffle: bool,
    seed: int,
) -> DataLoader:
    """Build a deterministic tensor loader for one fixed partition."""
    dataset = TensorDataset(
        torch.from_numpy(CNN_SEQUENCE_MATRICES[partition_name]),
        torch.from_numpy(CNN_LABEL_ARRAYS[partition_name]),
    )
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=int(CONFIG["runtime"]["num_workers"]),
        generator=generator,
    )


def predict_cnn(model: nn.Module, partition_name: str, batch_size: int) -> np.ndarray:
    """Predict configured sentiment labels in fixed row order."""
    model.eval()
    predicted_indices = []
    loader = make_cnn_loader(partition_name, batch_size, shuffle=False, seed=SEED)
    with torch.no_grad():
        for token_ids, _ in loader:
            logits = model(token_ids.to(DEVICE))
            predicted_indices.extend(logits.argmax(dim=1).cpu().tolist())
    return np.asarray([SENTIMENT_LABELS[index] for index in predicted_indices])


def train_word2vec_cnn(
    parameters: dict[str, object],
) -> tuple[dict[str, torch.Tensor], pd.DataFrame, dict[str, float], np.ndarray]:
    """Train one CNN candidate with validation-based early stopping."""
    reset_neural_random_state(SEED)
    model = Word2VecTextCNN(
        CNN_EMBEDDING_WEIGHTS,
        num_filters=int(parameters["num_filters"]),
        kernel_sizes=[int(size) for size in cnn_config["kernel_sizes"]],
        dropout=float(parameters["dropout"]),
        freeze_embeddings=bool(cnn_config["freeze_embeddings"]),
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=float(parameters["learning_rate"]),
        weight_decay=float(cnn_config["weight_decay"]),
    )
    loss_function = nn.CrossEntropyLoss()
    batch_size = int(cnn_config["batch_size"])
    training_loader = make_cnn_loader("train", batch_size, shuffle=True, seed=SEED)
    best_macro_f1 = -np.inf
    best_state = None
    epochs_without_improvement = 0
    history_rows = []

    for epoch in range(1, int(cnn_config["max_epochs"]) + 1):
        model.train()
        total_loss = 0.0
        observed_rows = 0
        for token_ids, labels in training_loader:
            token_ids = token_ids.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(token_ids)
            loss = loss_function(logits, labels)
            loss.backward()
            optimizer.step()
            batch_rows = len(labels)
            total_loss += float(loss.detach().cpu()) * batch_rows
            observed_rows += batch_rows

        validation_predictions = predict_cnn(model, "validation", batch_size)
        validation_metrics = aggregate_classification_metrics(
            TFIDF_TARGETS["validation"], validation_predictions
        )
        history_rows.append(
            {
                "epoch": epoch,
                "training_loss": total_loss / observed_rows,
                **validation_metrics,
            }
        )
        if validation_metrics["f1_macro"] > best_macro_f1 + 1e-12:
            best_macro_f1 = validation_metrics["f1_macro"]
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= int(cnn_config["patience"]):
                break

    if best_state is None:
        raise RuntimeError("CNN training did not produce a valid checkpoint.")
    model.load_state_dict(best_state)
    selected_predictions = predict_cnn(model, "validation", batch_size)
    selected_metrics = aggregate_classification_metrics(
        TFIDF_TARGETS["validation"], selected_predictions
    )
    return (
        best_state,
        pd.DataFrame(history_rows),
        selected_metrics,
        selected_predictions,
    )

In [51]:
cnn_parameter_candidates = list(ParameterGrid(cnn_config["search_space"]))
if not cnn_parameter_candidates:
    raise ValueError("The CNN search space is empty.")

cnn_tuning_rows = []
CNN_TRIAL_ARTIFACTS = {}
for trial_number, parameters in enumerate(cnn_parameter_candidates, start=1):
    model_state, history, metrics, validation_predictions = train_word2vec_cnn(
        parameters
    )
    best_epoch = int(history.loc[history["f1_macro"].idxmax(), "epoch"])
    cnn_tuning_rows.append(
        {
            "trial": trial_number,
            **parameters,
            "best_epoch": best_epoch,
            "epochs_ran": len(history),
            **metrics,
        }
    )
    CNN_TRIAL_ARTIFACTS[trial_number] = {
        "state_dict": model_state,
        "history": history,
        "validation_predictions": validation_predictions,
    }

cnn_tuning_results = (
    pd.DataFrame(cnn_tuning_rows)
    .sort_values(
        ["f1_macro", "f1_weighted", "accuracy", "trial"],
        ascending=[False, False, False, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)
best_cnn_trial = int(cnn_tuning_results.iloc[0]["trial"])
BEST_WORD2VEC_CNN_PARAMETERS = dict(cnn_parameter_candidates[best_cnn_trial - 1])
BEST_WORD2VEC_CNN_STATE = CNN_TRIAL_ARTIFACTS[best_cnn_trial]["state_dict"]
WORD2VEC_CNN_SELECTED_HISTORY = CNN_TRIAL_ARTIFACTS[best_cnn_trial]["history"]
WORD2VEC_CNN_VALIDATION_PREDICTIONS = CNN_TRIAL_ARTIFACTS[best_cnn_trial][
    "validation_predictions"
]

display(cnn_tuning_results.round(4))
print(f"Selected parameters: {BEST_WORD2VEC_CNN_PARAMETERS}")

,trial,dropout,learning_rate,num_filters,best_epoch,epochs_ran,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,4,0.5,0.001,128,4,6,0.6580,0.6482,0.6149,0.6256,0.6559,0.6580,0.6533
1,1,0.3,0.001,64,8,10,0.6590,0.6469,0.6158,0.6247,0.6575,0.6590,0.6535
2,2,0.3,0.001,128,5,7,0.6561,0.6533,0.6098,0.6206,0.6571,0.6561,0.6491
3,3,0.5,0.001,64,4,6,0.6561,0.6453,0.6089,0.6187,0.6542,0.6561,0.6496


Selected parameters: {'dropout': 0.5, 'learning_rate': 0.001, 'num_filters': 128}


In [52]:
word2vec_cnn_validation_metrics = aggregate_classification_metrics(
    TFIDF_TARGETS["validation"], WORD2VEC_CNN_VALIDATION_PREDICTIONS
)
word2vec_cnn_validation_per_class = per_class_metrics(
    TFIDF_TARGETS["validation"], WORD2VEC_CNN_VALIDATION_PREDICTIONS
)
word2vec_cnn_validation_confusion = pd.DataFrame(
    confusion_matrix(
        TFIDF_TARGETS["validation"],
        WORD2VEC_CNN_VALIDATION_PREDICTIONS,
        labels=SENTIMENT_LABELS,
    ),
    index=pd.Index(SENTIMENT_LABELS, name="actual"),
    columns=pd.Index(SENTIMENT_LABELS, name="predicted"),
)

display(pd.Series(word2vec_cnn_validation_metrics, name="value").to_frame().round(4))
display(word2vec_cnn_validation_per_class.round(4))

figure, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.lineplot(
    data=WORD2VEC_CNN_SELECTED_HISTORY,
    x="epoch",
    y="training_loss",
    marker="o",
    ax=axes[0],
)
axes[0].set_title("Selected CNN training loss")
axes[0].set_ylabel("Cross-entropy loss")
sns.heatmap(
    word2vec_cnn_validation_confusion,
    annot=True,
    fmt="d",
    cmap="Reds",
    cbar=False,
    ax=axes[1],
)
axes[1].set_title("Word2Vec + CNN validation confusion matrix")
figure.tight_layout()
plt.show()

,value
accuracy,0.6580
precision_macro,0.6482
recall_macro,0.6149
f1_macro,0.6256
precision_weighted,0.6559
recall_weighted,0.6580
f1_weighted,0.6533


,precision,recall,f1,support
sentiment,,,,
NEG,0.6806,0.6926,0.6865,1249
NEU,0.6499,0.7214,0.6838,1292
POS,0.6140,0.4309,0.5064,550


/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/3395069704.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [53]:
cnn_tuning_path = tuning_results_directory / "word2vec_cnn.csv"
cnn_history_path = tuning_results_directory / "word2vec_cnn_selected_history.csv"
cnn_validation_path = validation_results_directory / "word2vec_cnn.json"
cnn_tuning_results.to_csv(cnn_tuning_path, index=False)
WORD2VEC_CNN_SELECTED_HISTORY.to_csv(cnn_history_path, index=False)

cnn_artifact_directory = PATHS["artifacts"] / "models"
cnn_artifact_directory.mkdir(parents=True, exist_ok=True)
cnn_model_path = cnn_artifact_directory / "word2vec_cnn.pt"
torch.save(
    {
        "state_dict": BEST_WORD2VEC_CNN_STATE,
        "parameters": BEST_WORD2VEC_CNN_PARAMETERS,
        "kernel_sizes": list(cnn_config["kernel_sizes"]),
        "max_sequence_length": CNN_MAX_SEQUENCE_LENGTH,
        "label_order": SENTIMENT_LABELS,
    },
    cnn_model_path,
)

cnn_validation_result = {
    "experiment_id": "word2vec_cnn_validation",
    "representation": "word2vec_sequence",
    "classifier": "cnn",
    "seed": SEED,
    "device": str(DEVICE),
    "selection_metric": CONFIG["evaluation"]["primary_metric"],
    "selected_parameters": BEST_WORD2VEC_CNN_PARAMETERS,
    "selected_epoch": int(cnn_tuning_results.iloc[0]["best_epoch"]),
    "training_rows": len(PROCESSED_PARTITIONS["train"]),
    "validation_rows": len(PROCESSED_PARTITIONS["validation"]),
    "aggregate_metrics": word2vec_cnn_validation_metrics,
    "per_class_metrics": word2vec_cnn_validation_per_class.to_dict(orient="index"),
    "confusion_matrix": word2vec_cnn_validation_confusion.to_dict(orient="index"),
    "test_evaluated": False,
    "class_weighted": False,
}
with cnn_validation_path.open("w", encoding="utf-8") as result_file:
    json.dump(
        cnn_validation_result,
        result_file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    result_file.write("\n")

pd.DataFrame(
    {
        "result": [
            str(cnn_tuning_path.relative_to(PROJECT_ROOT)),
            str(cnn_history_path.relative_to(PROJECT_ROOT)),
            str(cnn_validation_path.relative_to(PROJECT_ROOT)),
            str(cnn_model_path.relative_to(PROJECT_ROOT)),
        ]
    }
)

,result
0,results/tuning/word2vec_cnn.csv
1,results/tuning/word2vec_cnn_selected_history.csv
2,results/validation/word2vec_cnn.json
3,artifacts/models/word2vec_cnn.pt


### Word2Vec CNN validation notes

The selected CNN checkpoint is the epoch with the strongest validation macro F1 within the best hyperparameter trial. Frozen training-only embeddings and unweighted cross-entropy define the baseline; class weighting will be tested separately. The test partition remains unevaluated.

### 8.6 Word2Vec with vanilla RNN

Reuse the audited training-vocabulary Word2Vec sequences from the CNN experiment and compute each tweet's retained length so padding is excluded from the recurrent state. A single-layer tanh RNN summarizes the ordered token embeddings, followed by dropout and a three-class output layer. The compact search varies hidden-state width and dropout, uses validation macro F1 for early stopping and model selection, and keeps the training-only embeddings frozen. The loss remains unweighted so class weighting can be evaluated separately; the test partition is not scored.

In [54]:
rnn_config = CONFIG["models"]["rnn"]
RNN_MAX_SEQUENCE_LENGTH = int(rnn_config["max_sequence_length"])
if RNN_MAX_SEQUENCE_LENGTH != CNN_MAX_SEQUENCE_LENGTH:
    raise ValueError("CNN and RNN sequence lengths must match.")

RNN_SEQUENCE_LENGTHS = {
    partition_name: np.maximum(
        (sequences != CNN_PADDING_INDEX).sum(axis=1), 1
    ).astype(np.int64)
    for partition_name, sequences in CNN_SEQUENCE_MATRICES.items()
}


class Word2VecVanillaRNN(nn.Module):
    """Single-layer tanh RNN over pretrained Word2Vec embeddings."""

    def __init__(
        self,
        embedding_weights: np.ndarray,
        hidden_size: int,
        dropout: float,
        freeze_embeddings: bool,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.from_numpy(embedding_weights.copy()),
            freeze=freeze_embeddings,
            padding_idx=CNN_PADDING_INDEX,
        )
        self.recurrent = nn.RNN(
            input_size=embedding_weights.shape[1],
            hidden_size=hidden_size,
            nonlinearity="tanh",
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, len(SENTIMENT_LABELS))

    def forward(
        self, token_ids: torch.Tensor, sequence_lengths: torch.Tensor
    ) -> torch.Tensor:
        embedded = self.embedding(token_ids)
        recurrent_output, _ = self.recurrent(embedded)
        positions = torch.arange(
            recurrent_output.shape[1], device=token_ids.device
        ).unsqueeze(0)
        final_positions = sequence_lengths.to(token_ids.device).unsqueeze(1) - 1
        final_state_mask = positions.eq(final_positions).unsqueeze(2)
        final_hidden = (recurrent_output * final_state_mask).sum(dim=1)
        return self.classifier(self.dropout(final_hidden))


rnn_length_audit = pd.DataFrame(
    {
        partition_name: {
            "minimum_retained_length": int(lengths.min()),
            "median_retained_length": float(np.median(lengths)),
            "maximum_retained_length": int(lengths.max()),
        }
        for partition_name, lengths in RNN_SEQUENCE_LENGTHS.items()
    }
).T
rnn_length_audit.index.name = "partition"
display(rnn_length_audit)

,minimum_retained_length,median_retained_length,maximum_retained_length
partition,,,
train,1.0,13.0,64.0
validation,1.0,13.0,64.0
test,1.0,13.0,52.0


In [55]:
def make_rnn_loader(
    partition_name: str,
    batch_size: int,
    shuffle: bool,
    seed: int,
) -> DataLoader:
    """Build a deterministic sequence-and-length loader."""
    dataset = TensorDataset(
        torch.from_numpy(CNN_SEQUENCE_MATRICES[partition_name]),
        torch.from_numpy(RNN_SEQUENCE_LENGTHS[partition_name]),
        torch.from_numpy(CNN_LABEL_ARRAYS[partition_name]),
    )
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=int(CONFIG["runtime"]["num_workers"]),
        generator=generator,
    )


def predict_rnn(model: nn.Module, partition_name: str, batch_size: int) -> np.ndarray:
    """Predict configured sentiment labels in fixed row order."""
    model.eval()
    predicted_indices = []
    loader = make_rnn_loader(partition_name, batch_size, shuffle=False, seed=SEED)
    with torch.no_grad():
        for token_ids, sequence_lengths, _ in loader:
            logits = model(token_ids.to(DEVICE), sequence_lengths)
            predicted_indices.extend(logits.argmax(dim=1).cpu().tolist())
    return np.asarray([SENTIMENT_LABELS[index] for index in predicted_indices])


def train_word2vec_rnn(
    parameters: dict[str, object],
) -> tuple[dict[str, torch.Tensor], pd.DataFrame, dict[str, float], np.ndarray]:
    """Train one vanilla RNN candidate with validation-based early stopping."""
    reset_neural_random_state(SEED)
    model = Word2VecVanillaRNN(
        CNN_EMBEDDING_WEIGHTS,
        hidden_size=int(parameters["hidden_size"]),
        dropout=float(parameters["dropout"]),
        freeze_embeddings=bool(rnn_config["freeze_embeddings"]),
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=float(parameters["learning_rate"]),
        weight_decay=float(rnn_config["weight_decay"]),
    )
    loss_function = nn.CrossEntropyLoss()
    batch_size = int(rnn_config["batch_size"])
    training_loader = make_rnn_loader("train", batch_size, True, SEED)
    best_macro_f1 = -np.inf
    best_state = None
    epochs_without_improvement = 0
    history_rows = []

    for epoch in range(1, int(rnn_config["max_epochs"]) + 1):
        model.train()
        total_loss = 0.0
        observed_rows = 0
        for token_ids, sequence_lengths, labels in training_loader:
            token_ids = token_ids.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(token_ids, sequence_lengths)
            loss = loss_function(logits, labels)
            loss.backward()
            optimizer.step()
            batch_rows = len(labels)
            total_loss += float(loss.detach().cpu()) * batch_rows
            observed_rows += batch_rows

        validation_predictions = predict_rnn(model, "validation", batch_size)
        validation_metrics = aggregate_classification_metrics(
            TFIDF_TARGETS["validation"], validation_predictions
        )
        history_rows.append(
            {
                "epoch": epoch,
                "training_loss": total_loss / observed_rows,
                **validation_metrics,
            }
        )
        if validation_metrics["f1_macro"] > best_macro_f1 + 1e-12:
            best_macro_f1 = validation_metrics["f1_macro"]
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= int(rnn_config["patience"]):
                break

    if best_state is None:
        raise RuntimeError("RNN training did not produce a valid checkpoint.")
    model.load_state_dict(best_state)
    selected_predictions = predict_rnn(model, "validation", batch_size)
    selected_metrics = aggregate_classification_metrics(
        TFIDF_TARGETS["validation"], selected_predictions
    )
    return (
        best_state,
        pd.DataFrame(history_rows),
        selected_metrics,
        selected_predictions,
    )

In [56]:
rnn_parameter_candidates = list(ParameterGrid(rnn_config["search_space"]))
if not rnn_parameter_candidates:
    raise ValueError("The RNN search space is empty.")

rnn_tuning_rows = []
RNN_TRIAL_ARTIFACTS = {}
for trial_number, parameters in enumerate(rnn_parameter_candidates, start=1):
    model_state, history, metrics, validation_predictions = train_word2vec_rnn(
        parameters
    )
    best_epoch = int(history.loc[history["f1_macro"].idxmax(), "epoch"])
    rnn_tuning_rows.append(
        {
            "trial": trial_number,
            **parameters,
            "best_epoch": best_epoch,
            "epochs_ran": len(history),
            **metrics,
        }
    )
    RNN_TRIAL_ARTIFACTS[trial_number] = {
        "state_dict": model_state,
        "history": history,
        "validation_predictions": validation_predictions,
    }

rnn_tuning_results = (
    pd.DataFrame(rnn_tuning_rows)
    .sort_values(
        ["f1_macro", "f1_weighted", "accuracy", "trial"],
        ascending=[False, False, False, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)
best_rnn_trial = int(rnn_tuning_results.iloc[0]["trial"])
BEST_WORD2VEC_RNN_PARAMETERS = dict(rnn_parameter_candidates[best_rnn_trial - 1])
BEST_WORD2VEC_RNN_STATE = RNN_TRIAL_ARTIFACTS[best_rnn_trial]["state_dict"]
WORD2VEC_RNN_SELECTED_HISTORY = RNN_TRIAL_ARTIFACTS[best_rnn_trial]["history"]
WORD2VEC_RNN_VALIDATION_PREDICTIONS = RNN_TRIAL_ARTIFACTS[best_rnn_trial][
    "validation_predictions"
]

display(rnn_tuning_results.round(4))
print(f"Selected parameters: {BEST_WORD2VEC_RNN_PARAMETERS}")

,trial,dropout,hidden_size,learning_rate,best_epoch,epochs_ran,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,2,0.3,128,0.001,4,6,0.6454,0.6303,0.6163,0.6177,0.6503,0.6454,0.6414
1,1,0.3,64,0.001,4,6,0.6506,0.6425,0.6064,0.6140,0.6539,0.6506,0.6433
2,3,0.5,64,0.001,4,6,0.6441,0.6394,0.6112,0.6135,0.6560,0.6441,0.6369
3,4,0.5,128,0.001,1,3,0.6402,0.6256,0.5926,0.6019,0.6387,0.6402,0.6344


Selected parameters: {'dropout': 0.3, 'hidden_size': 128, 'learning_rate': 0.001}


In [57]:
word2vec_rnn_validation_metrics = aggregate_classification_metrics(
    TFIDF_TARGETS["validation"], WORD2VEC_RNN_VALIDATION_PREDICTIONS
)
word2vec_rnn_validation_per_class = per_class_metrics(
    TFIDF_TARGETS["validation"], WORD2VEC_RNN_VALIDATION_PREDICTIONS
)
word2vec_rnn_validation_confusion = pd.DataFrame(
    confusion_matrix(
        TFIDF_TARGETS["validation"],
        WORD2VEC_RNN_VALIDATION_PREDICTIONS,
        labels=SENTIMENT_LABELS,
    ),
    index=pd.Index(SENTIMENT_LABELS, name="actual"),
    columns=pd.Index(SENTIMENT_LABELS, name="predicted"),
)

display(pd.Series(word2vec_rnn_validation_metrics, name="value").to_frame().round(4))
display(word2vec_rnn_validation_per_class.round(4))

figure, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.lineplot(
    data=WORD2VEC_RNN_SELECTED_HISTORY,
    x="epoch",
    y="training_loss",
    marker="o",
    ax=axes[0],
)
axes[0].set_title("Selected vanilla RNN training loss")
axes[0].set_ylabel("Cross-entropy loss")
sns.heatmap(
    word2vec_rnn_validation_confusion,
    annot=True,
    fmt="d",
    cmap="Purples",
    cbar=False,
    ax=axes[1],
)
axes[1].set_title("Word2Vec + vanilla RNN validation confusion matrix")
figure.tight_layout()
plt.show()

,value
accuracy,0.6454
precision_macro,0.6303
recall_macro,0.6163
f1_macro,0.6177
precision_weighted,0.6503
recall_weighted,0.6454
f1_weighted,0.6414


,precision,recall,f1,support
sentiment,,,,
NEG,0.6297,0.7814,0.6974,1249
NEU,0.7142,0.5820,0.6414,1292
POS,0.5471,0.4855,0.5145,550


/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/3903233427.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [58]:
rnn_tuning_path = tuning_results_directory / "word2vec_rnn.csv"
rnn_history_path = tuning_results_directory / "word2vec_rnn_selected_history.csv"
rnn_validation_path = validation_results_directory / "word2vec_rnn.json"
rnn_tuning_results.to_csv(rnn_tuning_path, index=False)
WORD2VEC_RNN_SELECTED_HISTORY.to_csv(rnn_history_path, index=False)

rnn_artifact_directory = PATHS["artifacts"] / "models"
rnn_artifact_directory.mkdir(parents=True, exist_ok=True)
rnn_model_path = rnn_artifact_directory / "word2vec_rnn.pt"
torch.save(
    {
        "state_dict": BEST_WORD2VEC_RNN_STATE,
        "parameters": BEST_WORD2VEC_RNN_PARAMETERS,
        "max_sequence_length": RNN_MAX_SEQUENCE_LENGTH,
        "label_order": SENTIMENT_LABELS,
    },
    rnn_model_path,
)

rnn_validation_result = {
    "experiment_id": "word2vec_rnn_validation",
    "representation": "word2vec_sequence",
    "classifier": "vanilla_rnn",
    "seed": SEED,
    "device": str(DEVICE),
    "selection_metric": CONFIG["evaluation"]["primary_metric"],
    "selected_parameters": BEST_WORD2VEC_RNN_PARAMETERS,
    "selected_epoch": int(rnn_tuning_results.iloc[0]["best_epoch"]),
    "training_rows": len(PROCESSED_PARTITIONS["train"]),
    "validation_rows": len(PROCESSED_PARTITIONS["validation"]),
    "aggregate_metrics": word2vec_rnn_validation_metrics,
    "per_class_metrics": word2vec_rnn_validation_per_class.to_dict(orient="index"),
    "confusion_matrix": word2vec_rnn_validation_confusion.to_dict(orient="index"),
    "test_evaluated": False,
    "class_weighted": False,
}
with rnn_validation_path.open("w", encoding="utf-8") as result_file:
    json.dump(
        rnn_validation_result,
        result_file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    result_file.write("\n")

pd.DataFrame(
    {
        "result": [
            str(rnn_tuning_path.relative_to(PROJECT_ROOT)),
            str(rnn_history_path.relative_to(PROJECT_ROOT)),
            str(rnn_validation_path.relative_to(PROJECT_ROOT)),
            str(rnn_model_path.relative_to(PROJECT_ROOT)),
        ]
    }
)

,result
0,results/tuning/word2vec_rnn.csv
1,results/tuning/word2vec_rnn_selected_history.csv
2,results/validation/word2vec_rnn.json
3,artifacts/models/word2vec_rnn.pt


### Word2Vec vanilla RNN validation notes

The selected vanilla RNN checkpoint is the epoch with the strongest validation macro F1 within the best hyperparameter trial. Length-aware final-state selection prevents right-padding from changing the recurrent summary. Frozen training-only embeddings and unweighted cross-entropy define the baseline; class weighting will be tested separately. The test partition remains unevaluated.

### 8.7 Word2Vec with LSTM

Reuse the audited Word2Vec sequences and retained lengths from the recurrent baseline. A single-layer LSTM summarizes ordered token embeddings with gated memory, followed by dropout and a three-class output layer. The compact search matches the vanilla RNN hidden-size and dropout grid, uses validation macro F1 for early stopping and selection, and keeps the training-only embeddings frozen. The loss remains unweighted so class weighting can be evaluated separately; the test partition is not scored.

In [59]:
lstm_config = CONFIG["models"]["lstm"]
LSTM_MAX_SEQUENCE_LENGTH = int(lstm_config["max_sequence_length"])
if LSTM_MAX_SEQUENCE_LENGTH != RNN_MAX_SEQUENCE_LENGTH:
    raise ValueError("RNN and LSTM sequence lengths must match.")


class Word2VecLSTM(nn.Module):
    """Single-layer LSTM over pretrained Word2Vec embeddings."""

    def __init__(
        self,
        embedding_weights: np.ndarray,
        hidden_size: int,
        dropout: float,
        freeze_embeddings: bool,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.from_numpy(embedding_weights.copy()),
            freeze=freeze_embeddings,
            padding_idx=CNN_PADDING_INDEX,
        )
        self.recurrent = nn.LSTM(
            input_size=embedding_weights.shape[1],
            hidden_size=hidden_size,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, len(SENTIMENT_LABELS))

    def forward(
        self, token_ids: torch.Tensor, sequence_lengths: torch.Tensor
    ) -> torch.Tensor:
        embedded = self.embedding(token_ids)
        recurrent_output, _ = self.recurrent(embedded)
        positions = torch.arange(
            recurrent_output.shape[1], device=token_ids.device
        ).unsqueeze(0)
        final_positions = sequence_lengths.to(token_ids.device).unsqueeze(1) - 1
        final_state_mask = positions.eq(final_positions).unsqueeze(2)
        final_hidden = (recurrent_output * final_state_mask).sum(dim=1)
        return self.classifier(self.dropout(final_hidden))

In [60]:
def predict_lstm(
    model: nn.Module, partition_name: str, batch_size: int
) -> np.ndarray:
    """Predict configured sentiment labels in fixed row order."""
    model.eval()
    predicted_indices = []
    loader = make_rnn_loader(partition_name, batch_size, shuffle=False, seed=SEED)
    with torch.no_grad():
        for token_ids, sequence_lengths, _ in loader:
            logits = model(token_ids.to(DEVICE), sequence_lengths)
            predicted_indices.extend(logits.argmax(dim=1).cpu().tolist())
    return np.asarray([SENTIMENT_LABELS[index] for index in predicted_indices])


def train_word2vec_lstm(
    parameters: dict[str, object],
) -> tuple[dict[str, torch.Tensor], pd.DataFrame, dict[str, float], np.ndarray]:
    """Train one LSTM candidate with validation-based early stopping."""
    reset_neural_random_state(SEED)
    model = Word2VecLSTM(
        CNN_EMBEDDING_WEIGHTS,
        hidden_size=int(parameters["hidden_size"]),
        dropout=float(parameters["dropout"]),
        freeze_embeddings=bool(lstm_config["freeze_embeddings"]),
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=float(parameters["learning_rate"]),
        weight_decay=float(lstm_config["weight_decay"]),
    )
    loss_function = nn.CrossEntropyLoss()
    batch_size = int(lstm_config["batch_size"])
    training_loader = make_rnn_loader("train", batch_size, True, SEED)
    best_macro_f1 = -np.inf
    best_state = None
    epochs_without_improvement = 0
    history_rows = []

    for epoch in range(1, int(lstm_config["max_epochs"]) + 1):
        model.train()
        total_loss = 0.0
        observed_rows = 0
        for token_ids, sequence_lengths, labels in training_loader:
            token_ids = token_ids.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(token_ids, sequence_lengths)
            loss = loss_function(logits, labels)
            loss.backward()
            optimizer.step()
            batch_rows = len(labels)
            total_loss += float(loss.detach().cpu()) * batch_rows
            observed_rows += batch_rows

        validation_predictions = predict_lstm(model, "validation", batch_size)
        validation_metrics = aggregate_classification_metrics(
            TFIDF_TARGETS["validation"], validation_predictions
        )
        history_rows.append(
            {
                "epoch": epoch,
                "training_loss": total_loss / observed_rows,
                **validation_metrics,
            }
        )
        if validation_metrics["f1_macro"] > best_macro_f1 + 1e-12:
            best_macro_f1 = validation_metrics["f1_macro"]
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= int(lstm_config["patience"]):
                break

    if best_state is None:
        raise RuntimeError("LSTM training did not produce a valid checkpoint.")
    model.load_state_dict(best_state)
    selected_predictions = predict_lstm(model, "validation", batch_size)
    selected_metrics = aggregate_classification_metrics(
        TFIDF_TARGETS["validation"], selected_predictions
    )
    return (
        best_state,
        pd.DataFrame(history_rows),
        selected_metrics,
        selected_predictions,
    )

In [61]:
lstm_parameter_candidates = list(ParameterGrid(lstm_config["search_space"]))
if not lstm_parameter_candidates:
    raise ValueError("The LSTM search space is empty.")

lstm_tuning_rows = []
LSTM_TRIAL_ARTIFACTS = {}
for trial_number, parameters in enumerate(lstm_parameter_candidates, start=1):
    model_state, history, metrics, validation_predictions = train_word2vec_lstm(
        parameters
    )
    best_epoch = int(history.loc[history["f1_macro"].idxmax(), "epoch"])
    lstm_tuning_rows.append(
        {
            "trial": trial_number,
            **parameters,
            "best_epoch": best_epoch,
            "epochs_ran": len(history),
            **metrics,
        }
    )
    LSTM_TRIAL_ARTIFACTS[trial_number] = {
        "state_dict": model_state,
        "history": history,
        "validation_predictions": validation_predictions,
    }

lstm_tuning_results = (
    pd.DataFrame(lstm_tuning_rows)
    .sort_values(
        ["f1_macro", "f1_weighted", "accuracy", "trial"],
        ascending=[False, False, False, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)
best_lstm_trial = int(lstm_tuning_results.iloc[0]["trial"])
BEST_WORD2VEC_LSTM_PARAMETERS = dict(
    lstm_parameter_candidates[best_lstm_trial - 1]
)
BEST_WORD2VEC_LSTM_STATE = LSTM_TRIAL_ARTIFACTS[best_lstm_trial]["state_dict"]
WORD2VEC_LSTM_SELECTED_HISTORY = LSTM_TRIAL_ARTIFACTS[best_lstm_trial]["history"]
WORD2VEC_LSTM_VALIDATION_PREDICTIONS = LSTM_TRIAL_ARTIFACTS[best_lstm_trial][
    "validation_predictions"
]

display(lstm_tuning_results.round(4))
print(f"Selected parameters: {BEST_WORD2VEC_LSTM_PARAMETERS}")

,trial,dropout,hidden_size,learning_rate,best_epoch,epochs_ran,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,3,0.5,64,0.001,8,10,0.6571,0.6405,0.6265,0.6296,0.6598,0.6571,0.6542
1,4,0.5,128,0.001,2,4,0.6538,0.6420,0.5998,0.6096,0.6507,0.6538,0.6453
2,1,0.3,64,0.001,2,4,0.6512,0.6376,0.6001,0.6095,0.6478,0.6512,0.6437
3,2,0.3,128,0.001,2,4,0.6496,0.6387,0.5934,0.6036,0.6461,0.6496,0.6406


Selected parameters: {'dropout': 0.5, 'hidden_size': 64, 'learning_rate': 0.001}


In [62]:
word2vec_lstm_validation_metrics = aggregate_classification_metrics(
    TFIDF_TARGETS["validation"], WORD2VEC_LSTM_VALIDATION_PREDICTIONS
)
word2vec_lstm_validation_per_class = per_class_metrics(
    TFIDF_TARGETS["validation"], WORD2VEC_LSTM_VALIDATION_PREDICTIONS
)
word2vec_lstm_validation_confusion = pd.DataFrame(
    confusion_matrix(
        TFIDF_TARGETS["validation"],
        WORD2VEC_LSTM_VALIDATION_PREDICTIONS,
        labels=SENTIMENT_LABELS,
    ),
    index=pd.Index(SENTIMENT_LABELS, name="actual"),
    columns=pd.Index(SENTIMENT_LABELS, name="predicted"),
)

display(
    pd.Series(word2vec_lstm_validation_metrics, name="value").to_frame().round(4)
)
display(word2vec_lstm_validation_per_class.round(4))

figure, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.lineplot(
    data=WORD2VEC_LSTM_SELECTED_HISTORY,
    x="epoch",
    y="training_loss",
    marker="o",
    ax=axes[0],
)
axes[0].set_title("Selected LSTM training loss")
axes[0].set_ylabel("Cross-entropy loss")
sns.heatmap(
    word2vec_lstm_validation_confusion,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    ax=axes[1],
)
axes[1].set_title("Word2Vec + LSTM validation confusion matrix")
figure.tight_layout()
plt.show()

,value
accuracy,0.6571
precision_macro,0.6405
recall_macro,0.6265
f1_macro,0.6296
precision_weighted,0.6598
recall_weighted,0.6571
f1_weighted,0.6542


,precision,recall,f1,support
sentiment,,,,
NEG,0.6417,0.7686,0.6995,1249
NEU,0.7197,0.6200,0.6661,1292
POS,0.5602,0.4909,0.5233,550


/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/2619389557.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [63]:
lstm_tuning_path = tuning_results_directory / "word2vec_lstm.csv"
lstm_history_path = (
    tuning_results_directory / "word2vec_lstm_selected_history.csv"
)
lstm_validation_path = validation_results_directory / "word2vec_lstm.json"
lstm_tuning_results.to_csv(lstm_tuning_path, index=False)
WORD2VEC_LSTM_SELECTED_HISTORY.to_csv(lstm_history_path, index=False)

lstm_artifact_directory = PATHS["artifacts"] / "models"
lstm_artifact_directory.mkdir(parents=True, exist_ok=True)
lstm_model_path = lstm_artifact_directory / "word2vec_lstm.pt"
torch.save(
    {
        "state_dict": BEST_WORD2VEC_LSTM_STATE,
        "parameters": BEST_WORD2VEC_LSTM_PARAMETERS,
        "max_sequence_length": LSTM_MAX_SEQUENCE_LENGTH,
        "label_order": SENTIMENT_LABELS,
    },
    lstm_model_path,
)

lstm_validation_result = {
    "experiment_id": "word2vec_lstm_validation",
    "representation": "word2vec_sequence",
    "classifier": "lstm",
    "seed": SEED,
    "device": str(DEVICE),
    "selection_metric": CONFIG["evaluation"]["primary_metric"],
    "selected_parameters": BEST_WORD2VEC_LSTM_PARAMETERS,
    "selected_epoch": int(lstm_tuning_results.iloc[0]["best_epoch"]),
    "training_rows": len(PROCESSED_PARTITIONS["train"]),
    "validation_rows": len(PROCESSED_PARTITIONS["validation"]),
    "aggregate_metrics": word2vec_lstm_validation_metrics,
    "per_class_metrics": word2vec_lstm_validation_per_class.to_dict(orient="index"),
    "confusion_matrix": word2vec_lstm_validation_confusion.to_dict(orient="index"),
    "test_evaluated": False,
    "class_weighted": False,
}
with lstm_validation_path.open("w", encoding="utf-8") as result_file:
    json.dump(
        lstm_validation_result,
        result_file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    result_file.write("\n")

pd.DataFrame(
    {
        "result": [
            str(lstm_tuning_path.relative_to(PROJECT_ROOT)),
            str(lstm_history_path.relative_to(PROJECT_ROOT)),
            str(lstm_validation_path.relative_to(PROJECT_ROOT)),
            str(lstm_model_path.relative_to(PROJECT_ROOT)),
        ]
    }
)

,result
0,results/tuning/word2vec_lstm.csv
1,results/tuning/word2vec_lstm_selected_history.csv
2,results/validation/word2vec_lstm.json
3,artifacts/models/word2vec_lstm.pt


### Word2Vec LSTM validation notes

The selected LSTM checkpoint is the epoch with the strongest validation macro F1 within the best hyperparameter trial. Length-aware final-state selection prevents right-padding from changing the recurrent summary. Frozen training-only embeddings and unweighted cross-entropy define the baseline; class weighting will be tested separately. The test partition remains unevaluated.

### 8.8 FastText with Gaussian Naive Bayes

Apply the same Gaussian Naive Bayes variance-smoothing search to the dense mean FastText document vectors. Keeping the candidate grid, empirical priors, split, ranking metrics, and deterministic tie-breaks identical to the Word2Vec experiment isolates the representation change. The test partition remains unscored.

In [64]:
fasttext_gnb_search_space = CONFIG["models"]["gaussian_nb"]["search_space"]
fasttext_gnb_parameter_candidates = list(
    ParameterGrid(fasttext_gnb_search_space)
)
if not fasttext_gnb_parameter_candidates:
    raise ValueError("The FastText Gaussian Naive Bayes search space is empty.")

fasttext_gnb_tuning_rows = []
for trial_number, parameters in enumerate(
    fasttext_gnb_parameter_candidates, start=1
):
    candidate_model = GaussianNB(**parameters)
    candidate_model.fit(FASTTEXT_DOCUMENT_MATRICES["train"], TFIDF_TARGETS["train"])
    validation_predictions = candidate_model.predict(
        FASTTEXT_DOCUMENT_MATRICES["validation"]
    )
    fasttext_gnb_tuning_rows.append(
        {
            "trial": trial_number,
            **parameters,
            **aggregate_classification_metrics(
                TFIDF_TARGETS["validation"], validation_predictions
            ),
        }
    )

fasttext_gnb_tuning_results = (
    pd.DataFrame(fasttext_gnb_tuning_rows)
    .sort_values(
        ["f1_macro", "f1_weighted", "accuracy", "trial"],
        ascending=[False, False, False, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)
best_fasttext_gnb_trial = int(fasttext_gnb_tuning_results.iloc[0]["trial"])
BEST_FASTTEXT_GNB_PARAMETERS = dict(
    fasttext_gnb_parameter_candidates[best_fasttext_gnb_trial - 1]
)
BEST_FASTTEXT_GNB = GaussianNB(**BEST_FASTTEXT_GNB_PARAMETERS).fit(
    FASTTEXT_DOCUMENT_MATRICES["train"], TFIDF_TARGETS["train"]
)
FASTTEXT_GNB_VALIDATION_PREDICTIONS = BEST_FASTTEXT_GNB.predict(
    FASTTEXT_DOCUMENT_MATRICES["validation"]
)

display(fasttext_gnb_tuning_results.round(6))
print(f"Selected parameters: {BEST_FASTTEXT_GNB_PARAMETERS}")

,trial,var_smoothing,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,1,0.000001,0.554837,0.568894,0.516586,0.505398,0.601832,0.554837,0.533496
1,2,0.000010,0.554837,0.568894,0.516586,0.505398,0.601832,0.554837,0.533496
2,3,0.000100,0.554837,0.568894,0.516586,0.505398,0.601832,0.554837,0.533496
3,4,0.001000,0.554190,0.568152,0.515722,0.504453,0.601226,0.554190,0.532715
4,5,0.010000,0.553219,0.568484,0.513957,0.502218,0.601395,0.553219,0.530643
5,6,0.100000,0.531543,0.576845,0.493269,0.472063,0.607024,0.531543,0.494377
6,7,1.000000,0.449369,0.598311,0.402512,0.324628,0.631821,0.449369,0.337701


Selected parameters: {'var_smoothing': 1e-06}


In [65]:
fasttext_gnb_validation_metrics = aggregate_classification_metrics(
    TFIDF_TARGETS["validation"], FASTTEXT_GNB_VALIDATION_PREDICTIONS
)
fasttext_gnb_validation_per_class = per_class_metrics(
    TFIDF_TARGETS["validation"], FASTTEXT_GNB_VALIDATION_PREDICTIONS
)
fasttext_gnb_validation_confusion = pd.DataFrame(
    confusion_matrix(
        TFIDF_TARGETS["validation"],
        FASTTEXT_GNB_VALIDATION_PREDICTIONS,
        labels=SENTIMENT_LABELS,
    ),
    index=pd.Index(SENTIMENT_LABELS, name="actual"),
    columns=pd.Index(SENTIMENT_LABELS, name="predicted"),
)

display(
    pd.Series(fasttext_gnb_validation_metrics, name="value").to_frame().round(4)
)
display(fasttext_gnb_validation_per_class.round(4))

figure, axis = plt.subplots(figsize=(6, 5))
sns.heatmap(
    fasttext_gnb_validation_confusion,
    annot=True,
    fmt="d",
    cmap="Greens",
    cbar=False,
    ax=axis,
)
axis.set_title("FastText + Gaussian NB validation confusion matrix")
figure.tight_layout()
plt.show()

,value
accuracy,0.5548
precision_macro,0.5689
recall_macro,0.5166
f1_macro,0.5054
precision_weighted,0.6018
recall_weighted,0.5548
f1_weighted,0.5335


,precision,recall,f1,support
sentiment,,,,
NEG,0.5189,0.8455,0.6431,1249
NEU,0.7532,0.3661,0.4927,1292
POS,0.4346,0.3382,0.3804,550


/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/2294431351.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [66]:
fasttext_gnb_tuning_path = tuning_results_directory / "fasttext_gaussian_nb.csv"
fasttext_gnb_tuning_results.to_csv(fasttext_gnb_tuning_path, index=False)
fasttext_gnb_validation_path = (
    validation_results_directory / "fasttext_gaussian_nb.json"
)
fasttext_gnb_validation_result = {
    "experiment_id": "fasttext_gaussian_nb_validation",
    "representation": "fasttext_mean",
    "classifier": "gaussian_naive_bayes",
    "seed": SEED,
    "selection_metric": CONFIG["evaluation"]["primary_metric"],
    "selected_parameters": BEST_FASTTEXT_GNB_PARAMETERS,
    "training_rows": len(PROCESSED_PARTITIONS["train"]),
    "validation_rows": len(PROCESSED_PARTITIONS["validation"]),
    "feature_count": FASTTEXT_DOCUMENT_MATRICES["train"].shape[1],
    "aggregate_metrics": fasttext_gnb_validation_metrics,
    "per_class_metrics": fasttext_gnb_validation_per_class.to_dict(orient="index"),
    "confusion_matrix": fasttext_gnb_validation_confusion.to_dict(orient="index"),
    "test_evaluated": False,
}
with fasttext_gnb_validation_path.open("w", encoding="utf-8") as result_file:
    json.dump(
        fasttext_gnb_validation_result,
        result_file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    result_file.write("\n")

pd.DataFrame(
    {
        "result": [
            str(fasttext_gnb_tuning_path.relative_to(PROJECT_ROOT)),
            str(fasttext_gnb_validation_path.relative_to(PROJECT_ROOT)),
        ]
    }
)

,result
0,results/tuning/fasttext_gaussian_nb.csv
1,results/validation/fasttext_gaussian_nb.json


### FastText Gaussian Naive Bayes validation notes

This baseline evaluates whether class-conditional Gaussian distributions can separate mean FastText embeddings. Selection and reporting use validation data only, with minority positive-sentiment performance shown explicitly. Test evaluation remains deferred.

### 8.9 FastText with Random Forest

Apply the same unweighted Random Forest search used for TF-IDF and Word2Vec to the dense mean FastText vectors. Holding the candidate grid and deterministic ranking rules constant makes performance differences attributable to the representation. Candidates are fitted on training data and ranked on validation macro F1, followed by weighted F1, accuracy, and configured trial order. The test partition remains unscored.

In [67]:
fasttext_rf_config = CONFIG["models"]["random_forest"]
fasttext_rf_fixed_parameters = {
    **fasttext_rf_config["fixed_parameters"],
    "random_state": SEED,
}
fasttext_rf_parameter_candidates = list(
    ParameterGrid(fasttext_rf_config["search_space"])
)
if not fasttext_rf_parameter_candidates:
    raise ValueError("The FastText Random Forest search space is empty.")

fasttext_rf_tuning_rows = []
for trial_number, parameters in enumerate(
    fasttext_rf_parameter_candidates, start=1
):
    candidate_model = RandomForestClassifier(
        **fasttext_rf_fixed_parameters,
        **parameters,
    )
    candidate_model.fit(FASTTEXT_DOCUMENT_MATRICES["train"], TFIDF_TARGETS["train"])
    validation_predictions = candidate_model.predict(
        FASTTEXT_DOCUMENT_MATRICES["validation"]
    )
    fasttext_rf_tuning_rows.append(
        {
            "trial": trial_number,
            **parameters,
            **aggregate_classification_metrics(
                TFIDF_TARGETS["validation"], validation_predictions
            ),
        }
    )

fasttext_rf_tuning_results = (
    pd.DataFrame(fasttext_rf_tuning_rows)
    .sort_values(
        ["f1_macro", "f1_weighted", "accuracy", "trial"],
        ascending=[False, False, False, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)
best_fasttext_rf_trial = int(fasttext_rf_tuning_results.iloc[0]["trial"])
BEST_FASTTEXT_RF_PARAMETERS = dict(
    fasttext_rf_parameter_candidates[best_fasttext_rf_trial - 1]
)
BEST_FASTTEXT_RF = RandomForestClassifier(
    **fasttext_rf_fixed_parameters,
    **BEST_FASTTEXT_RF_PARAMETERS,
).fit(FASTTEXT_DOCUMENT_MATRICES["train"], TFIDF_TARGETS["train"])
FASTTEXT_RF_VALIDATION_PREDICTIONS = BEST_FASTTEXT_RF.predict(
    FASTTEXT_DOCUMENT_MATRICES["validation"]
)

display(fasttext_rf_tuning_results.round(4))
print(f"Selected parameters: {BEST_FASTTEXT_RF_PARAMETERS}")

,trial,max_depth,min_samples_leaf,n_estimators,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,3,NaN,2,200,0.6561,0.6581,0.5894,0.5986,0.6593,0.6561,0.6421
1,7,40.0,2,200,0.6564,0.6566,0.5893,0.5982,0.6590,0.6564,0.6424
2,2,NaN,1,400,0.6551,0.6532,0.5887,0.5973,0.6569,0.6551,0.6412
3,6,40.0,1,400,0.6551,0.6512,0.5887,0.5971,0.6560,0.6551,0.6413
4,5,40.0,1,200,0.6558,0.6542,0.5885,0.5970,0.6575,0.6558,0.6415
5,4,NaN,2,400,0.6555,0.6565,0.5879,0.5966,0.6584,0.6555,0.6410
6,8,40.0,2,400,0.6555,0.6565,0.5879,0.5966,0.6583,0.6555,0.6410
7,1,NaN,1,200,0.6532,0.6513,0.5867,0.5954,0.6548,0.6532,0.6393


Selected parameters: {'max_depth': None, 'min_samples_leaf': 2, 'n_estimators': 200}


In [68]:
fasttext_rf_validation_metrics = aggregate_classification_metrics(
    TFIDF_TARGETS["validation"], FASTTEXT_RF_VALIDATION_PREDICTIONS
)
fasttext_rf_validation_per_class = per_class_metrics(
    TFIDF_TARGETS["validation"], FASTTEXT_RF_VALIDATION_PREDICTIONS
)
fasttext_rf_validation_confusion = pd.DataFrame(
    confusion_matrix(
        TFIDF_TARGETS["validation"],
        FASTTEXT_RF_VALIDATION_PREDICTIONS,
        labels=SENTIMENT_LABELS,
    ),
    index=pd.Index(SENTIMENT_LABELS, name="actual"),
    columns=pd.Index(SENTIMENT_LABELS, name="predicted"),
)

display(
    pd.Series(fasttext_rf_validation_metrics, name="value").to_frame().round(4)
)
display(fasttext_rf_validation_per_class.round(4))

figure, axis = plt.subplots(figsize=(6, 5))
sns.heatmap(
    fasttext_rf_validation_confusion,
    annot=True,
    fmt="d",
    cmap="YlGn",
    cbar=False,
    ax=axis,
)
axis.set_title("FastText + Random Forest validation confusion matrix")
figure.tight_layout()
plt.show()

,value
accuracy,0.6561
precision_macro,0.6581
recall_macro,0.5894
f1_macro,0.5986
precision_weighted,0.6593
recall_weighted,0.6561
f1_weighted,0.6421


,precision,recall,f1,support
sentiment,,,,
NEG,0.6279,0.7918,0.7004,1249
NEU,0.6915,0.6765,0.6839,1292
POS,0.6548,0.3000,0.4115,550


/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/2483282245.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [69]:
fasttext_rf_tuning_path = tuning_results_directory / "fasttext_random_forest.csv"
fasttext_rf_tuning_results.to_csv(fasttext_rf_tuning_path, index=False)
fasttext_rf_validation_path = (
    validation_results_directory / "fasttext_random_forest.json"
)
fasttext_rf_validation_result = {
    "experiment_id": "fasttext_random_forest_validation",
    "representation": "fasttext_mean",
    "classifier": "random_forest",
    "seed": SEED,
    "selection_metric": CONFIG["evaluation"]["primary_metric"],
    "fixed_parameters": fasttext_rf_fixed_parameters,
    "selected_parameters": BEST_FASTTEXT_RF_PARAMETERS,
    "training_rows": len(PROCESSED_PARTITIONS["train"]),
    "validation_rows": len(PROCESSED_PARTITIONS["validation"]),
    "feature_count": FASTTEXT_DOCUMENT_MATRICES["train"].shape[1],
    "aggregate_metrics": fasttext_rf_validation_metrics,
    "per_class_metrics": fasttext_rf_validation_per_class.to_dict(orient="index"),
    "confusion_matrix": fasttext_rf_validation_confusion.to_dict(orient="index"),
    "test_evaluated": False,
}
with fasttext_rf_validation_path.open("w", encoding="utf-8") as result_file:
    json.dump(
        fasttext_rf_validation_result,
        result_file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    result_file.write("\n")

pd.DataFrame(
    {
        "result": [
            str(fasttext_rf_tuning_path.relative_to(PROJECT_ROOT)),
            str(fasttext_rf_validation_path.relative_to(PROJECT_ROOT)),
        ]
    }
)

,result
0,results/tuning/fasttext_random_forest.csv
1,results/validation/fasttext_random_forest.json


### FastText Random Forest validation notes

This unweighted baseline uses the same model search and validation partition as the TF-IDF and Word2Vec Random Forest experiments. The comparison isolates the change to dense mean FastText features. Test evaluation and class weighting remain deferred.

### 8.10-8.12 FastText with CNN, vanilla RNN, and LSTM

Project every token through the frozen training-only FastText model, including held-out word forms represented from learned character n-grams. A deterministic lookup table stores those projected vectors without fitting on validation or test labels. The CNN, vanilla RNN, and LSTM architectures and hyperparameter grids match their Word2Vec counterparts, isolating the embedding representation. Each model uses unweighted cross-entropy, validation macro F1 for early stopping and selection, and leaves the test partition unscored.

In [70]:
fasttext_neural_model_names = ("cnn", "rnn", "lstm")
fasttext_sequence_lengths = {
    name: int(CONFIG["models"][name]["max_sequence_length"])
    for name in fasttext_neural_model_names
}
if len(set(fasttext_sequence_lengths.values())) != 1:
    raise ValueError("FastText neural models require one shared sequence length.")
FASTTEXT_MAX_SEQUENCE_LENGTH = next(iter(fasttext_sequence_lengths.values()))
FASTTEXT_PADDING_INDEX = 0
if FASTTEXT_PADDING_INDEX != CNN_PADDING_INDEX:
    raise ValueError("Shared neural architectures require padding index zero.")

fasttext_exact_tokens = list(FASTTEXT_MODEL.wv.index_to_key)
all_fasttext_tokens = {
    token
    for documents in FASTTEXT_TOKENIZED_PARTITIONS.values()
    for tokens in documents
    for token in tokens
}
fasttext_inferred_tokens = sorted(all_fasttext_tokens - set(fasttext_exact_tokens))
fasttext_sequence_tokens = fasttext_exact_tokens + fasttext_inferred_tokens
FASTTEXT_TOKEN_TO_INDEX = {
    token: index + 1 for index, token in enumerate(fasttext_sequence_tokens)
}
FASTTEXT_EMBEDDING_WEIGHTS = np.zeros(
    (len(FASTTEXT_TOKEN_TO_INDEX) + 1, FASTTEXT_MODEL.vector_size),
    dtype=np.float32,
)
FASTTEXT_EMBEDDING_WEIGHTS[1:] = np.vstack(
    [FASTTEXT_MODEL.wv[token] for token in fasttext_sequence_tokens]
).astype(np.float32, copy=False)


def encode_fasttext_sequence(tokens: list[str]) -> np.ndarray:
    """Map tokens to a padded sequence of frozen FastText vector IDs."""
    encoded = np.full(
        FASTTEXT_MAX_SEQUENCE_LENGTH, FASTTEXT_PADDING_INDEX, dtype=np.int64
    )
    retained_tokens = tokens[:FASTTEXT_MAX_SEQUENCE_LENGTH]
    encoded[: len(retained_tokens)] = [
        FASTTEXT_TOKEN_TO_INDEX[token] for token in retained_tokens
    ]
    return encoded


FASTTEXT_SEQUENCE_MATRICES = {
    partition_name: np.vstack(
        [encode_fasttext_sequence(tokens) for tokens in tokenized_documents]
    )
    for partition_name, tokenized_documents in FASTTEXT_TOKENIZED_PARTITIONS.items()
}
FASTTEXT_SEQUENCE_LENGTHS = {
    partition_name: np.maximum(
        (sequences != FASTTEXT_PADDING_INDEX).sum(axis=1), 1
    ).astype(np.int64)
    for partition_name, sequences in FASTTEXT_SEQUENCE_MATRICES.items()
}

In [71]:
fasttext_sequence_audit_rows = []
for partition_name, tokenized_documents in FASTTEXT_TOKENIZED_PARTITIONS.items():
    sequences = FASTTEXT_SEQUENCE_MATRICES[partition_name]
    expected_shape = (len(tokenized_documents), FASTTEXT_MAX_SEQUENCE_LENGTH)
    if sequences.shape != expected_shape or sequences.dtype != np.int64:
        raise ValueError(f"Invalid FastText sequences for {partition_name}.")
    if sequences.min() < 0 or sequences.max() >= len(FASTTEXT_EMBEDDING_WEIGHTS):
        raise ValueError(f"Out-of-range FastText index in {partition_name}.")

    total_tokens = sum(len(tokens) for tokens in tokenized_documents)
    retained_tokens = sum(
        min(len(tokens), FASTTEXT_MAX_SEQUENCE_LENGTH)
        for tokens in tokenized_documents
    )
    exact_tokens = sum(
        token in fasttext_exact_vocabulary
        for tokens in tokenized_documents
        for token in tokens[:FASTTEXT_MAX_SEQUENCE_LENGTH]
    )
    fasttext_sequence_audit_rows.append(
        {
            "partition": partition_name,
            "rows": len(sequences),
            "sequence_length": FASTTEXT_MAX_SEQUENCE_LENGTH,
            "truncated_rows": sum(
                len(tokens) > FASTTEXT_MAX_SEQUENCE_LENGTH
                for tokens in tokenized_documents
            ),
            "retained_token_percent": 100 * retained_tokens / total_tokens,
            "subword_inferred_percent": (
                100 * (retained_tokens - exact_tokens) / retained_tokens
            ),
        }
    )

fasttext_sequence_audit = pd.DataFrame(fasttext_sequence_audit_rows).set_index(
    "partition"
)
fasttext_neural_embedding_summary = pd.Series(
    {
        "embedding_rows": len(FASTTEXT_EMBEDDING_WEIGHTS),
        "exact_training_tokens": len(fasttext_exact_tokens),
        "subword_inferred_token_types": len(fasttext_inferred_tokens),
        "embedding_dimensions": FASTTEXT_EMBEDDING_WEIGHTS.shape[1],
        "padding_index": FASTTEXT_PADDING_INDEX,
    },
    name="value",
).to_frame()
display(fasttext_neural_embedding_summary)
display(
    fasttext_sequence_audit.round(
        {"retained_token_percent": 2, "subword_inferred_percent": 2}
    )
)

,value
embedding_rows,27482
exact_training_tokens,8255
subword_inferred_token_types,19226
embedding_dimensions,200
padding_index,0


,rows,sequence_length,truncated_rows,retained_token_percent,subword_inferred_percent
partition,,,,,
train,9273,64,3,100.00,9.74
validation,3091,64,1,99.88,14.61
test,3091,64,0,100.00,14.63


In [72]:
def make_fasttext_neural_loader(
    partition_name: str, batch_size: int, shuffle: bool, seed: int
) -> DataLoader:
    """Build a deterministic FastText sequence loader."""
    dataset = TensorDataset(
        torch.from_numpy(FASTTEXT_SEQUENCE_MATRICES[partition_name]),
        torch.from_numpy(FASTTEXT_SEQUENCE_LENGTHS[partition_name]),
        torch.from_numpy(CNN_LABEL_ARRAYS[partition_name]),
    )
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=int(CONFIG["runtime"]["num_workers"]),
        generator=generator,
    )


def build_fasttext_neural_model(
    model_name: str, parameters: dict[str, object]
) -> nn.Module:
    """Instantiate one representation-agnostic neural architecture."""
    model_config = CONFIG["models"][model_name]
    if model_name == "cnn":
        return Word2VecTextCNN(
            FASTTEXT_EMBEDDING_WEIGHTS,
            num_filters=int(parameters["num_filters"]),
            kernel_sizes=[int(size) for size in model_config["kernel_sizes"]],
            dropout=float(parameters["dropout"]),
            freeze_embeddings=bool(model_config["freeze_embeddings"]),
        )
    if model_name == "rnn":
        return Word2VecVanillaRNN(
            FASTTEXT_EMBEDDING_WEIGHTS,
            hidden_size=int(parameters["hidden_size"]),
            dropout=float(parameters["dropout"]),
            freeze_embeddings=bool(model_config["freeze_embeddings"]),
        )
    if model_name == "lstm":
        return Word2VecLSTM(
            FASTTEXT_EMBEDDING_WEIGHTS,
            hidden_size=int(parameters["hidden_size"]),
            dropout=float(parameters["dropout"]),
            freeze_embeddings=bool(model_config["freeze_embeddings"]),
        )
    raise ValueError(f"Unsupported FastText neural model: {model_name}")


def predict_fasttext_neural(
    model: nn.Module, partition_name: str, batch_size: int
) -> np.ndarray:
    """Predict sentiment labels in fixed partition order."""
    model.eval()
    predicted_indices = []
    loader = make_fasttext_neural_loader(
        partition_name, batch_size, shuffle=False, seed=SEED
    )
    with torch.no_grad():
        for token_ids, sequence_lengths, _ in loader:
            token_ids = token_ids.to(DEVICE)
            if isinstance(model, Word2VecTextCNN):
                logits = model(token_ids)
            else:
                logits = model(token_ids, sequence_lengths)
            predicted_indices.extend(logits.argmax(dim=1).cpu().tolist())
    return np.asarray([SENTIMENT_LABELS[index] for index in predicted_indices])


def train_fasttext_neural(
    model_name: str, parameters: dict[str, object], seed: int = SEED
) -> tuple[dict[str, torch.Tensor], pd.DataFrame, dict[str, float], np.ndarray]:
    """Train one FastText neural candidate with validation early stopping."""
    model_config = CONFIG["models"][model_name]
    reset_neural_random_state(seed)
    model = build_fasttext_neural_model(model_name, parameters).to(DEVICE)
    optimizer = torch.optim.AdamW(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=float(parameters["learning_rate"]),
        weight_decay=float(model_config["weight_decay"]),
    )
    loss_function = nn.CrossEntropyLoss()
    batch_size = int(model_config["batch_size"])
    training_loader = make_fasttext_neural_loader(
        "train", batch_size, shuffle=True, seed=seed
    )
    best_macro_f1 = -np.inf
    best_state = None
    epochs_without_improvement = 0
    history_rows = []

    for epoch in range(1, int(model_config["max_epochs"]) + 1):
        model.train()
        total_loss = 0.0
        observed_rows = 0
        for token_ids, sequence_lengths, labels in training_loader:
            token_ids = token_ids.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            if model_name == "cnn":
                logits = model(token_ids)
            else:
                logits = model(token_ids, sequence_lengths)
            loss = loss_function(logits, labels)
            loss.backward()
            optimizer.step()
            batch_rows = len(labels)
            total_loss += float(loss.detach().cpu()) * batch_rows
            observed_rows += batch_rows

        validation_predictions = predict_fasttext_neural(
            model, "validation", batch_size
        )
        validation_metrics = aggregate_classification_metrics(
            TFIDF_TARGETS["validation"], validation_predictions
        )
        history_rows.append(
            {
                "epoch": epoch,
                "training_loss": total_loss / observed_rows,
                **validation_metrics,
            }
        )
        if validation_metrics["f1_macro"] > best_macro_f1 + 1e-12:
            best_macro_f1 = validation_metrics["f1_macro"]
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= int(model_config["patience"]):
                break

    if best_state is None:
        raise RuntimeError(f"{model_name} did not produce a valid checkpoint.")
    model.load_state_dict(best_state)
    selected_predictions = predict_fasttext_neural(
        model, "validation", batch_size
    )
    selected_metrics = aggregate_classification_metrics(
        TFIDF_TARGETS["validation"], selected_predictions
    )
    return (
        best_state,
        pd.DataFrame(history_rows),
        selected_metrics,
        selected_predictions,
    )

In [73]:
FASTTEXT_NEURAL_EXPERIMENTS = {}
for model_name in fasttext_neural_model_names:
    model_config = CONFIG["models"][model_name]
    parameter_candidates = list(ParameterGrid(model_config["search_space"]))
    if not parameter_candidates:
        raise ValueError(f"The FastText {model_name} search space is empty.")

    tuning_rows = []
    trial_artifacts = {}
    for trial_number, parameters in enumerate(parameter_candidates, start=1):
        state, history, metrics, predictions = train_fasttext_neural(
            model_name, parameters
        )
        best_epoch = int(history.loc[history["f1_macro"].idxmax(), "epoch"])
        tuning_rows.append(
            {
                "trial": trial_number,
                **parameters,
                "best_epoch": best_epoch,
                "epochs_ran": len(history),
                **metrics,
            }
        )
        trial_artifacts[trial_number] = {
            "state_dict": state,
            "history": history,
            "predictions": predictions,
        }

    tuning_results = (
        pd.DataFrame(tuning_rows)
        .sort_values(
            ["f1_macro", "f1_weighted", "accuracy", "trial"],
            ascending=[False, False, False, True],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )
    best_trial = int(tuning_results.iloc[0]["trial"])
    selected = trial_artifacts[best_trial]
    FASTTEXT_NEURAL_EXPERIMENTS[model_name] = {
        "parameters": dict(parameter_candidates[best_trial - 1]),
        "selected_epoch": int(tuning_results.iloc[0]["best_epoch"]),
        "state_dict": selected["state_dict"],
        "history": selected["history"],
        "predictions": selected["predictions"],
        "tuning_results": tuning_results,
    }
    print(f"FastText {model_name.upper()}:")
    display(tuning_results.round(4))
    selected_parameters = FASTTEXT_NEURAL_EXPERIMENTS[model_name]["parameters"]
    print(f"Selected parameters: {selected_parameters}")

FastText CNN:


,trial,dropout,learning_rate,num_filters,best_epoch,epochs_ran,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,2,0.3,0.001,128,5,7,0.6610,0.6482,0.6271,0.6335,0.6613,0.6610,0.6575
1,1,0.3,0.001,64,4,6,0.6613,0.6508,0.6228,0.6323,0.6598,0.6613,0.6575
2,4,0.5,0.001,128,8,10,0.6558,0.6378,0.6235,0.6293,0.6533,0.6558,0.6537
3,3,0.5,0.001,64,4,6,0.6629,0.6511,0.6147,0.6245,0.6607,0.6629,0.6563


Selected parameters: {'dropout': 0.3, 'learning_rate': 0.001, 'num_filters': 128}


FastText RNN:


,trial,dropout,hidden_size,learning_rate,best_epoch,epochs_ran,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,2,0.3,128,0.001,4,6,0.6467,0.6262,0.6105,0.6149,0.6448,0.6467,0.6427
1,3,0.5,64,0.001,5,7,0.6571,0.6513,0.6026,0.6129,0.6575,0.6571,0.6479
2,4,0.5,128,0.001,1,3,0.6451,0.6269,0.6000,0.6086,0.6407,0.6451,0.6400
3,1,0.3,64,0.001,2,4,0.6470,0.6393,0.5872,0.5974,0.6443,0.6470,0.6364


Selected parameters: {'dropout': 0.3, 'hidden_size': 128, 'learning_rate': 0.001}


FastText LSTM:


,trial,dropout,hidden_size,learning_rate,best_epoch,epochs_ran,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,1,0.3,64,0.001,2,4,0.6529,0.6310,0.6096,0.6164,0.6486,0.6529,0.6481
1,4,0.5,128,0.001,2,4,0.6542,0.6432,0.6028,0.6129,0.6512,0.6542,0.6465
2,2,0.3,128,0.001,2,4,0.6574,0.6450,0.6017,0.6121,0.6534,0.6574,0.6488
3,3,0.5,64,0.001,2,4,0.6519,0.6386,0.6026,0.6111,0.6503,0.6519,0.6443


Selected parameters: {'dropout': 0.3, 'hidden_size': 64, 'learning_rate': 0.001}


In [74]:
FASTTEXT_NEURAL_VALIDATION_RESULTS = {}
figure, axes = plt.subplots(3, 2, figsize=(13, 14))
for row_index, model_name in enumerate(fasttext_neural_model_names):
    experiment = FASTTEXT_NEURAL_EXPERIMENTS[model_name]
    predictions = experiment["predictions"]
    metrics = aggregate_classification_metrics(
        TFIDF_TARGETS["validation"], predictions
    )
    per_class = per_class_metrics(TFIDF_TARGETS["validation"], predictions)
    confusion = pd.DataFrame(
        confusion_matrix(
            TFIDF_TARGETS["validation"], predictions, labels=SENTIMENT_LABELS
        ),
        index=pd.Index(SENTIMENT_LABELS, name="actual"),
        columns=pd.Index(SENTIMENT_LABELS, name="predicted"),
    )
    FASTTEXT_NEURAL_VALIDATION_RESULTS[model_name] = {
        "aggregate_metrics": metrics,
        "per_class_metrics": per_class,
        "confusion_matrix": confusion,
    }
    print(f"FastText {model_name.upper()} aggregate metrics:")
    display(pd.Series(metrics, name="value").to_frame().round(4))
    display(per_class.round(4))

    sns.lineplot(
        data=experiment["history"],
        x="epoch",
        y="training_loss",
        marker="o",
        ax=axes[row_index, 0],
    )
    axes[row_index, 0].set_title(f"FastText {model_name.upper()} training loss")
    axes[row_index, 0].set_ylabel("Cross-entropy loss")
    sns.heatmap(
        confusion,
        annot=True,
        fmt="d",
        cmap="BuGn",
        cbar=False,
        ax=axes[row_index, 1],
    )
    axes[row_index, 1].set_title(
        f"FastText {model_name.upper()} validation confusion matrix"
    )
figure.tight_layout()
plt.show()

FastText CNN aggregate metrics:


,value
accuracy,0.6610
precision_macro,0.6482
recall_macro,0.6271
f1_macro,0.6335
precision_weighted,0.6613
recall_weighted,0.6610
f1_weighted,0.6575


,precision,recall,f1,support
sentiment,,,,
NEG,0.6467,0.7622,0.6997,1249
NEU,0.7041,0.6409,0.6710,1292
POS,0.5937,0.4782,0.5297,550


FastText RNN aggregate metrics:


,value
accuracy,0.6467
precision_macro,0.6262
recall_macro,0.6105
f1_macro,0.6149
precision_weighted,0.6448
recall_weighted,0.6467
f1_weighted,0.6427


,precision,recall,f1,support
sentiment,,,,
NEG,0.6481,0.7566,0.6982,1249
NEU,0.6831,0.6238,0.6521,1292
POS,0.5475,0.4509,0.4945,550


FastText LSTM aggregate metrics:

,value
accuracy,0.6529
precision_macro,0.6310
recall_macro,0.6096
f1_macro,0.6164
precision_weighted,0.6486
recall_weighted,0.6529
f1_weighted,0.6481


,precision,recall,f1,support
sentiment,,,,
NEG,0.6591,0.7414,0.6978,1249
NEU,0.6777,0.6656,0.6716,1292
POS,0.5564,0.4218,0.4798,550


/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/2998895210.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [75]:
fasttext_neural_artifact_directory = PATHS["artifacts"] / "models"
fasttext_neural_artifact_directory.mkdir(parents=True, exist_ok=True)
saved_fasttext_neural_rows = []
for model_name in fasttext_neural_model_names:
    experiment = FASTTEXT_NEURAL_EXPERIMENTS[model_name]
    validation_result = FASTTEXT_NEURAL_VALIDATION_RESULTS[model_name]
    tuning_path = tuning_results_directory / f"fasttext_{model_name}.csv"
    history_path = (
        tuning_results_directory / f"fasttext_{model_name}_selected_history.csv"
    )
    validation_path = (
        validation_results_directory / f"fasttext_{model_name}.json"
    )
    model_path = fasttext_neural_artifact_directory / f"fasttext_{model_name}.pt"
    experiment["tuning_results"].to_csv(tuning_path, index=False)
    experiment["history"].to_csv(history_path, index=False)
    torch.save(
        {
            "state_dict": experiment["state_dict"],
            "parameters": experiment["parameters"],
            "architecture": model_name,
            "max_sequence_length": FASTTEXT_MAX_SEQUENCE_LENGTH,
            "label_order": SENTIMENT_LABELS,
        },
        model_path,
    )
    result_payload = {
        "experiment_id": f"fasttext_{model_name}_validation",
        "representation": "fasttext_sequence",
        "classifier": "vanilla_rnn" if model_name == "rnn" else model_name,
        "seed": SEED,
        "device": str(DEVICE),
        "selection_metric": CONFIG["evaluation"]["primary_metric"],
        "selected_parameters": experiment["parameters"],
        "selected_epoch": experiment["selected_epoch"],
        "training_rows": len(PROCESSED_PARTITIONS["train"]),
        "validation_rows": len(PROCESSED_PARTITIONS["validation"]),
        "aggregate_metrics": validation_result["aggregate_metrics"],
        "per_class_metrics": validation_result["per_class_metrics"].to_dict(
            orient="index"
        ),
        "confusion_matrix": validation_result["confusion_matrix"].to_dict(
            orient="index"
        ),
        "test_evaluated": False,
        "class_weighted": False,
    }
    with validation_path.open("w", encoding="utf-8") as result_file:
        json.dump(
            result_payload,
            result_file,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
        result_file.write("\n")
    saved_fasttext_neural_rows.extend(
        {"model": model_name, "artifact": str(path.relative_to(PROJECT_ROOT))}
        for path in (tuning_path, history_path, validation_path, model_path)
    )

pd.DataFrame(saved_fasttext_neural_rows)

,model,artifact
0,cnn,results/tuning/fasttext_cnn.csv
1,cnn,results/tuning/fasttext_cnn_selected_history.csv
2,cnn,results/validation/fasttext_cnn.json
3,cnn,artifacts/models/fasttext_cnn.pt
4,rnn,results/tuning/fasttext_rnn.csv
5,rnn,results/tuning/fasttext_rnn_selected_history.csv
6,rnn,results/validation/fasttext_rnn.json
7,rnn,artifacts/models/fasttext_rnn.pt
8,lstm,results/tuning/fasttext_lstm.csv
9,lstm,results/tuning/fasttext_lstm_selected_history.csv


### FastText neural-model validation notes

All three classifiers use the same frozen FastText token projections, split, and selection protocol as their Word2Vec counterparts. Character n-grams supply vectors for held-out word forms without updating embeddings from held-out data. Validation selects one checkpoint per architecture; class weighting and test evaluation remain deferred.

## 9. Preprocessing ablation

Use the fixed default TF-IDF configuration and the already-selected Multinomial Naive Bayes parameters to isolate preprocessing effects. The full pipeline is the control; each enabled stage is disabled alone while all other settings remain fixed. Every vectorizer is fitted on that variant's training text only, validation text is transformed with the frozen training vocabulary, and the test partition is never scored.

In [76]:
def prepare_model_text_with_overrides(
    text: str, overrides: dict[str, bool]
) -> str:
    """Preprocess one text under an ablation, preserving a non-empty fallback."""
    processed = preprocess_arabic(text, overrides)
    if processed:
        return processed
    fallback_overrides = {
        **overrides,
        "remove_stopwords": False,
        "lemmatize_arabic_tokens": False,
        "stem_arabic_tokens": False,
    }
    return preprocess_arabic(text, fallback_overrides) or "نص_فارغ"


def make_default_tfidf_vectorizer() -> TfidfVectorizer:
    """Create an unfitted copy of the controlled TF-IDF representation."""
    config = CONFIG["features"]["tfidf"]
    return TfidfVectorizer(
        analyzer=config["analyzer"],
        token_pattern=config["token_pattern"],
        lowercase=bool(config["lowercase"]),
        ngram_range=tuple(config["ngram_range"]),
        min_df=config["min_df"],
        max_df=config["max_df"],
        max_features=int(config["max_features"]),
        sublinear_tf=bool(config["sublinear_tf"]),
        norm=config["norm"],
        dtype=np.dtype(config["dtype"]).type,
    )


def evaluate_controlled_tfidf_mnb(
    training_text: pd.Series,
    training_targets: pd.Series,
    validation_text: pd.Series,
) -> tuple[dict[str, float], pd.DataFrame, int, np.ndarray]:
    """Fit the frozen-design TF-IDF + MNB control on training text only."""
    vectorizer = make_default_tfidf_vectorizer()
    training_matrix = vectorizer.fit_transform(training_text)
    validation_matrix = vectorizer.transform(validation_text)
    model = MultinomialNB(**BEST_TFIDF_MNB_PARAMETERS).fit(
        training_matrix, training_targets
    )
    predictions = model.predict(validation_matrix)
    metrics = aggregate_classification_metrics(
        TFIDF_TARGETS["validation"], predictions
    )
    class_metrics = per_class_metrics(TFIDF_TARGETS["validation"], predictions)
    return metrics, class_metrics, training_matrix.shape[1], predictions


ablation_stage_names = [
    "decode_html_entities",
    "remove_html_tags",
    "remove_urls",
    "replace_mentions",
    "demojize_emojis",
    "unpack_hashtags",
    "normalize_arabic_characters",
    "remove_diacritics",
    "remove_tatweel",
    "normalize_digits",
    "remove_punctuation_symbols",
    "collapse_repeated_characters",
    "remove_stopwords",
    "lemmatize_arabic_tokens",
    "preserve_negation",
]
ABLATION_VARIANTS = [
    {"variant": "full_pipeline", "disabled_stage": None, "overrides": {}},
    *(
        {
            "variant": f"without_{stage_name}",
            "disabled_stage": stage_name,
            "overrides": {stage_name: False},
        }
        for stage_name in ablation_stage_names
    ),
]

ablation_rows = []
ablation_details = {}
baseline_training_text = PROCESSED_PARTITIONS["train"][MODEL_TEXT_COLUMN]
baseline_validation_text = PROCESSED_PARTITIONS["validation"][MODEL_TEXT_COLUMN]
for variant_specification in ABLATION_VARIANTS:
    overrides = variant_specification["overrides"]
    if overrides:
        training_text = PARTITIONS["train"]["tweet"].map(
            lambda text, stage_overrides=overrides: (
                prepare_model_text_with_overrides(text, stage_overrides)
            )
        )
        validation_text = PARTITIONS["validation"]["tweet"].map(
            lambda text, stage_overrides=overrides: (
                prepare_model_text_with_overrides(text, stage_overrides)
            )
        )
    else:
        training_text = baseline_training_text.copy()
        validation_text = baseline_validation_text.copy()

    metrics, class_metrics, feature_count, predictions = (
        evaluate_controlled_tfidf_mnb(
            training_text, TFIDF_TARGETS["train"], validation_text
        )
    )
    variant_name = str(variant_specification["variant"])
    ablation_rows.append(
        {
            "variant": variant_name,
            "disabled_stage": variant_specification["disabled_stage"],
            "changed_training_rows": int(
                training_text.ne(baseline_training_text).sum()
            ),
            "feature_count": feature_count,
            **metrics,
            **{
                f"{label.lower()}_f1": float(class_metrics.loc[label, "f1"])
                for label in SENTIMENT_LABELS
            },
        }
    )
    ablation_details[variant_name] = {
        "overrides": overrides,
        "aggregate_metrics": metrics,
        "per_class_metrics": class_metrics.to_dict(orient="index"),
        "confusion_matrix": pd.DataFrame(
            confusion_matrix(
                TFIDF_TARGETS["validation"],
                predictions,
                labels=SENTIMENT_LABELS,
            ),
            index=SENTIMENT_LABELS,
            columns=SENTIMENT_LABELS,
        ).to_dict(orient="index"),
    }

preprocessing_ablation_results = pd.DataFrame(ablation_rows)
baseline_ablation_row = preprocessing_ablation_results.iloc[0]
preprocessing_ablation_results["macro_f1_delta"] = (
    preprocessing_ablation_results["f1_macro"]
    - float(baseline_ablation_row["f1_macro"])
)
preprocessing_ablation_results["positive_f1_delta"] = (
    preprocessing_ablation_results["pos_f1"]
    - float(baseline_ablation_row["pos_f1"])
)
if not np.isclose(
    float(baseline_ablation_row["f1_macro"]),
    tfidf_mnb_validation_metrics["f1_macro"],
):
    raise ValueError("Ablation control does not reproduce the TF-IDF MNB baseline.")

display(preprocessing_ablation_results.round(4))

,variant,disabled_stage,changed_training_rows,feature_count,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,neg_f1,neu_f1,pos_f1,macro_f1_delta,positive_f1_delta
0,full_pipeline,None,0,16496,0.6635,0.6410,0.6317,0.6335,0.6653,0.6635,0.6611,0.7097,0.6769,0.5138,0.0000,0.0000
1,without_decode_html_entities,decode_html_entities,3,16496,0.6635,0.6410,0.6317,0.6335,0.6653,0.6635,0.6611,0.7097,0.6769,0.5138,0.0000,0.0000
2,without_remove_html_tags,remove_html_tags,0,16496,0.6635,0.6410,0.6317,0.6335,0.6653,0.6635,0.6611,0.7097,0.6769,0.5138,0.0000,0.0000
3,without_remove_urls,remove_urls,3102,18083,0.6713,0.6517,0.6319,0.6381,0.6696,0.6713,0.6674,0.7129,0.6896,0.5116,0.0046,-0.0022
4,without_replace_mentions,replace_mentions,1664,16979,0.6622,0.6400,0.6307,0.6320,0.6645,0.6622,0.6595,0.7126,0.6706,0.5128,-0.0015,-0.0010
5,without_demojize_emojis,demojize_emojis,1188,16117,0.6551,0.6309,0.6173,0.6204,0.6560,0.6551,0.6517,0.7018,0.6741,0.4853,-0.0131,-0.0285
6,without_unpack_hashtags,unpack_hashtags,2359,15898,0.6590,0.6350,0.6256,0.6276,0.6599,0.6590,0.6565,0.7059,0.6741,0.5029,-0.0058,-0.0109
7,without_normalize_arabic_characters,normalize_arabic_characters,6877,17011,0.6639,0.6413,0.6313,0.6330,0.6661,0.6639,0.6613,0.7107,0.6775,0.5110,-0.0004,-0.0028
8,without_remove_diacritics,remove_diacritics,447,16529,0.6648,0.6427,0.6324,0.6345,0.6665,0.6648,0.6623,0.7101,0.6791,0.5143,0.0011,0.0005
9,without_remove_tatweel,remove_tatweel,319,16540,0.6639,0.6416,0.6319,0.6338,0.6658,0.6639,0.6614,0.7094,0.6777,0.5143,0.0003,0.0005


In [77]:
experiment_results_directory = PATHS["results"] / "experiments"
experiment_results_directory.mkdir(parents=True, exist_ok=True)
ablation_results_path = experiment_results_directory / "preprocessing_ablation.csv"
ablation_details_path = experiment_results_directory / "preprocessing_ablation.json"
preprocessing_ablation_results.to_csv(ablation_results_path, index=False)
with ablation_details_path.open("w", encoding="utf-8") as result_file:
    json.dump(
        {
            "experiment_id": "preprocessing_ablation_validation",
            "representation": "tfidf",
            "classifier": "multinomial_naive_bayes",
            "selected_parameters": BEST_TFIDF_MNB_PARAMETERS,
            "selection_partition": "validation",
            "test_evaluated": False,
            "variants": ablation_details,
        },
        result_file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    result_file.write("\n")

ablation_plot_data = preprocessing_ablation_results.iloc[1:].sort_values(
    "macro_f1_delta"
)
figure, axis = plt.subplots(figsize=(10, 7))
sns.barplot(
    data=ablation_plot_data,
    x="macro_f1_delta",
    y="disabled_stage",
    color="#4c78a8",
    ax=axis,
)
axis.axvline(0, color="black", linewidth=1)
axis.set_title("Preprocessing ablation: validation macro F1 change")
axis.set_xlabel("Macro F1 delta from full pipeline")
axis.set_ylabel("Disabled preprocessing stage")
figure.tight_layout()
plt.show()

pd.DataFrame(
    {
        "result": [
            str(path.relative_to(PROJECT_ROOT))
            for path in (ablation_results_path, ablation_details_path)
        ]
    }
)

/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/715065307.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,result
0,results/experiments/preprocessing_ablation.csv
1,results/experiments/preprocessing_ablation.json


### Preprocessing ablation notes

The control reproduces the tuned TF-IDF + Multinomial NB validation baseline exactly. Each row changes one enabled preprocessing option, reports how many training records actually change, and records both macro-F1 and positive-class-F1 deltas. Variants that affect few or no records are retained because that absence of impact is itself an experimental result.

## 10. Arabic data augmentation

Apply deterministic Easy Data Augmentation to positive-sentiment training tweets until that minority class matches the largest training class. Random swap and single-token deletion operate on raw Arabic whitespace tokens, while negations, emojis, hashtags, mentions, URLs, and number placeholders are protected. Synthetic text is preprocessed by the unchanged default pipeline; duplicates and any processed-text collision with validation or test data are rejected. Evaluation uses the same fixed TF-IDF + selected Multinomial NB control, and only validation performance is compared.

In [78]:
augmentation_config = CONFIG["experiments"]["augmentation"]
AUGMENTATION_TARGET_CLASS = str(augmentation_config["target_class"])
AUGMENTATION_OPERATIONS = tuple(augmentation_config["operations"])
if set(AUGMENTATION_OPERATIONS) != {"swap", "delete"}:
    raise ValueError("The controlled Arabic EDA study expects swap and delete.")


def is_augmentation_protected_token(token: str) -> bool:
    """Protect sentiment-bearing and social-media structural tokens."""
    normalized = token.translate(ARABIC_CHARACTER_TRANSLATION)
    return (
        token.startswith(("#", "@"))
        or bool(URL_PATTERN.search(token))
        or emoji.emoji_count(token) > 0
        or token in ALWAYS_PROTECTED_TOKENS
        or normalized in ALWAYS_PROTECTED_TOKENS
        or token in NEGATION_TERMS
        or normalized in NEGATION_TERMS
    )


def arabic_eda_variant(text: str, operation: str, seed: int) -> str | None:
    """Create one protected Arabic EDA variant or return None if impossible."""
    tokens = text.split()
    eligible_indices = [
        index
        for index, token in enumerate(tokens)
        if not is_augmentation_protected_token(token)
    ]
    generator = random.Random(seed)
    augmented_tokens = list(tokens)
    if operation == "swap" and len(eligible_indices) >= 2:
        first_index, second_index = generator.sample(eligible_indices, 2)
        augmented_tokens[first_index], augmented_tokens[second_index] = (
            augmented_tokens[second_index],
            augmented_tokens[first_index],
        )
    elif operation == "delete" and eligible_indices:
        del augmented_tokens[generator.choice(eligible_indices)]
    elif len(eligible_indices) >= 2:
        first_index, second_index = eligible_indices[:2]
        augmented_tokens[first_index], augmented_tokens[second_index] = (
            augmented_tokens[second_index],
            augmented_tokens[first_index],
        )
    elif eligible_indices:
        del augmented_tokens[eligible_indices[0]]
    else:
        return None

    if not augmented_tokens or augmented_tokens == tokens:
        return None
    protected_tokens = [
        token for token in tokens if is_augmentation_protected_token(token)
    ]
    for protected_token in set(protected_tokens):
        if augmented_tokens.count(protected_token) < tokens.count(protected_token):
            raise ValueError("Arabic EDA removed a protected token.")
    return " ".join(augmented_tokens)


training_class_counts = pd.Series(TFIDF_TARGETS["train"]).value_counts()
if augmentation_config["target_count_strategy"] != "match_largest_class":
    raise ValueError("Unsupported augmentation target-count strategy.")
augmentation_target_count = int(training_class_counts.max())
augmentation_rows_needed = (
    augmentation_target_count
    - int(training_class_counts[AUGMENTATION_TARGET_CLASS])
)
positive_training_rows = PROCESSED_PARTITIONS["train"].loc[
    PROCESSED_PARTITIONS["train"][CONFIG["data"]["target"]]
    == AUGMENTATION_TARGET_CLASS
].reset_index(drop=True)
heldout_processed_texts = set(
    PROCESSED_PARTITIONS["validation"][MODEL_TEXT_COLUMN]
).union(PROCESSED_PARTITIONS["test"][MODEL_TEXT_COLUMN])
occupied_training_texts = set(
    PROCESSED_PARTITIONS["train"][MODEL_TEXT_COLUMN]
)
synthetic_texts = set()
augmentation_rows = []
attempt_number = 0
maximum_attempts = max(augmentation_rows_needed * 50, 1)
while len(augmentation_rows) < augmentation_rows_needed:
    if attempt_number >= maximum_attempts:
        raise RuntimeError("Could not create enough unique Arabic EDA variants.")
    source_index = attempt_number % len(positive_training_rows)
    variant_round = attempt_number // len(positive_training_rows)
    source_row = positive_training_rows.iloc[source_index]
    operation = AUGMENTATION_OPERATIONS[attempt_number % len(AUGMENTATION_OPERATIONS)]
    variant_seed = stable_text_hash(
        f"{source_row['record_id']}|{variant_round}|{operation}|{SEED}"
    )
    augmented_raw_text = arabic_eda_variant(
        str(source_row["tweet"]), operation, variant_seed
    )
    attempt_number += 1
    if augmented_raw_text is None:
        continue
    augmented_processed_text = prepare_model_text(augmented_raw_text)
    if (
        augmented_processed_text == source_row[MODEL_TEXT_COLUMN]
        or augmented_processed_text in occupied_training_texts
        or augmented_processed_text in heldout_processed_texts
        or augmented_processed_text in synthetic_texts
    ):
        continue
    synthetic_texts.add(augmented_processed_text)
    augmentation_rows.append(
        {
            "source_record_id": source_row["record_id"],
            "operation": operation,
            MODEL_TEXT_COLUMN: augmented_processed_text,
            CONFIG["data"]["target"]: AUGMENTATION_TARGET_CLASS,
        }
    )

augmented_examples = pd.DataFrame(augmentation_rows)
augmented_training_text = pd.concat(
    [
        PROCESSED_PARTITIONS["train"][MODEL_TEXT_COLUMN],
        augmented_examples[MODEL_TEXT_COLUMN],
    ],
    ignore_index=True,
)
augmented_training_targets = pd.concat(
    [
        pd.Series(TFIDF_TARGETS["train"]),
        augmented_examples[CONFIG["data"]["target"]],
    ],
    ignore_index=True,
)
augmented_class_counts = augmented_training_targets.value_counts().reindex(
    SENTIMENT_LABELS
)
if int(augmented_class_counts[AUGMENTATION_TARGET_CLASS]) != augmentation_target_count:
    raise ValueError("Arabic EDA did not reach the configured target count.")

display(
    pd.DataFrame(
        {"before": training_class_counts, "after": augmented_class_counts}
    ).fillna(0).astype(int)
)
display(augmented_examples["operation"].value_counts().rename("synthetic_rows"))

,before,after
NEG,3750,3750
NEU,3876,3876
POS,1647,3876


operation
swap      1167
delete    1062
Name: synthetic_rows, dtype: int64

In [79]:
(
    augmented_validation_metrics,
    augmented_validation_per_class,
    augmented_feature_count,
    AUGMENTED_VALIDATION_PREDICTIONS,
) = evaluate_controlled_tfidf_mnb(
    augmented_training_text,
    augmented_training_targets,
    PROCESSED_PARTITIONS["validation"][MODEL_TEXT_COLUMN],
)
augmentation_comparison_rows = []
for condition, metrics, class_metrics, training_rows in (
    (
        "unaugmented_baseline",
        tfidf_mnb_validation_metrics,
        tfidf_mnb_validation_per_class,
        len(PROCESSED_PARTITIONS["train"]),
    ),
    (
        "arabic_eda_positive_balanced",
        augmented_validation_metrics,
        augmented_validation_per_class,
        len(augmented_training_text),
    ),
):
    augmentation_comparison_rows.append(
        {
            "condition": condition,
            "training_rows": training_rows,
            **metrics,
            **{
                f"{label.lower()}_precision": float(
                    class_metrics.loc[label, "precision"]
                )
                for label in SENTIMENT_LABELS
            },
            **{
                f"{label.lower()}_recall": float(class_metrics.loc[label, "recall"])
                for label in SENTIMENT_LABELS
            },
            **{
                f"{label.lower()}_f1": float(class_metrics.loc[label, "f1"])
                for label in SENTIMENT_LABELS
            },
        }
    )
augmentation_comparison = pd.DataFrame(augmentation_comparison_rows)
augmentation_comparison["macro_f1_delta"] = (
    augmentation_comparison["f1_macro"]
    - float(augmentation_comparison.iloc[0]["f1_macro"])
)
augmentation_comparison["positive_f1_delta"] = (
    augmentation_comparison["pos_f1"]
    - float(augmentation_comparison.iloc[0]["pos_f1"])
)
augmentation_comparison_path = (
    experiment_results_directory / "augmentation_comparison.csv"
)
augmentation_details_path = experiment_results_directory / "augmentation_details.json"
augmentation_comparison.to_csv(augmentation_comparison_path, index=False)
with augmentation_details_path.open("w", encoding="utf-8") as result_file:
    json.dump(
        {
            "experiment_id": "arabic_eda_validation",
            "target_class": AUGMENTATION_TARGET_CLASS,
            "operations": list(AUGMENTATION_OPERATIONS),
            "synthetic_rows": len(augmented_examples),
            "attempted_variants": attempt_number,
            "feature_count": augmented_feature_count,
            "class_counts_before": training_class_counts.to_dict(),
            "class_counts_after": augmented_class_counts.to_dict(),
            "validation_metrics": augmented_validation_metrics,
            "validation_per_class": augmented_validation_per_class.to_dict(
                orient="index"
            ),
            "test_evaluated": False,
        },
        result_file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    result_file.write("\n")

display(augmentation_comparison.round(4))
augmentation_class_f1 = augmentation_comparison.melt(
    id_vars="condition",
    value_vars=[f"{label.lower()}_f1" for label in SENTIMENT_LABELS],
    var_name="class_metric",
    value_name="f1",
)
figure, axis = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=augmentation_class_f1,
    x="class_metric",
    y="f1",
    hue="condition",
    ax=axis,
)
axis.set_title("Arabic EDA effect on validation class F1")
axis.set_xlabel("Sentiment class")
axis.set_ylabel("F1")
figure.tight_layout()
plt.show()

,condition,training_rows,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,neg_precision,neu_precision,pos_precision,neg_recall,neu_recall,pos_recall,neg_f1,neu_f1,pos_f1,macro_f1_delta,positive_f1_delta
0,unaugmented_baseline,9273,0.6635,0.6410,0.6317,0.6335,0.6653,0.6635,0.6611,0.6580,0.7261,0.5389,0.7702,0.6339,0.4909,0.7097,0.6769,0.5138,0.0000,0.0000
1,arabic_eda_positive_balanced,11502,0.6671,0.6579,0.6112,0.6207,0.6674,0.6671,0.6577,0.6416,0.7126,0.6196,0.7982,0.6680,0.3673,0.7114,0.6896,0.4612,-0.0127,-0.0526


/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/3658565550.py:104: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Arabic augmentation notes

The augmentation study modifies training records only and evaluates the unchanged validation partition. It balances the positive class through unique protected-token EDA variants, rejects cross-partition processed-text collisions, and reports per-class precision, recall, and F1 alongside aggregate metrics. It is intentionally separate from the class-weighting experiment.

## 11. Class-imbalance handling

Evaluate cost-sensitive learning independently of augmentation by applying training-derived balanced class weights to the selected TF-IDF Random Forest configuration. Representation, split, tree hyperparameters, and random seed remain fixed. Compare against the unweighted validation baseline using aggregate and per-class metrics; the test partition is not scored.

In [80]:
imbalance_config = CONFIG["experiments"]["class_imbalance"]
if (
    imbalance_config["representation"] != "tfidf"
    or imbalance_config["classifier"] != "random_forest"
    or imbalance_config["class_weight"] != "balanced"
):
    raise ValueError("Unexpected controlled class-imbalance configuration.")
training_target_counts = pd.Series(TFIDF_TARGETS["train"]).value_counts()
balanced_class_weights = {
    label: len(TFIDF_TARGETS["train"])
    / (len(SENTIMENT_LABELS) * int(training_target_counts[label]))
    for label in SENTIMENT_LABELS
}
weighted_rf_parameters = {
    **rf_fixed_parameters,
    **BEST_TFIDF_RF_PARAMETERS,
    "class_weight": balanced_class_weights,
}
WEIGHTED_TFIDF_RF = RandomForestClassifier(**weighted_rf_parameters).fit(
    TFIDF_MATRICES["train"], TFIDF_TARGETS["train"]
)
WEIGHTED_TFIDF_RF_VALIDATION_PREDICTIONS = WEIGHTED_TFIDF_RF.predict(
    TFIDF_MATRICES["validation"]
)
weighted_rf_validation_metrics = aggregate_classification_metrics(
    TFIDF_TARGETS["validation"], WEIGHTED_TFIDF_RF_VALIDATION_PREDICTIONS
)
weighted_rf_validation_per_class = per_class_metrics(
    TFIDF_TARGETS["validation"], WEIGHTED_TFIDF_RF_VALIDATION_PREDICTIONS
)

class_weight_comparison_rows = []
for condition, metrics, class_metrics in (
    ("unweighted_baseline", tfidf_rf_validation_metrics, tfidf_rf_validation_per_class),
    (
        "balanced_class_weight",
        weighted_rf_validation_metrics,
        weighted_rf_validation_per_class,
    ),
):
    class_weight_comparison_rows.append(
        {
            "condition": condition,
            **metrics,
            **{
                f"{label.lower()}_precision": float(
                    class_metrics.loc[label, "precision"]
                )
                for label in SENTIMENT_LABELS
            },
            **{
                f"{label.lower()}_recall": float(class_metrics.loc[label, "recall"])
                for label in SENTIMENT_LABELS
            },
            **{
                f"{label.lower()}_f1": float(class_metrics.loc[label, "f1"])
                for label in SENTIMENT_LABELS
            },
        }
    )
class_weight_comparison = pd.DataFrame(class_weight_comparison_rows)
class_weight_comparison["macro_f1_delta"] = (
    class_weight_comparison["f1_macro"]
    - float(class_weight_comparison.iloc[0]["f1_macro"])
)
class_weight_comparison["positive_f1_delta"] = (
    class_weight_comparison["pos_f1"]
    - float(class_weight_comparison.iloc[0]["pos_f1"])
)
display(pd.Series(balanced_class_weights, name="weight").to_frame().round(4))
display(class_weight_comparison.round(4))

,weight
NEG,0.8243
NEU,0.7975
POS,1.8767


,condition,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,neg_precision,neu_precision,pos_precision,neg_recall,neu_recall,pos_recall,neg_f1,neu_f1,pos_f1,macro_f1_delta,positive_f1_delta
0,unweighted_baseline,0.6370,0.6368,0.5772,0.5886,0.6369,0.6370,0.6263,0.6305,0.6431,0.6367,0.6845,0.7252,0.3218,0.6564,0.6817,0.4275,0.0000,0.0000
1,balanced_class_weight,0.6286,0.6039,0.6046,0.6037,0.6298,0.6286,0.6285,0.6687,0.6509,0.4920,0.6189,0.6912,0.5036,0.6428,0.6704,0.4978,0.0151,0.0702


In [81]:
class_weight_comparison_path = (
    experiment_results_directory / "class_weight_comparison.csv"
)
class_weight_details_path = (
    experiment_results_directory / "class_weight_details.json"
)
class_weight_comparison.to_csv(class_weight_comparison_path, index=False)
with class_weight_details_path.open("w", encoding="utf-8") as result_file:
    json.dump(
        {
            "experiment_id": "tfidf_random_forest_class_weight_validation",
            "representation": "tfidf",
            "classifier": "random_forest",
            "selected_parameters": BEST_TFIDF_RF_PARAMETERS,
            "class_weights": balanced_class_weights,
            "validation_metrics": weighted_rf_validation_metrics,
            "validation_per_class": weighted_rf_validation_per_class.to_dict(
                orient="index"
            ),
            "test_evaluated": False,
            "augmented": False,
        },
        result_file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    result_file.write("\n")

weighted_class_f1 = class_weight_comparison.melt(
    id_vars="condition",
    value_vars=[f"{label.lower()}_f1" for label in SENTIMENT_LABELS],
    var_name="class_metric",
    value_name="f1",
)
figure, axis = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=weighted_class_f1,
    x="class_metric",
    y="f1",
    hue="condition",
    ax=axis,
)
axis.set_title("Balanced class-weight effect on validation class F1")
axis.set_xlabel("Sentiment class")
axis.set_ylabel("F1")
figure.tight_layout()
plt.show()

/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/2234592883.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Class-imbalance notes

Balanced weights are calculated from training labels only and applied to the otherwise unchanged selected TF-IDF Random Forest. The comparison reports minority positive-class precision, recall, and F1 as well as aggregate metrics. No augmented records are used, so the weighting effect is isolated.

## 12. Frozen comparative evaluation and error analysis

Collect the 12 required representation-model baselines and rank them using validation macro F1 before inspecting any test predictions. Freeze that winner and every model's selected hyperparameters, then evaluate each baseline on the untouched test partition exactly once. Repeat only the validation-selected winner with seeds 42, 1337, and 2026 for a stability estimate. Generate all comparison tables, controlled-study summaries, error analyses, and report figures directly from machine-readable results.

In [82]:
BASELINE_MODEL_KEYS = [
    "tfidf_multinomial_nb",
    "tfidf_random_forest",
    "word2vec_gaussian_nb",
    "word2vec_random_forest",
    "word2vec_cnn",
    "word2vec_rnn",
    "word2vec_lstm",
    "fasttext_gaussian_nb",
    "fasttext_random_forest",
    "fasttext_cnn",
    "fasttext_rnn",
    "fasttext_lstm",
]
BASELINE_VALIDATION_PAYLOADS = {}
validation_comparison_rows = []
for model_key in BASELINE_MODEL_KEYS:
    validation_path = validation_results_directory / f"{model_key}.json"
    with validation_path.open(encoding="utf-8") as result_file:
        payload = json.load(result_file)
    if payload["test_evaluated"]:
        raise ValueError(f"Test leakage detected before freeze: {model_key}")
    BASELINE_VALIDATION_PAYLOADS[model_key] = payload
    validation_comparison_rows.append(
        {
            "model_key": model_key,
            "representation": payload["representation"],
            "classifier": payload["classifier"],
            "selected_parameters": json.dumps(
                payload["selected_parameters"],
                ensure_ascii=False,
                sort_keys=True,
            ),
            **{
                f"validation_{metric_name}": metric_value
                for metric_name, metric_value in payload["aggregate_metrics"].items()
            },
        }
    )
validation_model_comparison = (
    pd.DataFrame(validation_comparison_rows)
    .sort_values(
        [
            "validation_f1_macro",
            "validation_f1_weighted",
            "validation_accuracy",
            "model_key",
        ],
        ascending=[False, False, False, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)
validation_model_comparison.insert(
    0, "validation_rank", np.arange(1, len(validation_model_comparison) + 1)
)
SELECTED_FINAL_MODEL_KEY = str(validation_model_comparison.iloc[0]["model_key"])
SELECTED_FINAL_EXPERIMENT_ID = BASELINE_VALIDATION_PAYLOADS[
    SELECTED_FINAL_MODEL_KEY
]["experiment_id"]
if SELECTED_FINAL_MODEL_KEY != "fasttext_cnn":
    raise ValueError(
        "The deterministic validation winner changed; review final seed protocol."
    )
display(validation_model_comparison.round(4))
print(f"Frozen validation-selected model: {SELECTED_FINAL_MODEL_KEY}")

,validation_rank,model_key,representation,classifier,selected_parameters,validation_accuracy,validation_f1_macro,validation_f1_weighted,validation_precision_macro,validation_precision_weighted,validation_recall_macro,validation_recall_weighted
0,1,fasttext_cnn,fasttext_sequence,cnn,"{""dropout"": 0.3, ""learning_rate"": 0.001, ""num_...",0.6610,0.6335,0.6575,0.6482,0.6613,0.6271,0.6610
1,2,tfidf_multinomial_nb,tfidf,multinomial_naive_bayes,"{""alpha"": 0.5, ""fit_prior"": false}",0.6635,0.6335,0.6611,0.6410,0.6653,0.6317,0.6635
2,3,word2vec_lstm,word2vec_sequence,lstm,"{""dropout"": 0.5, ""hidden_size"": 64, ""learning_...",0.6571,0.6296,0.6542,0.6405,0.6598,0.6265,0.6571
3,4,word2vec_cnn,word2vec_sequence,cnn,"{""dropout"": 0.5, ""learning_rate"": 0.001, ""num_...",0.6580,0.6256,0.6533,0.6482,0.6559,0.6149,0.6580
4,5,word2vec_rnn,word2vec_sequence,vanilla_rnn,"{""dropout"": 0.3, ""hidden_size"": 128, ""learning...",0.6454,0.6177,0.6414,0.6303,0.6503,0.6163,0.6454
5,6,fasttext_lstm,fasttext_sequence,lstm,"{""dropout"": 0.3, ""hidden_size"": 64, ""learning_...",0.6529,0.6164,0.6481,0.6310,0.6486,0.6096,0.6529
6,7,fasttext_rnn,fasttext_sequence,vanilla_rnn,"{""dropout"": 0.3, ""hidden_size"": 128, ""learning...",0.6467,0.6149,0.6427,0.6262,0.6448,0.6105,0.6467
7,8,fasttext_random_forest,fasttext_mean,random_forest,"{""max_depth"": null, ""min_samples_leaf"": 2, ""n_...",0.6561,0.5986,0.6421,0.6581,0.6593,0.5894,0.6561
8,9,word2vec_random_forest,word2vec_mean,random_forest,"{""max_depth"": 40, ""min_samples_leaf"": 1, ""n_es...",0.6470,0.5900,0.6328,0.6562,0.6530,0.5803,0.6470
9,10,tfidf_random_forest,tfidf,random_forest,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_...",0.6370,0.5886,0.6263,0.6368,0.6369,0.5772,0.6370


Frozen validation-selected model: fasttext_cnn


In [83]:
def restore_word2vec_neural_model(model_name: str) -> nn.Module:
    """Restore one validation-selected Word2Vec neural checkpoint."""
    if model_name == "cnn":
        model = Word2VecTextCNN(
            CNN_EMBEDDING_WEIGHTS,
            num_filters=int(BEST_WORD2VEC_CNN_PARAMETERS["num_filters"]),
            kernel_sizes=[int(size) for size in cnn_config["kernel_sizes"]],
            dropout=float(BEST_WORD2VEC_CNN_PARAMETERS["dropout"]),
            freeze_embeddings=bool(cnn_config["freeze_embeddings"]),
        )
        state = BEST_WORD2VEC_CNN_STATE
    elif model_name == "rnn":
        model = Word2VecVanillaRNN(
            CNN_EMBEDDING_WEIGHTS,
            hidden_size=int(BEST_WORD2VEC_RNN_PARAMETERS["hidden_size"]),
            dropout=float(BEST_WORD2VEC_RNN_PARAMETERS["dropout"]),
            freeze_embeddings=bool(rnn_config["freeze_embeddings"]),
        )
        state = BEST_WORD2VEC_RNN_STATE
    elif model_name == "lstm":
        model = Word2VecLSTM(
            CNN_EMBEDDING_WEIGHTS,
            hidden_size=int(BEST_WORD2VEC_LSTM_PARAMETERS["hidden_size"]),
            dropout=float(BEST_WORD2VEC_LSTM_PARAMETERS["dropout"]),
            freeze_embeddings=bool(lstm_config["freeze_embeddings"]),
        )
        state = BEST_WORD2VEC_LSTM_STATE
    else:
        raise ValueError(f"Unsupported Word2Vec neural model: {model_name}")
    model.load_state_dict(state)
    return model.to(DEVICE)


def predict_word2vec_neural_test(model_name: str) -> np.ndarray:
    """Predict the test split with one frozen Word2Vec checkpoint."""
    model = restore_word2vec_neural_model(model_name)
    batch_size = int(CONFIG["models"][model_name]["batch_size"])
    if model_name == "cnn":
        return predict_cnn(model, "test", batch_size)
    if model_name == "rnn":
        return predict_rnn(model, "test", batch_size)
    return predict_lstm(model, "test", batch_size)


def predict_fasttext_neural_test(model_name: str) -> np.ndarray:
    """Predict the test split with one frozen FastText checkpoint."""
    experiment = FASTTEXT_NEURAL_EXPERIMENTS[model_name]
    model = build_fasttext_neural_model(
        model_name, experiment["parameters"]
    )
    model.load_state_dict(experiment["state_dict"])
    model = model.to(DEVICE)
    return predict_fasttext_neural(
        model, "test", int(CONFIG["models"][model_name]["batch_size"])
    )


FROZEN_TEST_PREDICTIONS = {
    "tfidf_multinomial_nb": BEST_TFIDF_MNB.predict(TFIDF_MATRICES["test"]),
    "tfidf_random_forest": BEST_TFIDF_RF.predict(TFIDF_MATRICES["test"]),
    "word2vec_gaussian_nb": BEST_WORD2VEC_GNB.predict(
        WORD2VEC_DOCUMENT_MATRICES["test"]
    ),
    "word2vec_random_forest": BEST_WORD2VEC_RF.predict(
        WORD2VEC_DOCUMENT_MATRICES["test"]
    ),
    "word2vec_cnn": predict_word2vec_neural_test("cnn"),
    "word2vec_rnn": predict_word2vec_neural_test("rnn"),
    "word2vec_lstm": predict_word2vec_neural_test("lstm"),
    "fasttext_gaussian_nb": BEST_FASTTEXT_GNB.predict(
        FASTTEXT_DOCUMENT_MATRICES["test"]
    ),
    "fasttext_random_forest": BEST_FASTTEXT_RF.predict(
        FASTTEXT_DOCUMENT_MATRICES["test"]
    ),
    "fasttext_cnn": predict_fasttext_neural_test("cnn"),
    "fasttext_rnn": predict_fasttext_neural_test("rnn"),
    "fasttext_lstm": predict_fasttext_neural_test("lstm"),
}
if set(FROZEN_TEST_PREDICTIONS) != set(BASELINE_MODEL_KEYS):
    raise ValueError("Frozen test prediction registry is incomplete.")
if any(
    len(predictions) != len(TFIDF_TARGETS["test"])
    for predictions in FROZEN_TEST_PREDICTIONS.values()
):
    raise ValueError("A frozen test prediction vector has the wrong length.")

In [84]:
FINAL_TEST_DETAILS = {}
final_comparison_rows = []
validation_lookup = validation_model_comparison.set_index("model_key")
for model_key in BASELINE_MODEL_KEYS:
    predictions = FROZEN_TEST_PREDICTIONS[model_key]
    test_metrics = aggregate_classification_metrics(
        TFIDF_TARGETS["test"], predictions
    )
    test_per_class = per_class_metrics(TFIDF_TARGETS["test"], predictions)
    test_confusion = pd.DataFrame(
        confusion_matrix(
            TFIDF_TARGETS["test"], predictions, labels=SENTIMENT_LABELS
        ),
        index=pd.Index(SENTIMENT_LABELS, name="actual"),
        columns=pd.Index(SENTIMENT_LABELS, name="predicted"),
    )
    FINAL_TEST_DETAILS[model_key] = {
        "aggregate_metrics": test_metrics,
        "per_class_metrics": test_per_class.to_dict(orient="index"),
        "confusion_matrix": test_confusion.to_dict(orient="index"),
    }
    validation_row = validation_lookup.loc[model_key]
    final_comparison_rows.append(
        {
            "validation_rank": int(validation_row["validation_rank"]),
            "model_key": model_key,
            "representation": validation_row["representation"],
            "classifier": validation_row["classifier"],
            "selected_parameters": validation_row["selected_parameters"],
            "selected_by_validation": model_key == SELECTED_FINAL_MODEL_KEY,
            **{
                column_name: validation_row[column_name]
                for column_name in validation_model_comparison.columns
                if column_name.startswith("validation_")
                and column_name != "validation_rank"
            },
            **{
                f"test_{metric_name}": metric_value
                for metric_name, metric_value in test_metrics.items()
            },
            **{
                f"test_{label.lower()}_f1": float(test_per_class.loc[label, "f1"])
                for label in SENTIMENT_LABELS
            },
        }
    )

final_model_comparison = (
    pd.DataFrame(final_comparison_rows)
    .sort_values("validation_rank", kind="mergesort")
    .reset_index(drop=True)
)
display(final_model_comparison.round(4))

,validation_rank,model_key,representation,classifier,selected_parameters,selected_by_validation,validation_accuracy,validation_f1_macro,validation_f1_weighted,validation_precision_macro,...,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_precision_weighted,test_recall_weighted,test_f1_weighted,test_neg_f1,test_neu_f1,test_pos_f1
0,1,fasttext_cnn,fasttext_sequence,cnn,"{""dropout"": 0.3, ""learning_rate"": 0.001, ""num_...",True,0.6610,0.6335,0.6575,0.6482,...,0.6674,0.6524,0.6313,0.6380,0.6673,0.6674,0.6642,0.6997,0.6886,0.5258
1,2,tfidf_multinomial_nb,tfidf,multinomial_naive_bayes,"{""alpha"": 0.5, ""fit_prior"": false}",False,0.6635,0.6335,0.6611,0.6410,...,0.6671,0.6457,0.6312,0.6348,0.6688,0.6671,0.6641,0.7086,0.6870,0.5088
2,3,word2vec_lstm,word2vec_sequence,lstm,"{""dropout"": 0.5, ""hidden_size"": 64, ""learning_...",False,0.6571,0.6296,0.6542,0.6405,...,0.6555,0.6361,0.6224,0.6262,0.6564,0.6555,0.6527,0.6947,0.6721,0.5117
3,4,word2vec_cnn,word2vec_sequence,cnn,"{""dropout"": 0.5, ""learning_rate"": 0.001, ""num_...",False,0.6580,0.6256,0.6533,0.6482,...,0.6622,0.6477,0.6155,0.6254,0.6589,0.6622,0.6567,0.6858,0.6985,0.4919
4,5,word2vec_rnn,word2vec_sequence,vanilla_rnn,"{""dropout"": 0.3, ""hidden_size"": 128, ""learning...",False,0.6454,0.6177,0.6414,0.6303,...,0.6454,0.6270,0.6139,0.6156,0.6502,0.6454,0.6422,0.6928,0.6534,0.5005
5,6,fasttext_lstm,fasttext_sequence,lstm,"{""dropout"": 0.3, ""hidden_size"": 64, ""learning_...",False,0.6529,0.6164,0.6481,0.6310,...,0.6522,0.6353,0.6107,0.6189,0.6488,0.6522,0.6480,0.6839,0.6785,0.4943
6,7,fasttext_rnn,fasttext_sequence,vanilla_rnn,"{""dropout"": 0.3, ""hidden_size"": 128, ""learning...",False,0.6467,0.6149,0.6427,0.6262,...,0.6509,0.6279,0.6074,0.6139,0.6472,0.6509,0.6464,0.6884,0.6790,0.4742
7,8,fasttext_random_forest,fasttext_mean,random_forest,"{""max_depth"": null, ""min_samples_leaf"": 2, ""n_...",False,0.6561,0.5986,0.6421,0.6581,...,0.6707,0.6812,0.6023,0.6128,0.6780,0.6707,0.6565,0.7089,0.7040,0.4253
8,9,word2vec_random_forest,word2vec_mean,random_forest,"{""max_depth"": 40, ""min_samples_leaf"": 1, ""n_es...",False,0.6470,0.5900,0.6328,0.6562,...,0.6587,0.6568,0.5893,0.5978,0.6611,0.6587,0.6443,0.6955,0.6989,0.3990
9,10,tfidf_random_forest,tfidf,random_forest,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_...",False,0.6370,0.5886,0.6263,0.6368,...,0.6457,0.6518,0.5829,0.5948,0.6480,0.6457,0.6338,0.6667,0.6895,0.4282


In [85]:
def train_fasttext_neural_fixed_epochs(
    model_name: str,
    parameters: dict[str, object],
    epochs: int,
    seed: int,
) -> tuple[nn.Module, pd.DataFrame]:
    """Retrain one frozen FastText neural configuration for fixed epochs."""
    model_config = CONFIG["models"][model_name]
    reset_neural_random_state(seed)
    model = build_fasttext_neural_model(model_name, parameters).to(DEVICE)
    optimizer = torch.optim.AdamW(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=float(parameters["learning_rate"]),
        weight_decay=float(model_config["weight_decay"]),
    )
    loss_function = nn.CrossEntropyLoss()
    batch_size = int(model_config["batch_size"])
    training_loader = make_fasttext_neural_loader(
        "train", batch_size, shuffle=True, seed=seed
    )
    history_rows = []
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        observed_rows = 0
        for token_ids, sequence_lengths, labels in training_loader:
            token_ids = token_ids.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            if model_name == "cnn":
                logits = model(token_ids)
            else:
                logits = model(token_ids, sequence_lengths)
            loss = loss_function(logits, labels)
            loss.backward()
            optimizer.step()
            batch_rows = len(labels)
            total_loss += float(loss.detach().cpu()) * batch_rows
            observed_rows += batch_rows
        history_rows.append(
            {"epoch": epoch, "training_loss": total_loss / observed_rows}
        )
    return model, pd.DataFrame(history_rows)


selected_model_name = "cnn"
selected_experiment = FASTTEXT_NEURAL_EXPERIMENTS[selected_model_name]
selected_parameters = selected_experiment["parameters"]
selected_epochs = int(selected_experiment["selected_epoch"])
selected_seed_rows = []
SELECTED_MODEL_SEED_PREDICTIONS = {}
SELECTED_MODEL_SEED_HISTORIES = {}
for final_seed in CONFIG["evaluation"]["final_seeds"]:
    seeded_model, seeded_history = train_fasttext_neural_fixed_epochs(
        selected_model_name, selected_parameters, selected_epochs, int(final_seed)
    )
    seeded_predictions = predict_fasttext_neural(
        seeded_model,
        "test",
        int(CONFIG["models"][selected_model_name]["batch_size"]),
    )
    seeded_metrics = aggregate_classification_metrics(
        TFIDF_TARGETS["test"], seeded_predictions
    )
    seeded_per_class = per_class_metrics(
        TFIDF_TARGETS["test"], seeded_predictions
    )
    SELECTED_MODEL_SEED_PREDICTIONS[int(final_seed)] = seeded_predictions
    SELECTED_MODEL_SEED_HISTORIES[int(final_seed)] = seeded_history
    selected_seed_rows.append(
        {
            "seed": int(final_seed),
            "epochs": selected_epochs,
            **seeded_metrics,
            **{
                f"{label.lower()}_f1": float(seeded_per_class.loc[label, "f1"])
                for label in SENTIMENT_LABELS
            },
        }
    )
selected_model_seed_results = pd.DataFrame(selected_seed_rows)
if not np.array_equal(
    SELECTED_MODEL_SEED_PREDICTIONS[SEED],
    FROZEN_TEST_PREDICTIONS[SELECTED_FINAL_MODEL_KEY],
):
    raise ValueError("Fixed-epoch seed-42 retraining did not reproduce the winner.")
selected_model_seed_summary = selected_model_seed_results.drop(
    columns=["seed", "epochs"]
).agg(["mean", "std"])
display(selected_model_seed_results.round(4))
display(selected_model_seed_summary.round(4))

,seed,epochs,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,neg_f1,neu_f1,pos_f1
0,42,5,0.6674,0.6524,0.6313,0.6380,0.6673,0.6674,0.6642,0.6997,0.6886,0.5258
1,1337,5,0.6503,0.6597,0.5805,0.5905,0.6560,0.6503,0.6349,0.6705,0.6996,0.4015
2,2026,5,0.6457,0.6586,0.5928,0.6024,0.6604,0.6457,0.6345,0.6959,0.6479,0.4635


,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,neg_f1,neu_f1,pos_f1
mean,0.6545,0.6569,0.6015,0.6103,0.6612,0.6545,0.6445,0.6887,0.6787,0.4636
std,0.0114,0.0039,0.0265,0.0247,0.0057,0.0114,0.0170,0.0159,0.0272,0.0621


In [86]:
selected_test_predictions = SELECTED_MODEL_SEED_PREDICTIONS[SEED]
test_error_frame = PROCESSED_PARTITIONS["test"].copy().reset_index(drop=True)
test_error_frame["predicted_sentiment"] = selected_test_predictions
test_error_frame["correct"] = (
    test_error_frame[CONFIG["data"]["target"]]
    == test_error_frame["predicted_sentiment"]
)
test_error_frame["processed_token_count"] = (
    test_error_frame[MODEL_TEXT_COLUMN].str.split().str.len()
)


def summarize_test_subgroups(frame: pd.DataFrame, group_column: str) -> pd.DataFrame:
    """Calculate fixed-label test metrics for one auxiliary grouping."""
    rows = []
    for group_value, group_frame in frame.groupby(group_column, sort=True):
        actual = group_frame[CONFIG["data"]["target"]]
        predicted = group_frame["predicted_sentiment"]
        metrics = aggregate_classification_metrics(actual, predicted)
        class_metrics = per_class_metrics(actual, predicted)
        rows.append(
            {
                group_column: group_value,
                "rows": len(group_frame),
                **metrics,
                **{
                    f"{label.lower()}_f1": float(class_metrics.loc[label, "f1"])
                    for label in SENTIMENT_LABELS
                },
            }
        )
    return pd.DataFrame(rows)


selected_error_by_sarcasm = summarize_test_subgroups(test_error_frame, "sarcasm")
selected_error_by_dialect = summarize_test_subgroups(test_error_frame, "dialect")
selected_confusion_pairs = (
    test_error_frame.loc[
        ~test_error_frame["correct"],
        [CONFIG["data"]["target"], "predicted_sentiment"],
    ]
    .value_counts()
    .rename("errors")
    .reset_index()
)
error_example_limit = int(
    CONFIG["experiments"]["final_evaluation"]["error_example_limit"]
)
selected_error_record_audit = (
    test_error_frame.loc[~test_error_frame["correct"]]
    .sort_values("record_id", kind="mergesort")
    .loc[
        :,
        [
            "record_id",
            CONFIG["data"]["target"],
            "predicted_sentiment",
            "sarcasm",
            "dialect",
            "processed_token_count",
        ],
    ]
    .head(error_example_limit)
)
display(selected_error_by_sarcasm.round(4))
display(selected_error_by_dialect.round(4))
display(selected_confusion_pairs)

,sarcasm,rows,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,neg_f1,neu_f1,pos_f1
0,False,2504,0.6398,0.6281,0.6176,0.6158,0.6559,0.6398,0.6417,0.5959,0.7126,0.5388
1,True,587,0.7853,0.4427,0.4716,0.4475,0.8294,0.7853,0.8039,0.8775,0.2937,0.1714


,dialect,rows,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,neg_f1,neu_f1,pos_f1
0,egypt,610,0.6279,0.6027,0.5411,0.5542,0.6169,0.6279,0.6067,0.7364,0.5000,0.4262
1,gulf,171,0.6140,0.5893,0.5976,0.5929,0.6181,0.6140,0.6157,0.6829,0.5586,0.5373
2,levant,123,0.6179,0.6266,0.6163,0.6206,0.6228,0.6179,0.6197,0.6538,0.5301,0.6780
3,magreb,8,0.5000,0.3111,0.3111,0.3111,0.5000,0.5000,0.5000,0.6000,0.3333,0.0000
4,msa,2179,0.6861,0.6645,0.6488,0.6539,0.6869,0.6861,0.6844,0.6893,0.7343,0.5382


,sentiment,predicted_sentiment,errors
0,NEU,NEG,332
1,NEG,NEU,227
2,POS,NEG,175
3,POS,NEU,114
4,NEU,POS,103
5,NEG,POS,77


In [87]:
final_results_directory = PATHS["results"] / "final"
final_results_directory.mkdir(parents=True, exist_ok=True)
final_comparison_path = final_results_directory / "model_comparison.csv"
final_test_details_path = final_results_directory / "test_metrics.json"
seed_results_path = final_results_directory / "selected_model_seed_metrics.csv"
seed_summary_path = final_results_directory / "selected_model_seed_summary.json"
sarcasm_error_path = final_results_directory / "error_analysis_by_sarcasm.csv"
dialect_error_path = final_results_directory / "error_analysis_by_dialect.csv"
confusion_pairs_path = final_results_directory / "error_confusion_pairs.csv"
error_records_path = final_results_directory / "error_record_audit.csv"

final_model_comparison.to_csv(final_comparison_path, index=False)
selected_model_seed_results.to_csv(seed_results_path, index=False)
selected_error_by_sarcasm.to_csv(sarcasm_error_path, index=False)
selected_error_by_dialect.to_csv(dialect_error_path, index=False)
selected_confusion_pairs.to_csv(confusion_pairs_path, index=False)
selected_error_record_audit.to_csv(error_records_path, index=False)
with final_test_details_path.open("w", encoding="utf-8") as result_file:
    json.dump(
        {
            "selection_metric": CONFIG["evaluation"]["primary_metric"],
            "selected_model_key": SELECTED_FINAL_MODEL_KEY,
            "selected_experiment_id": SELECTED_FINAL_EXPERIMENT_ID,
            "selection_completed_before_test": True,
            "test_rows": len(PROCESSED_PARTITIONS["test"]),
            "models": FINAL_TEST_DETAILS,
        },
        result_file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    result_file.write("\n")
with seed_summary_path.open("w", encoding="utf-8") as result_file:
    json.dump(
        {
            "model_key": SELECTED_FINAL_MODEL_KEY,
            "parameters": selected_parameters,
            "fixed_epochs": selected_epochs,
            "seeds": [int(seed) for seed in CONFIG["evaluation"]["final_seeds"]],
            "summary": selected_model_seed_summary.to_dict(orient="index"),
        },
        result_file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    result_file.write("\n")

In [88]:
final_model_comparison["representation_family"] = (
    final_model_comparison["representation"].str.split("_").str[0]
)
classifier_family_map = {
    "multinomial_naive_bayes": "traditional",
    "gaussian_naive_bayes": "traditional",
    "random_forest": "traditional",
    "cnn": "non_sequential_neural",
    "vanilla_rnn": "sequential_neural",
    "lstm": "sequential_neural",
}
final_model_comparison["classifier_family"] = final_model_comparison[
    "classifier"
].map(classifier_family_map)
representation_summary = (
    final_model_comparison.groupby("representation_family", as_index=False)
    .agg(
        models=("model_key", "size"),
        mean_validation_macro_f1=("validation_f1_macro", "mean"),
        best_validation_macro_f1=("validation_f1_macro", "max"),
        mean_test_macro_f1=("test_f1_macro", "mean"),
        best_test_macro_f1=("test_f1_macro", "max"),
    )
    .sort_values("best_validation_macro_f1", ascending=False)
)
architecture_summary = (
    final_model_comparison.groupby("classifier_family", as_index=False)
    .agg(
        models=("model_key", "size"),
        mean_validation_macro_f1=("validation_f1_macro", "mean"),
        mean_test_macro_f1=("test_f1_macro", "mean"),
        best_test_macro_f1=("test_f1_macro", "max"),
    )
    .sort_values("mean_test_macro_f1", ascending=False)
)
tuning_gain_rows = []
for tuning_path in sorted(tuning_results_directory.glob("*.csv")):
    if tuning_path.stem.endswith("selected_history"):
        continue
    tuning_frame = pd.read_csv(tuning_path)
    if "trial" not in tuning_frame or "f1_macro" not in tuning_frame:
        continue
    selected_row = tuning_frame.iloc[0]
    first_trial_row = tuning_frame.loc[tuning_frame["trial"] == 1].iloc[0]
    tuning_gain_rows.append(
        {
            "model_key": tuning_path.stem,
            "first_trial_macro_f1": float(first_trial_row["f1_macro"]),
            "selected_macro_f1": float(selected_row["f1_macro"]),
            "macro_f1_gain": float(
                selected_row["f1_macro"] - first_trial_row["f1_macro"]
            ),
            "selected_trial": int(selected_row["trial"]),
            "trials": len(tuning_frame),
        }
    )
tuning_gain_summary = pd.DataFrame(tuning_gain_rows).sort_values(
    "macro_f1_gain", ascending=False
)
display(representation_summary.round(4))
display(architecture_summary.round(4))
display(tuning_gain_summary.round(4))

,representation_family,models,mean_validation_macro_f1,best_validation_macro_f1,mean_test_macro_f1,best_test_macro_f1
0,fasttext,5,0.5938,0.6335,0.6017,0.6380
1,tfidf,2,0.6110,0.6335,0.6148,0.6348
2,word2vec,5,0.5982,0.6296,0.6017,0.6262


,classifier_family,models,mean_validation_macro_f1,mean_test_macro_f1,best_test_macro_f1
0,non_sequential_neural,2,0.6295,0.6317,0.6380
1,sequential_neural,4,0.6197,0.6186,0.6262
2,traditional,6,0.5740,0.5847,0.6348


,model_key,first_trial_macro_f1,selected_macro_f1,macro_f1_gain,selected_trial,trials
5,tfidf_multinomial_nb,0.6089,0.6335,0.0246,6,10
9,word2vec_lstm,0.6095,0.6296,0.0201,3,4
4,fasttext_rnn,0.5974,0.6149,0.0175,2,4
11,word2vec_rnn,0.6140,0.6177,0.0038,2,4
3,fasttext_random_forest,0.5954,0.5986,0.0032,3,8
10,word2vec_random_forest,0.5884,0.5900,0.0016,5,8
0,fasttext_cnn,0.6323,0.6335,0.0012,2,4
7,word2vec_cnn,0.6247,0.6256,0.0009,4,4
8,word2vec_gaussian_nb,0.5273,0.5279,0.0006,4,7
1,fasttext_gaussian_nb,0.5054,0.5054,0.0000,1,7


In [89]:
report_tables_directory = PATHS["tables"]
report_figures_directory = PATHS["figures"]
report_tables_directory.mkdir(parents=True, exist_ok=True)
report_figures_directory.mkdir(parents=True, exist_ok=True)

report_table_frames = {
    "model_comparison.csv": final_model_comparison,
    "representation_summary.csv": representation_summary,
    "architecture_summary.csv": architecture_summary,
    "tuning_gain_summary.csv": tuning_gain_summary,
    "preprocessing_ablation.csv": preprocessing_ablation_results,
    "augmentation_comparison.csv": augmentation_comparison,
    "class_weight_comparison.csv": class_weight_comparison,
    "selected_model_seed_metrics.csv": selected_model_seed_results,
    "error_analysis_by_sarcasm.csv": selected_error_by_sarcasm,
    "error_analysis_by_dialect.csv": selected_error_by_dialect,
    "error_confusion_pairs.csv": selected_confusion_pairs,
}
for filename, frame in report_table_frames.items():
    frame.to_csv(report_tables_directory / filename, index=False)

comparison_plot = final_model_comparison.copy()
comparison_plot["model_label"] = (
    comparison_plot["representation_family"]
    + " + "
    + comparison_plot["classifier"]
)
comparison_long = comparison_plot.melt(
    id_vars=["model_label", "validation_rank"],
    value_vars=["validation_f1_macro", "test_f1_macro"],
    var_name="partition_metric",
    value_name="macro_f1",
).sort_values("validation_rank")
figure, axis = plt.subplots(figsize=(12, 8))
sns.barplot(
    data=comparison_long,
    x="macro_f1",
    y="model_label",
    hue="partition_metric",
    orient="h",
    ax=axis,
)
axis.set_title("Validation and test macro F1 by required model combination")
axis.set_xlabel("Macro F1")
axis.set_ylabel("Representation + classifier")
figure.tight_layout()
model_comparison_figure_path = (
    report_figures_directory / "validation_test_model_comparison.png"
)
figure.savefig(model_comparison_figure_path, dpi=200, bbox_inches="tight")
plt.show()

selected_confusion_frame = pd.DataFrame.from_dict(
    FINAL_TEST_DETAILS[SELECTED_FINAL_MODEL_KEY]["confusion_matrix"],
    orient="index",
).reindex(index=SENTIMENT_LABELS, columns=SENTIMENT_LABELS)
figure, axis = plt.subplots(figsize=(6, 5))
sns.heatmap(
    selected_confusion_frame,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    ax=axis,
)
axis.set_title("Selected FastText CNN test confusion matrix")
axis.set_xlabel("Predicted")
axis.set_ylabel("Actual")
figure.tight_layout()
selected_confusion_figure_path = (
    report_figures_directory / "selected_model_test_confusion.png"
)
figure.savefig(selected_confusion_figure_path, dpi=200, bbox_inches="tight")
plt.show()

figure, axis = plt.subplots(figsize=(10, 7))
sns.barplot(
    data=ablation_plot_data,
    x="macro_f1_delta",
    y="disabled_stage",
    color="#4c78a8",
    ax=axis,
)
axis.axvline(0, color="black", linewidth=1)
axis.set_title("Preprocessing ablation: validation macro F1 change")
axis.set_xlabel("Macro F1 delta from full pipeline")
axis.set_ylabel("Disabled stage")
figure.tight_layout()
ablation_figure_path = report_figures_directory / "preprocessing_ablation.png"
figure.savefig(ablation_figure_path, dpi=200, bbox_inches="tight")
plt.show()

figure, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
sns.barplot(
    data=augmentation_class_f1,
    x="class_metric",
    y="f1",
    hue="condition",
    ax=axes[0],
)
axes[0].set_title("Arabic EDA validation class F1")
axes[0].set_xlabel("Sentiment class")
axes[0].set_ylabel("F1")
sns.barplot(
    data=weighted_class_f1,
    x="class_metric",
    y="f1",
    hue="condition",
    ax=axes[1],
)
axes[1].set_title("Balanced-weight validation class F1")
axes[1].set_xlabel("Sentiment class")
axes[1].set_ylabel("F1")
figure.tight_layout()
controlled_studies_figure_path = (
    report_figures_directory / "controlled_studies_class_f1.png"
)
figure.savefig(controlled_studies_figure_path, dpi=200, bbox_inches="tight")
plt.show()

figure, axis = plt.subplots(figsize=(8, 5))
sns.pointplot(
    data=selected_model_seed_results,
    x="seed",
    y="f1_macro",
    color="#7a5195",
    ax=axis,
)
axis.set_title("Selected FastText CNN test macro F1 across seeds")
axis.set_xlabel("Training seed")
axis.set_ylabel("Test macro F1")
figure.tight_layout()
seed_stability_figure_path = report_figures_directory / "seed_stability.png"
figure.savefig(seed_stability_figure_path, dpi=200, bbox_inches="tight")
plt.show()

generated_report_assets = [
    *(report_tables_directory / filename for filename in report_table_frames),
    model_comparison_figure_path,
    selected_confusion_figure_path,
    ablation_figure_path,
    controlled_studies_figure_path,
    seed_stability_figure_path,
]
pd.DataFrame(
    {
        "artifact": [
            str(path.relative_to(PROJECT_ROOT))
            for path in generated_report_assets
        ]
    }
)

/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/3084187996.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/3084187996.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/3084187996.py:91: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/3084187996.py:119: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/3084187996.py:121: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  figure, axis = plt.subplots(figsize=(8, 5))
/var/folders/tk/dy9ktf4s05b7lhv4_j0bl_740000gn/T/ipykernel_57195/3084187996.py:135: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,artifact
0,reports/tables/model_comparison.csv
1,reports/tables/representation_summary.csv
2,reports/tables/architecture_summary.csv
3,reports/tables/tuning_gain_summary.csv
4,reports/tables/preprocessing_ablation.csv
5,reports/tables/augmentation_comparison.csv
6,reports/tables/class_weight_comparison.csv
7,reports/tables/selected_model_seed_metrics.csv
8,reports/tables/error_analysis_by_sarcasm.csv
9,reports/tables/error_analysis_by_dialect.csv


### Final evaluation notes

The winning baseline is frozen solely from validation macro F1 before test predictions are created. All 12 required representation-model combinations are then evaluated on the untouched test partition, and only the winner is retrained at fixed selected epochs across three prespecified seeds. Auxiliary sarcasm and dialect analyses describe test errors without changing model decisions; small subgroups, especially Maghrebi, must be interpreted cautiously. The generated CSV tables and PNG figures are the sole quantitative inputs intended for the later report. The PDF report and README remain deliberately deferred for collaborative completion.